# Resume Classification: Data Preparation and Model Development

This notebook records the main data preparation and modeling work for the ResumeClassification project. The deployed application uses a fine-tuned ModernBERT model behind a FastAPI service, with a Streamlit interface.

The work is divided into data-quality checks, duplicate and label-conflict handling, split verification, BERT baseline experiments, ModernBERT tuning, final evaluation, and packaging the selected model for inference.

## Results from the completed run

| Item | Result |
|---|---:|
| Source dataset records | 13,389 |
| Records in the prepared dataset | 11,864 |
| Job categories | 43 |
| Training split | 9,473 |
| Validation split | 1,203 |
| Test split | 1,188 |
| BERT baseline test accuracy | 88.30% |
| ModernBERT test accuracy | 92.09% |
| ModernBERT macro F1 | 92.12% |
| ModernBERT weighted F1 | 92.06% |

The comparison uses the same held-out test split. BERT was evaluated with a 512-token limit and ModernBERT with a 2,048-token limit, matching each model's configured inference setup; the input limits therefore differ.

## Before running

- This is a Kaggle-oriented development notebook, not a one-click script. It expects artifacts under `/kaggle/working/resume_classification` and includes recovery/verification cells for sessions where that workspace has already been prepared.
- Dataset preparation and training cells can be expensive. The tuning cells require a GPU and can take a long time; do not run every cell from top to bottom without checking the section notes.
- The final model and dataset are not bundled with this notebook. Restore or download the approved data/model artifacts in the intended environment first.
- This public copy has its saved cell outputs and execution counters cleared. The original outputs included excerpts from resume text during data-quality review, so they are not included in the repository. The source code and the recorded aggregate metrics remain available.

## 1. Environment and hardware checks

Record Python and library versions, CUDA availability, and the GPU/RAM resources exposed by the current session.

In [ ]:
import sys
import platform
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
        print(
            "GPU memory (GB):",
            round(
                torch.cuda.get_device_properties(0).total_memory / 1024**3,
                2
            )
        )
except ImportError:
    print("PyTorch is not installed in this environment.")

In [ ]:
import torch
import subprocess
import shutil
import psutil

# 1. Verify how many GPUs PyTorch can access
gpu_count = torch.cuda.device_count()

print("CUDA available:", torch.cuda.is_available())
print("Number of visible GPUs:", gpu_count)

# 2. Display information for every detected GPU
total_gpu_memory = 0

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    memory_gib = props.total_memory / (1024 ** 3)
    total_gpu_memory += memory_gib

    print(f"\nGPU {i}: {props.name}")
    print(f"  Total GPU memory: {memory_gib:.2f} GiB")

print(f"\nAggregate GPU memory: {total_gpu_memory:.2f} GiB")

# 3. Check system RAM separately
ram = psutil.virtual_memory()

print("\nSystem RAM:")
print(f"  Total: {ram.total / (1024 ** 3):.2f} GiB")
print(f"  Available: {ram.available / (1024 ** 3):.2f} GiB")

# 4. Show NVIDIA's GPU status, if available
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)

## 2. Load and preserve the raw dataset

Load the ResumeAtlas dataset and save a verified raw copy before preprocessing. The raw file is retained separately from all prepared datasets.

In [ ]:
from datasets import load_dataset

# Dataset source
DATASET_ID = "ahmedheakl/resume-atlas"

# Download and load the training split
dataset = load_dataset(DATASET_ID, split="train")

# Convert to Pandas for convenient inspection
df = dataset.to_pandas()

# Verify the actual dataset
print("=" * 50)
print("DATASET LOADING REPORT")
print("=" * 50)

print("Dataset ID:", DATASET_ID)
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))
print("Column names:", df.columns.tolist())

print("\nDataset features:")
print(dataset.features)

print("\nPandas data types:")
print(df.dtypes)

print("\nFirst 5 rows (structure only):")
print(df.head().drop(columns=["Text"], errors="ignore"))

print("\nDataset loaded successfully.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import sys

import pandas as pd
import numpy as np
import psutil
import torch

# 1. Persistent project directory
PROJECT_DIR = Path("/kaggle/working/resume_classification")
RAW_DIR = PROJECT_DIR / "data" / "raw"
REPORT_DIR = PROJECT_DIR / "reports"
CONFIG_DIR = PROJECT_DIR / "metadata"

for folder in [RAW_DIR, REPORT_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Stop if the loaded dataset is not available in memory
if "df" not in globals() or "dataset" not in globals():
    raise RuntimeError(
        "The loaded dataset is missing. Run the cache-aware loading cell "
        "provided below after a session restart."
    )

# 2. Save the original loaded data without preprocessing
PARQUET_PATH = RAW_DIR / "resume_atlas_raw.parquet"
CSV_PATH = RAW_DIR / "resume_atlas_raw.csv"

if not PARQUET_PATH.exists():
    df.to_parquet(PARQUET_PATH, index=False)

if not CSV_PATH.exists():
    df.to_csv(CSV_PATH, index=False, encoding="utf-8")

# 3. Verify that the saved Parquet file can be reloaded
saved_df = pd.read_parquet(PARQUET_PATH)

assert saved_df.shape == df.shape, "Dataset shape verification failed."
assert saved_df.columns.tolist() == df.columns.tolist(), (
    "Dataset columns verification failed."
)

# Compare selected records against the current DataFrame
sample_indices = sorted(set([
    0,
    len(df) // 2,
    len(df) - 1
]))

for i in sample_indices:
    for col in df.columns:
        original = df.iloc[i][col]
        restored = saved_df.iloc[i][col]

        if pd.isna(original) and pd.isna(restored):
            continue

        assert original == restored, (
            f"Saved data mismatch at row {i}, column {col}"
        )

# 4. Save dataset metadata
dataset_metadata = {
    "dataset_id": "ahmedheakl/resume-atlas",
    "source": "Hugging Face Datasets",
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "rows": int(len(df)),
    "columns": df.columns.tolist(),
    "features": str(dataset.features),
    "pandas_dtypes": {
        col: str(dtype)
        for col, dtype in df.dtypes.items()
    },
    "preprocessing_applied": False,
    "artifacts": {
        "parquet": str(PARQUET_PATH),
        "csv": str(CSV_PATH)
    }
}

with open(CONFIG_DIR / "dataset_metadata.json", "w") as f:
    json.dump(dataset_metadata, f, indent=2)

# 5. Save hardware and environment report
gpu_details = []

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)

        gpu_details.append({
            "index": i,
            "name": props.name,
            "memory_gib": round(
                props.total_memory / (1024 ** 3), 2
            )
        })

environment_report = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpus": gpu_details,
    "system_ram_gib": round(
        psutil.virtual_memory().total / (1024 ** 3), 2
    )
}

with open(REPORT_DIR / "environment_report.json", "w") as f:
    json.dump(environment_report, f, indent=2)

# 6. Print verification report
print("PERSISTENCE CHECK")
print("=" * 50)
print("Rows verified:", len(saved_df))
print("Columns verified:", saved_df.columns.tolist())
print("Selected records verified:", len(sample_indices))
print("Preprocessing applied: No")

print("\nSaved files:")
for path in [
    PARQUET_PATH,
    CSV_PATH,
    CONFIG_DIR / "dataset_metadata.json",
    REPORT_DIR / "environment_report.json"
]:
    print(
        f"{path} | "
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

print("\nSUCCESS: Saved dataset was reloaded and verified.")

## 3. Dataset quality, exact duplicates, and label conflicts

Audit empty/short records, repeated text, and records assigned to conflicting categories. These checks are run before creating the final splits.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import re

import pandas as pd
import numpy as np

# ============================================================
# 1. RESTORE DATA FROM PERSISTENT STORAGE
# ============================================================

PROJECT_DIR = Path("/kaggle/working/resume_classification")
RAW_DIR = PROJECT_DIR / "data" / "raw"
REPORT_DIR = PROJECT_DIR / "reports"
CONFIG_DIR = PROJECT_DIR / "metadata"

for folder in [RAW_DIR, REPORT_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = RAW_DIR / "resume_atlas_raw.parquet"

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    print("Loaded the saved dataset. No download required.")
else:
    from datasets import load_dataset

    ds = load_dataset("ahmedheakl/resume-atlas", split="train")
    df = ds.to_pandas()
    df.to_parquet(PARQUET_PATH, index=False)
    print("Saved dataset was unavailable; downloaded and saved a new copy.")

LABEL_COL = "Category"
TEXT_COL = "Text"

assert LABEL_COL in df.columns, f"Missing column: {LABEL_COL}"
assert TEXT_COL in df.columns, f"Missing column: {TEXT_COL}"

print("Dataset shape:", df.shape)

# All following transformations are temporary audit representations.
# The original dataframe is not modified.

labels = df[LABEL_COL].astype("string")
texts = df[TEXT_COL].astype("string")

label_stripped = labels.fillna("").str.strip()
text_stripped = texts.fillna("").str.strip()

text_lengths = text_stripped.str.len()
word_counts = text_stripped.str.split().str.len()

# ============================================================
# 2. MISSING VALUES AND BASIC VALIDATION
# ============================================================

missing_categories = int(labels.isna().sum())
blank_categories = int(
    (labels.notna() & label_stripped.eq("")).sum()
)

missing_texts = int(texts.isna().sum())
blank_texts = int(
    (texts.notna() & text_stripped.eq("")).sum()
)

category_counts = label_stripped[
    label_stripped.ne("")
].value_counts()

length_description = {
    "characters": {
        key: float(value)
        for key, value in text_lengths.describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        ).items()
    },
    "whitespace_separated_words": {
        key: float(value)
        for key, value in word_counts.describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        ).items()
    }
}

# ============================================================
# 3. EXACT AND NORMALIZED DUPLICATES
# ============================================================

# Exact duplicate rows: category and text both match.
exact_duplicate_rows = int(
    df.duplicated(subset=[LABEL_COL, TEXT_COL]).sum()
)

# Normalize only for duplicate detection:
# case-folding and whitespace normalization.
# This is NOT the cleaned training text.
normalized_text = (
    text_stripped
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

valid_text_mask = normalized_text.ne("")

duplicate_mask = (
    valid_text_mask
    & normalized_text.duplicated(keep=False)
)

duplicate_rows = int(duplicate_mask.sum())
duplicate_extra_occurrences = int(
    (valid_text_mask & normalized_text.duplicated()).sum()
)

audit_duplicates = pd.DataFrame({
    "row_index": df.index,
    "normalized_text": normalized_text,
    "category": label_stripped
})

audit_duplicates = audit_duplicates[
    audit_duplicates["normalized_text"].ne("")
]

duplicate_groups = []
conflicting_groups = []

for normalized, group in audit_duplicates.groupby(
    "normalized_text", sort=False
):
    if len(group) < 2:
        continue

    categories = sorted(group["category"].dropna().unique().tolist())

    group_record = {
        "text_sha256": hashlib.sha256(
            normalized.encode("utf-8")
        ).hexdigest(),
        "record_count": int(len(group)),
        "category_count": len(categories),
        "categories": " | ".join(categories),
        "row_indices": " | ".join(
            map(str, group["row_index"].tolist())
        )
    }

    duplicate_groups.append(group_record)

    if len(categories) > 1:
        conflicting_groups.append(group_record)

duplicate_groups_df = pd.DataFrame(duplicate_groups)
conflicting_groups_df = pd.DataFrame(conflicting_groups)

# ============================================================
# 4. CATEGORY-NAME VARIANTS
# ============================================================

label_audit = pd.DataFrame({
    "original_category": label_stripped
})

label_audit = label_audit[
    label_audit["original_category"].ne("")
].copy()

label_audit["normalized_category"] = (
    label_audit["original_category"]
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
)

category_variants = []

for normalized, group in label_audit.groupby(
    "normalized_category"
):
    variants = sorted(group["original_category"].unique().tolist())

    if len(variants) > 1:
        category_variants.append({
            "normalized_form": normalized,
            "number_of_variants": len(variants),
            "original_variants": " | ".join(variants),
            "records_affected": int(len(group))
        })

category_variants_df = pd.DataFrame(category_variants)

# ============================================================
# 5. TEXT FORMATTING AND TECHNICAL-TERM INDICATORS
# ============================================================

raw_text = texts.fillna("")

formatting_indicators = {
    "rows_with_newlines": int(
        raw_text.str.contains(r"[\r\n]", regex=True).sum()
    ),
    "rows_with_tabs": int(
        raw_text.str.contains("\t", regex=False).sum()
    ),
    "rows_with_replacement_characters": int(
        raw_text.str.contains("\uFFFD", regex=False).sum()
    ),
    "rows_with_null_bytes": int(
        raw_text.str.contains("\x00", regex=False).sum()
    ),
    "rows_with_non_ascii_characters": int(
        raw_text.str.contains(r"[^\x00-\x7F]", regex=True).sum()
    ),
    "rows_with_excessive_whitespace": int(
        raw_text.str.contains(r"[ \t]{3,}", regex=True).sum()
    )
}

technical_terms = [
    r"C\+\+",
    r"C#",
    r"\.NET",
    r"CI/CD",
    r"\bSQL\b",
    r"\bPython\b",
    r"\bJavaScript\b",
    r"\bNode\.js\b",
    r"\bTypeScript\b",
    r"\bHTML\b",
    r"\bCSS\b",
    r"\bAWS\b",
    r"\bDocker\b"
]

technical_term_counts = {}

for pattern in technical_terms:
    technical_term_counts[pattern] = int(
        raw_text.str.contains(
            pattern, case=False, regex=True
        ).sum()
    )

# Common personal/contact data indicators.
# These are approximate pattern checks, not a complete PII detector.
pii_indicators = {
    "rows_with_email_like_strings": int(
        raw_text.str.contains(
            r"\b[\w.+%-]+@[\w.-]+\.[A-Za-z]{2,}\b",
            case=False, regex=True
        ).sum()
    ),
    "rows_with_url_like_strings": int(
        raw_text.str.contains(
            r"(?:https?://|www\.)\S+",
            case=False, regex=True
        ).sum()
    ),
    "rows_with_phone_like_strings": int(
        raw_text.str.contains(
            r"(?<!\w)\+?\d[\d ().-]{6,}\d(?!\w)",
            regex=True
        ).sum()
    )
}

# ============================================================
# 6. PRINT AUDIT RESULTS
# ============================================================

print("\n" + "=" * 65)
print("INITIAL DATA-QUALITY AUDIT")
print("=" * 65)

print("\n[1] Dataset")
print("Records:", len(df))
print("Columns:", df.columns.tolist())
print("Distinct nonblank categories:", len(category_counts))

print("\n[2] Missing and short records")
print("Missing categories:", missing_categories)
print("Blank categories:", blank_categories)
print("Missing resume texts:", missing_texts)
print("Blank resume texts:", blank_texts)
print("Resumes under 50 characters:", int((text_lengths < 50).sum()))
print("Resumes under 20 words:", int((word_counts < 20).sum()))

print("\n[3] Category distribution")
print("Largest category:", int(category_counts.max()))
print("Smallest category:", int(category_counts.min()))
print("Median category size:", float(category_counts.median()))
print("Largest/smallest ratio:",
      round(category_counts.max() / category_counts.min(), 2))
print("\n10 largest categories:")
print(category_counts.head(10).to_string())
print("\n10 smallest categories:")
print(category_counts.tail(10).to_string())

print("\n[4] Resume lengths")
print(pd.DataFrame({
    "characters": text_lengths,
    "whitespace_separated_words": word_counts
}).describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(1))

print("\n[5] Duplicates and conflicting labels")
print("Exact duplicate rows:", exact_duplicate_rows)
print("Rows in normalized duplicate groups:", duplicate_rows)
print("Extra duplicate occurrences:", duplicate_extra_occurrences)
print("Normalized duplicate groups:", len(duplicate_groups_df))
print("Groups with conflicting labels:", len(conflicting_groups_df))

print("\n[6] Category-name variants")
print("Potential category-name inconsistencies:",
      len(category_variants_df))
if not category_variants_df.empty:
    print(category_variants_df.head(20).to_string(index=False))

print("\n[7] Formatting indicators")
print(json.dumps(formatting_indicators, indent=2))

print("\n[8] Technical-term indicators")
print(json.dumps(technical_term_counts, indent=2))

print("\n[9] Approximate contact-data indicators")
print(json.dumps(pii_indicators, indent=2))

# ============================================================
# 7. DISPLAY MASKED SAMPLE PREVIEWS
# ============================================================

def safe_preview(value, max_chars=500):
    if pd.isna(value):
        return "[MISSING TEXT]"

    value = str(value)

    value = re.sub(
        r"\b[\w.+%-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL]", value
    )
    value = re.sub(
        r"(?:https?://|www\.)\S+",
        "[URL]", value, flags=re.IGNORECASE
    )
    value = re.sub(
        r"(?<!\w)\+?\d[\d ().-]{6,}\d(?!\w)",
        "[PHONE/NUMBER]", value
    )

    value = re.sub(r"\s+", " ", value).strip()

    return value[:max_chars] + (
        " ...[TRUNCATED]" if len(value) > max_chars else ""
    )

print("\n[10] Five representative resume previews")

sample = df.sample(n=min(5, len(df)), random_state=42)

for row_index, row in sample.iterrows():
    print(f"\nRow index: {row_index}")
    print("Category:", row[LABEL_COL])
    print("Preview:", safe_preview(row[TEXT_COL]))

# ============================================================
# 8. SAVE AUDIT ARTIFACTS FOR SESSION RECOVERY
# ============================================================

category_counts.rename_axis("Category").reset_index(
    name="record_count"
).to_csv(REPORT_DIR / "category_distribution.csv", index=False)

pd.DataFrame({
    "row_index": df.index,
    "character_count": text_lengths,
    "word_count": word_counts
}).to_csv(REPORT_DIR / "text_length_by_row.csv", index=False)

duplicate_groups_df.to_csv(
    REPORT_DIR / "duplicate_text_groups.csv", index=False
)

conflicting_groups_df.to_csv(
    REPORT_DIR / "conflicting_label_groups.csv", index=False
)

category_variants_df.to_csv(
    REPORT_DIR / "category_name_variants.csv", index=False
)

audit_report = {
    "dataset_id": "ahmedheakl/resume-atlas",
    "rows": int(len(df)),
    "columns": df.columns.tolist(),
    "distinct_nonblank_categories": int(len(category_counts)),
    "missing_categories": missing_categories,
    "blank_categories": blank_categories,
    "missing_resume_texts": missing_texts,
    "blank_resume_texts": blank_texts,
    "resumes_under_50_characters": int((text_lengths < 50).sum()),
    "resumes_under_20_words": int((word_counts < 20).sum()),
    "exact_duplicate_rows": exact_duplicate_rows,
    "normalized_duplicate_rows": duplicate_rows,
    "extra_duplicate_occurrences": duplicate_extra_occurrences,
    "normalized_duplicate_groups": len(duplicate_groups_df),
    "conflicting_label_groups": len(conflicting_groups_df),
    "category_name_variant_groups": len(category_variants_df),
    "character_length_summary": length_description["characters"],
    "word_count_summary": length_description["whitespace_separated_words"],
    "formatting_indicators": formatting_indicators,
    "technical_term_counts": technical_term_counts,
    "contact_data_indicators": pii_indicators,
    "cleaning_applied": False
}

with open(REPORT_DIR / "initial_data_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit_report, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 65)
print("AUDIT SAVED SUCCESSFULLY")
print("=" * 65)

for path in sorted(REPORT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size / 1024:.1f} KB")

print("\nOriginal dataset remains unchanged.")

In [ ]:
from pathlib import Path
import hashlib
import json
import re

import pandas as pd

# --------------------------------------------------
# 1. Restore the original saved dataset
# --------------------------------------------------
PROJECT_DIR = Path("/kaggle/working/resume_classification")
RAW_PATH = PROJECT_DIR / "data" / "raw" / "resume_atlas_raw.parquet"
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Saved raw dataset not found: {RAW_PATH}"
    )

df = pd.read_parquet(RAW_PATH)

assert {"Category", "Text"}.issubset(df.columns)
print("Loaded saved raw dataset:", df.shape)

# --------------------------------------------------
# 2. Build a detection-only normalized text
# --------------------------------------------------
# Case folding + whitespace normalization only.
# Do not use this as training text.
text = df["Text"].fillna("").astype(str)
labels = df["Category"].fillna("").astype(str).str.strip()

normalized = (
    text.str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)

audit = pd.DataFrame({
    "row_index": df.index,
    "normalized_text": normalized,
    "category": labels
})

audit = audit[audit["normalized_text"].ne("")]

# Review repeated-text groups with conflicting labels.
conflict_rows = []

for normalized_text, group in audit.groupby(
    "normalized_text", sort=False
):
    categories = sorted(group["category"].unique())

    if len(categories) > 1:
        conflict_rows.append({
            "text_sha256": hashlib.sha256(
                normalized_text.encode("utf-8")
            ).hexdigest(),
            "record_count": len(group),
            "categories": " | ".join(categories),
            "row_indices": " | ".join(
                map(str, group["row_index"].tolist())
            ),
            # This text is stored only in the private Kaggle working area.
            "normalized_text": normalized_text
        })

conflicts = pd.DataFrame(conflict_rows)

print("\nCONFLICT REVIEW")
print("Conflicting text groups:", len(conflicts))

if not conflicts.empty:
    print("\nNumber of groups by number of categories:")
    print(
        conflicts["categories"]
        .str.count(r"\|")
        .add(1)
        .value_counts()
        .sort_index()
        .rename_axis("number_of_categories")
        .to_string()
    )

    print("\nFive conflict examples (text truncated):")
    for _, row in conflicts.head(5).iterrows():
        print("\nCategories:", row["categories"])
        print("Rows:", row["row_indices"])
        print("Text:", row["normalized_text"][:350])

    # Contains resume text: keep private; never upload publicly.
    conflicts.to_csv(
        REPORT_DIR / "conflicting_texts_private_review.csv",
        index=False
    )

# --------------------------------------------------
# 3. Check technical terms and punctuation
# --------------------------------------------------
patterns = {
    "C++": r"C\+\+",
    "C#": r"C#",
    ".NET": r"\.NET",
    "CI/CD": r"CI/CD",
    "Node.js": r"Node\.js",
    "SQL": r"\bSQL\b",
    "Python": r"\bPython\b",
    "JavaScript": r"\bJavaScript\b",
    "email-like marker": r"\bemail\b",
    "phone-like marker": r"\b(?:phone|mobile|cell)\b",
    "long digit sequence": r"\d{6,}",
    "punctuation present": r"[^\w\s]",
    "multiple spaces": r" {2,}"
}

term_results = []

for name, pattern in patterns.items():
    matching = text.str.contains(
        pattern, case=False, regex=True, na=False
    )
    term_results.append({
        "pattern": name,
        "matching_rows": int(matching.sum()),
        "percentage": round(100 * matching.mean(), 2)
    })

term_report = pd.DataFrame(term_results)

print("\nTECHNICAL-TERM AND TEXT-FORMAT CHECK")
print(term_report.to_string(index=False))

term_report.to_csv(
    REPORT_DIR / "technical_term_diagnostics.csv",
    index=False
)

# --------------------------------------------------
# 4. Inspect a small set of matching examples
# --------------------------------------------------
# Mask email-like and phone-like values for display.
def redact_preview(value, limit=350):
    value = str(value)
    value = re.sub(
        r"\b[\w.+%-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL]", value
    )
    value = re.sub(
        r"(?<!\w)\+?\d[\d ().-]{6,}\d(?!\w)",
        "[NUMBER]", value
    )
    value = re.sub(
        r"\s+", " ", value
    ).strip()
    return value[:limit]

print("\nEXAMPLES CONTAINING TECHNICAL TERMS")

for term in ["Python", "SQL", "C++", "C#", ".NET", "CI/CD"]:
    pattern = patterns[term]
    matching = text[text.str.contains(
        pattern, case=False, regex=True, na=False
    )]

    print(f"\n{term}: {len(matching)} rows")

    for idx, value in matching.head(2).items():
        print(f"Row {idx}: {redact_preview(value)}")

# --------------------------------------------------
# 5. Save a small machine-readable summary
# --------------------------------------------------
summary = {
    "dataset_rows": int(len(df)),
    "conflicting_text_groups": int(len(conflicts)),
    "technical_term_diagnostics": term_results,
    "source_dataset_modified": False
}

with open(
    REPORT_DIR / "focused_data_diagnostics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nSaved diagnostic reports to:", REPORT_DIR)
print("Original raw dataset was not modified.")

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import hashlib
import json
import re

import pandas as pd

# 1. Restore persistent data
PROJECT_DIR = Path("/kaggle/working/resume_classification")
RAW_PATH = PROJECT_DIR / "data/raw/resume_atlas_raw.parquet"
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(RAW_PATH)

text = df["Text"].fillna("").astype(str)
labels = df["Category"].fillna("").astype(str).str.strip()

# Detection-only normalization; this is not training preprocessing.
normalized = (
    text.str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)

audit = pd.DataFrame({
    "row_index": df.index,
    "normalized_text": normalized,
    "category": labels
})

audit = audit[audit["normalized_text"].ne("")]

# 2. Recalculate all repeated-text groups
duplicate_groups = []
conflict_groups = []
pair_counts = Counter()

for value, group in audit.groupby("normalized_text", sort=False):
    if len(group) < 2:
        continue

    categories = sorted(group["category"].unique())

    item = {
        "text_hash": hashlib.sha256(
            value.encode("utf-8")
        ).hexdigest(),
        "records": int(len(group)),
        "category_count": len(categories),
        "categories": " | ".join(categories),
        "row_indices": " | ".join(
            map(str, group["row_index"].tolist())
        )
    }

    duplicate_groups.append(item)

    if len(categories) > 1:
        conflict_groups.append(item)

        for pair in combinations(categories, 2):
            pair_counts[pair] += 1

duplicates_df = pd.DataFrame(duplicate_groups)
conflicts_df = pd.DataFrame(conflict_groups)

duplicate_records = int(duplicates_df["records"].sum())
conflict_records = int(conflicts_df["records"].sum())

print("=" * 60)
print("DUPLICATE IMPACT")
print("=" * 60)
print("Repeated-text groups:", len(duplicates_df))
print("Records in repeated-text groups:", duplicate_records)
print("Additional occurrences beyond one per group:",
      duplicate_records - len(duplicates_df))
print("Conflicting-label groups:", len(conflicts_df))
print("Records belonging to conflicting groups:", conflict_records)
print("Groups with consistent labels:",
      len(duplicates_df) - len(conflicts_df))

print("\nMost common conflicting category pairs:")
for pair, count in pair_counts.most_common(15):
    print(f"{pair[0]} <-> {pair[1]}: {count} groups")

# 3. Diagnose signs of previous text preprocessing
print("\n" + "=" * 60)
print("TEXT-PREPROCESSING FINGERPRINT")
print("=" * 60)

checks = {
    "Rows containing punctuation": r"[^\w\s]",
    "Rows containing uppercase letters": r"[A-Z]",
    "Rows containing newlines": r"[\r\n]",
    "Rows containing repeated spaces": r" {2,}",
}

preprocessing_results = {}

for name, pattern in checks.items():
    count = int(text.str.contains(pattern, regex=True, na=False).sum())
    preprocessing_results[name] = count
    print(f"{name}: {count}")

print("\nCommon stop-word presence (whole-word matches):")

stop_words = ["the", "and", "is", "of", "with", "for", "in", "to", "a", "an"]
stop_word_results = {}

for word in stop_words:
    count = int(
        text.str.contains(
            rf"\b{re.escape(word)}\b",
            case=False,
            regex=True,
            na=False
        ).sum()
    )
    stop_word_results[word] = count
    print(f"{word}: {count} rows")

# 4. Search for variants of terms whose punctuation may have been removed
term_variants = {
    "cplusplus": r"\bcplusplus\b",
    "c plus plus": r"\bc\s+plus\s+plus\b",
    "csharp": r"\bcsharp\b",
    "c sharp": r"\bc\s+sharp\b",
    "dotnet": r"\bdotnet\b",
    "dot net": r"\bdot\s+net\b",
    "net framework": r"\bnet\s+framework\b",
    "ci cd": r"\bci\s+cd\b",
    "cicd": r"\bcicd\b",
    "nodejs": r"\bnodejs\b",
    "node js": r"\bnode\s+js\b",
    "javascript": r"\bjavascript\b",
    "sql": r"\bsql\b",
    "python": r"\bpython\b",
}

term_results = {}

print("\n" + "=" * 60)
print("TECHNICAL TERM VARIANTS")
print("=" * 60)

for term, pattern in term_variants.items():
    count = int(
        text.str.contains(
            pattern, case=False, regex=True, na=False
        ).sum()
    )
    term_results[term] = count
    print(f"{term}: {count} rows")

# 5. Save only non-text metadata for duplicate/conflict review
duplicates_df.to_csv(
    REPORT_DIR / "duplicate_groups_metadata.csv",
    index=False
)
conflicts_df.to_csv(
    REPORT_DIR / "conflicting_groups_metadata.csv",
    index=False
)

# Remove the earlier derived report that duplicated complete resume text.
old_private_report = REPORT_DIR / "conflicting_texts_private_review.csv"

if old_private_report.exists():
    old_private_report.unlink()
    print("\nRemoved redundant full-text conflict report.")

summary = {
    "dataset_rows": int(len(df)),
    "repeated_text_groups": int(len(duplicates_df)),
    "records_in_repeated_text_groups": duplicate_records,
    "conflicting_label_groups": int(len(conflicts_df)),
    "records_in_conflicting_groups": conflict_records,
    "preprocessing_indicators": preprocessing_results,
    "stop_word_row_counts": stop_word_results,
    "technical_term_variant_counts": term_results,
    "raw_dataset_modified": False
}

with open(
    REPORT_DIR / "preprocessing_diagnostics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nSaved diagnostics to:", REPORT_DIR)
print("Original dataset remains unchanged.")

## 4. Build the primary dataset and inspect near-duplicates

Remove records with unresolved conflicting labels, deduplicate repeated records, and review high-similarity pairs with different labels. Similarity results are a review aid, not an automatic relabeling rule.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd

# =========================================================
# 1. RESTORE RAW DATA AND SET UP PERSISTENT DIRECTORIES
# =========================================================

PROJECT_DIR = Path("/kaggle/working/resume_classification")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"
METADATA_DIR = PROJECT_DIR / "metadata"

for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "resume_atlas_raw.parquet"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_PATH}. "
        "Do not download another copy without checking the project files."
    )

raw_df = pd.read_parquet(RAW_PATH)

assert list(raw_df.columns) == ["Category", "Text"]
assert len(raw_df) == 13389, (
    f"Unexpected row count: {len(raw_df)}. "
    "Recheck the source before proceeding."
)

print("Restored raw dataset:", raw_df.shape)

# Work on a separate DataFrame.
# Never overwrite the raw dataset.
base = raw_df.copy()
base.insert(0, "source_row_index", raw_df.index)

# =========================================================
# 2. CREATE DETECTION KEYS
# =========================================================

# This normalization is used only to identify equivalent text.
# It is NOT a new text-cleaning operation.
base["_text_key"] = (
    base["Text"]
    .fillna("")
    .astype(str)
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

base["_label_key"] = (
    base["Category"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
)

invalid_mask = (
    base["_text_key"].eq("")
    | base["_label_key"].eq("")
)

valid_base = base.loc[~invalid_mask].copy()

# =========================================================
# 3. IDENTIFY TEXT GROUPS WITH CONFLICTING LABELS
# =========================================================

labels_per_text = (
    valid_base.groupby("_text_key")["_label_key"]
    .nunique()
)

conflict_keys = set(
    labels_per_text[labels_per_text > 1].index
)

conflict_mask = base["_text_key"].isin(conflict_keys)

# Create a metadata-only conflict report.
# No resume text is copied into these audit reports.
conflict_group_records = []

for key in sorted(conflict_keys):
    group = valid_base.loc[valid_base["_text_key"].eq(key)]

    original_categories = sorted(
        group["Category"]
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    conflict_group_records.append({
        "text_sha256": hashlib.sha256(
            key.encode("utf-8")
        ).hexdigest(),
        "record_count": int(len(group)),
        "category_count": len(original_categories),
        "categories": " | ".join(original_categories),
        "source_row_indices": " | ".join(
            map(str, group["source_row_index"].tolist())
        )
    })

conflict_groups_df = pd.DataFrame(conflict_group_records)

# Exclude entire conflicting text groups.
excluded_mask = conflict_mask | invalid_mask
eligible = base.loc[~excluded_mask].copy()

# =========================================================
# 4. DEDUPLICATE REMAINING CONSISTENT-LABEL RECORDS
# =========================================================

# Every repeated text remaining here has one consistent category.
duplicate_drop_mask = eligible["_text_key"].duplicated(keep="first")

# Identify the retained source record for every dropped duplicate.
kept_record_lookup = (
    eligible.loc[~duplicate_drop_mask]
    .set_index("_text_key")["source_row_index"]
    .to_dict()
)

dropped_duplicates = eligible.loc[
    duplicate_drop_mask,
    ["source_row_index", "Category", "_text_key"]
].copy()

dropped_duplicates["kept_source_row_index"] = (
    dropped_duplicates["_text_key"].map(kept_record_lookup)
)

dropped_duplicates["text_sha256"] = (
    dropped_duplicates["_text_key"]
    .map(lambda value: hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest())
)

dropped_duplicates["reason"] = "Repeated text; same label retained"

dropped_duplicates = dropped_duplicates.drop(columns=["_text_key"])

# Keep one instance of every eligible normalized text.
primary = eligible.loc[
    ~duplicate_drop_mask,
    ["source_row_index", "Category", "Text"]
].copy()

primary = primary.reset_index(drop=True)

# =========================================================
# 5. CREATE AUDIT TRAILS
# =========================================================

excluded_rows = base.loc[
    excluded_mask,
    ["source_row_index", "Category", "_text_key"]
].copy()

excluded_rows["reason"] = "Conflicting labels or invalid text/category"

excluded_rows.loc[
    conflict_mask.loc[excluded_rows.index],
    "reason"
] = "Quarantined: identical normalized text has conflicting labels"

excluded_rows["text_sha256"] = (
    excluded_rows["_text_key"]
    .map(lambda value: hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest())
)

excluded_rows = excluded_rows.drop(columns=["_text_key"])

# Save audit files without complete resume text.
conflict_groups_df.to_csv(
    REPORT_DIR / "primary_conflict_groups.csv",
    index=False
)

excluded_rows.to_csv(
    REPORT_DIR / "primary_excluded_rows.csv",
    index=False
)

dropped_duplicates.to_csv(
    REPORT_DIR / "primary_deduplicated_rows.csv",
    index=False
)

# =========================================================
# 6. SAVE THE PRIMARY DATASET
# =========================================================

PRIMARY_PARQUET = PROCESSED_DIR / "primary_dataset.parquet"
PRIMARY_CSV = PROCESSED_DIR / "primary_dataset.csv"

primary.to_parquet(PRIMARY_PARQUET, index=False)
primary.to_csv(PRIMARY_CSV, index=False, encoding="utf-8")

primary_category_counts = (
    primary["Category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="record_count")
)

primary_category_counts.to_csv(
    REPORT_DIR / "primary_category_distribution.csv",
    index=False
)

# =========================================================
# 7. VERIFY SAVED DATA AND CHECK EXPECTED CLEANUP
# =========================================================

saved_primary = pd.read_parquet(PRIMARY_PARQUET)

assert saved_primary.shape == primary.shape
assert saved_primary.columns.tolist() == [
    "source_row_index", "Category", "Text"
]
assert saved_primary["Category"].notna().all()
assert saved_primary["Text"].notna().all()
assert saved_primary["Category"].astype(str).str.strip().ne("").all()
assert saved_primary["Text"].astype(str).str.strip().ne("").all()

# No repeated normalized resume text remains.
saved_text_key = (
    saved_primary["Text"]
    .astype(str)
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

assert not saved_text_key.duplicated().any()

# Verify no normalized text group has multiple categories.
check_groups = pd.DataFrame({
    "text_key": saved_text_key,
    "category": saved_primary["Category"].astype(str).str.strip().str.casefold()
})

assert (
    check_groups.groupby("text_key")["category"].nunique().max() == 1
)

# Save metadata documenting the rules and actual counts.
metadata = {
    "dataset_id": "ahmedheakl/resume-atlas",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "raw_records": int(len(raw_df)),
    "conflicting_text_groups_quarantined": int(len(conflict_groups_df)),
    "records_quarantined_or_invalid": int(excluded_mask.sum()),
    "same_label_duplicate_records_removed": int(duplicate_drop_mask.sum()),
    "primary_records": int(len(saved_primary)),
    "primary_categories": int(saved_primary["Category"].nunique()),
    "columns": saved_primary.columns.tolist(),
    "text_cleaning_applied": False,
    "deduplication": "case-insensitive, whitespace-normalized text",
    "conflict_policy": "quarantine complete conflicting groups; do not invent labels",
    "raw_dataset_modified": False,
    "primary_dataset_path": str(PRIMARY_PARQUET),
    "csv_copy_path": str(PRIMARY_CSV),
    "source_row_index_note": (
        "Audit metadata only; must not be used as a model input feature."
    )
}

with open(
    METADATA_DIR / "primary_dataset_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Save project progress for recovery after a session restart.
status = f"""# ResumeClassification — Project Status

## Current phase
Primary dataset preparation.

## Dataset source
Hugging Face: ahmedheakl/resume-atlas

## Persistent artifacts
- data/raw/resume_atlas_raw.parquet: Original downloaded data, unchanged.
- data/raw/resume_atlas_raw.csv: Original CSV copy.
- data/processed/primary_dataset.parquet: Primary deduplicated dataset.
- data/processed/primary_dataset.csv: CSV copy of primary dataset.
- metadata/dataset_metadata.json: Raw dataset metadata.
- metadata/primary_dataset_metadata.json: Primary dataset preparation details.
- reports/environment_report.json: Python/GPU/system information.
- reports/initial_data_audit.json: Initial data-quality findings.
- reports/preprocessing_diagnostics.json: Stop-word and technical-term diagnostics.
- reports/primary_conflict_groups.csv: Conflicting duplicate groups, metadata only.
- reports/primary_excluded_rows.csv: Exclusion reasons and source row indices.
- reports/primary_deduplicated_rows.csv: Dropped duplicates and retained row IDs.
- reports/primary_category_distribution.csv: Primary category counts.

## Current preparation rules
- Raw text retained without additional cleaning.
- Identical normalized text with conflicting labels is quarantined as a complete group.
- Repeated normalized text with a consistent label is deduplicated; first source row retained.
- Source row index is tracking metadata, not a model feature.

## Next steps
1. Verify primary dataset counts and category distribution.
2. Investigate near-duplicates before splitting.
3. Create leakage-aware train/validation/test splits and persist them.
4. Fine-tune models only after the split artifacts are verified.

## Important limitation
Resume Atlas appears already lowercased, punctuation-stripped, and stop-word-stripped.
Lost information cannot reliably be reconstructed from this dataset.
"""

(PROJECT_DIR / "PROJECT_STATUS.md").write_text(
    status, encoding="utf-8"
)

# =========================================================
# 8. PRINT A CHECKPOINT REPORT
# =========================================================

print("=" * 65)
print("PRIMARY DATASET CREATION REPORT")
print("=" * 65)

print("Raw records:", len(raw_df))
print("Conflicting text groups:", len(conflict_groups_df))
print("Records excluded/quarantined:", int(excluded_mask.sum()))
print("Consistent-label duplicates removed:", int(duplicate_drop_mask.sum()))
print("Primary records:", len(saved_primary))
print("Primary categories:", saved_primary["Category"].nunique())
print("Repeated text remaining:", int(saved_text_key.duplicated().sum()))

print("\nPrimary category distribution — 10 smallest:")
print(primary_category_counts.tail(10).to_string(index=False))

print("\nSaved and verified:")
print(PRIMARY_PARQUET)
print(PRIMARY_CSV)
print(METADATA_DIR / "primary_dataset_metadata.json")
print(REPORT_DIR / "primary_conflict_groups.csv")
print(PROJECT_DIR / "PROJECT_STATUS.md")

print("\nSUCCESS: Primary dataset saved; raw dataset unchanged.")

In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# ======================================================
# 1. RESTORE THE SAVED PRIMARY DATASET
# ======================================================

PROJECT_DIR = Path("/kaggle/working/resume_classification")
PRIMARY_PATH = PROJECT_DIR / "data/processed/primary_dataset.parquet"
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not PRIMARY_PATH.exists():
    raise FileNotFoundError(f"Missing primary dataset: {PRIMARY_PATH}")

df = pd.read_parquet(PRIMARY_PATH)

assert len(df) == 11916, f"Unexpected row count: {len(df)}"
assert {"source_row_index", "Category", "Text"}.issubset(df.columns)

texts = df["Text"].fillna("").astype(str)
labels = df["Category"].astype(str).str.strip()

print("Primary dataset restored:", df.shape)

# ======================================================
# 2. BUILD CHARACTER N-GRAM VECTORS
# ======================================================
# Character n-grams can identify similar text even when
# punctuation is missing or a few characters differ.
# These vectors are for auditing only, not model training.

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.98,
    max_features=200_000,
    sublinear_tf=True,
    dtype=np.float32
)

X = vectorizer.fit_transform(texts)

print("TF-IDF matrix shape:", X.shape)
print("TF-IDF nonzero values:", X.nnz)

# ======================================================
# 3. FIND EACH RESUME'S 10 NEAREST NEIGHBOURS
# ======================================================
# Similarity is 1 - cosine distance.
# Self-matches are ignored.
# This is a candidate audit, not an exhaustive all-pairs
# comparison.

k = min(11, len(df))

nn = NearestNeighbors(
    n_neighbors=k,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nn.fit(X)
distances, indices = nn.kneighbors(X)

pairs = {}

for i in range(len(df)):
    for distance, j in zip(distances[i], indices[i]):
        if i == j:
            continue

        # Keep each unordered pair once.
        a, b = sorted((i, int(j)))
        similarity = float(1.0 - distance)

        # Retain matches worth investigating.
        if similarity < 0.80:
            continue

        key = (a, b)

        if key not in pairs or similarity > pairs[key]:
            pairs[key] = similarity

print("Candidate pairs with similarity >= 0.80:", len(pairs))

# ======================================================
# 4. CREATE A PRIVACY-CONSCIOUS REVIEW REPORT
# ======================================================

records = []

for (a, b), similarity in pairs.items():
    text_a = texts.iloc[a]
    text_b = texts.iloc[b]

    label_a = labels.iloc[a]
    label_b = labels.iloc[b]

    length_a = len(text_a)
    length_b = len(text_b)

    if similarity >= 0.93:
        band = "Very high similarity"
    elif similarity >= 0.85:
        band = "High similarity"
    else:
        band = "Possible similarity"

    records.append({
        "source_row_a": int(df.iloc[a]["source_row_index"]),
        "source_row_b": int(df.iloc[b]["source_row_index"]),
        "category_a": label_a,
        "category_b": label_b,
        "same_category": label_a == label_b,
        "cosine_similarity": round(similarity, 5),
        "similarity_band": band,
        "characters_a": length_a,
        "characters_b": length_b,
        "length_ratio": round(
            min(length_a, length_b) / max(length_a, length_b, 1), 4
        ),
        "hash_a": hashlib.sha256(text_a.encode("utf-8")).hexdigest(),
        "hash_b": hashlib.sha256(text_b.encode("utf-8")).hexdigest()
    })

columns = [
    "source_row_a", "source_row_b",
    "category_a", "category_b", "same_category",
    "cosine_similarity", "similarity_band",
    "characters_a", "characters_b", "length_ratio",
    "hash_a", "hash_b"
]

pairs_df = pd.DataFrame(records, columns=columns)

if not pairs_df.empty:
    pairs_df = pairs_df.sort_values(
        "cosine_similarity", ascending=False
    ).reset_index(drop=True)

    pairs_df.to_csv(
        REPORT_DIR / "near_duplicate_candidates.csv",
        index=False
    )

print("\nNEAR-DUPLICATE SUMMARY")
if pairs_df.empty:
    print("No candidate pairs found at the specified threshold.")
else:
    print(pairs_df["similarity_band"].value_counts().to_string())

    print("\nHigh-similarity pairs with different labels:")
    label_conflicts = pairs_df[
        (~pairs_df["same_category"])
        & (pairs_df["cosine_similarity"] >= 0.85)
    ]

    print("Count:", len(label_conflicts))

    print("\nTop 15 candidate pairs (no resume text displayed):")
    print(
        pairs_df.head(15)[[
            "source_row_a", "source_row_b",
            "category_a", "category_b",
            "cosine_similarity", "similarity_band"
        ]].to_string(index=False)
    )

    label_conflicts.to_csv(
        REPORT_DIR / "near_duplicate_label_disagreements.csv",
        index=False
    )

# ======================================================
# 5. AUDIT LENGTH AND OBVIOUS CONTACT-DATA INDICATORS
# ======================================================
# These are rough signals; they do not reliably identify
# or redact all personal data.

contact_checks = {
    "email_marker": r"\b(?:email|e-mail|gmail|hotmail|outlook)\b",
    "flattened_email_domain": (
        r"(?:gmailcom|yahoocom|hotmailcom|outlookcom|"
        r"icloudcom|emailcom)"
    ),
    "contact_marker": r"\b(?:phone|mobile|cell|telephone|contact|tel)\b",
    "long_digit_sequence": r"\d{6,}",
    "address_marker": r"\b(?:street|avenue|road|address|postcode|zip)\b"
}

privacy_report = []

for name, pattern in contact_checks.items():
    mask = texts.str.contains(pattern, case=False, regex=True, na=False)

    privacy_report.append({
        "indicator": name,
        "rows_flagged": int(mask.sum()),
        "percentage": round(float(mask.mean() * 100), 2)
    })

privacy_df = pd.DataFrame(privacy_report)
privacy_df.to_csv(
    REPORT_DIR / "privacy_indicator_audit.csv",
    index=False
)

lengths = texts.str.len()
words = texts.str.split().str.len()

# ======================================================
# 6. VERIFY STRUCTURAL INTEGRITY
# ======================================================

normalized = (
    texts.str.casefold()
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
)

category_counts = labels.value_counts()

verification = {
    "rows": int(len(df)),
    "columns": df.columns.tolist(),
    "missing_categories": int(df["Category"].isna().sum()),
    "blank_categories": int(labels.eq("").sum()),
    "missing_texts": int(df["Text"].isna().sum()),
    "blank_texts": int(texts.str.strip().eq("").sum()),
    "duplicate_normalized_texts": int(normalized.duplicated().sum()),
    "categories": int(labels.nunique()),
    "smallest_category_count": int(category_counts.min()),
    "largest_category_count": int(category_counts.max()),
    "texts_under_50_characters": int((lengths < 50).sum()),
    "texts_under_20_words": int((words < 20).sum()),
    "near_duplicate_candidate_pairs": int(len(pairs_df)),
    "near_duplicate_method": "Character TF-IDF cosine similarity; top 10 neighbours per record",
    "near_duplicate_threshold": 0.80,
    "near_duplicate_removal_performed": False,
    "dataset_modified": False
}

with open(
    REPORT_DIR / "final_structural_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(verification, f, indent=2)

print("\nPRIVACY INDICATORS")
print(privacy_df.to_string(index=False))

print("\nSTRUCTURAL VERIFICATION")
print(json.dumps(verification, indent=2))

assert verification["missing_categories"] == 0
assert verification["blank_categories"] == 0
assert verification["missing_texts"] == 0
assert verification["blank_texts"] == 0
assert verification["duplicate_normalized_texts"] == 0
assert verification["categories"] == 43

print("\nSUCCESS: Structural checks passed.")
print("Primary dataset remains unchanged.")
print("Reports saved under:", REPORT_DIR)

In [ ]:
from pathlib import Path
import re
import json

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/kaggle/working/resume_classification")
PRIMARY_PATH = PROJECT_DIR / "data/processed/primary_dataset.parquet"
REPORT_DIR = PROJECT_DIR / "reports"

pairs_path = REPORT_DIR / "near_duplicate_label_disagreements.csv"

if not PRIMARY_PATH.exists() or not pairs_path.exists():
    raise FileNotFoundError(
        "The primary dataset or near-duplicate disagreement report is missing."
    )

df = pd.read_parquet(PRIMARY_PATH)
pairs = pd.read_csv(pairs_path)

# Map original source row indices to the retained primary records.
record_lookup = df.set_index("source_row_index")

def normalize_text(value):
    return re.sub(r"\s+", " ", str(value).casefold()).strip()

def token_jaccard(a, b):
    # Set-based overlap: measures shared distinct tokens.
    tokens_a = set(normalize_text(a).split())
    tokens_b = set(normalize_text(b).split())

    if not tokens_a and not tokens_b:
        return 1.0

    return len(tokens_a & tokens_b) / len(tokens_a | tokens_b)

def length_ratio(a, b):
    la, lb = len(a), len(b)
    return min(la, lb) / max(la, lb, 1)

results = []

for _, pair in pairs.iterrows():
    source_a = int(pair["source_row_a"])
    source_b = int(pair["source_row_b"])

    if source_a not in record_lookup.index or source_b not in record_lookup.index:
        raise ValueError(f"Source row not found: {source_a}, {source_b}")

    row_a = record_lookup.loc[source_a]
    row_b = record_lookup.loc[source_b]

    text_a = normalize_text(row_a["Text"])
    text_b = normalize_text(row_b["Text"])

    tokens_a = set(text_a.split())
    tokens_b = set(text_b.split())

    intersection = tokens_a & tokens_b

    results.append({
        "source_row_a": source_a,
        "source_row_b": source_b,
        "category_a": row_a["Category"],
        "category_b": row_b["Category"],
        "tfidf_char_similarity": float(pair["cosine_similarity"]),
        "normalized_text_exact_match": text_a == text_b,
        "token_jaccard": round(token_jaccard(text_a, text_b), 4),
        "length_ratio": round(length_ratio(text_a, text_b), 4),
        "length_a": len(text_a),
        "length_b": len(text_b),
        "shared_unique_tokens": len(intersection),
        "category_disagreement": row_a["Category"] != row_b["Category"]
    })

review = pd.DataFrame(results)

review = review.sort_values(
    ["tfidf_char_similarity", "token_jaccard"],
    ascending=False
).reset_index(drop=True)

# These are review tiers, not automatic deletion decisions.
review["review_tier"] = np.select(
    [
        (review["tfidf_char_similarity"] >= 0.93)
        & (review["token_jaccard"] >= 0.80),
        (review["tfidf_char_similarity"] >= 0.85)
        & (review["token_jaccard"] >= 0.60)
    ],
    ["High-priority review", "Medium-priority review"],
    default="Manual review"
)

output_path = REPORT_DIR / "label_disagreement_validation.csv"
review.to_csv(output_path, index=False)

print("Label-disagreement pairs validated:", len(review))
print("Exact normalized text matches:", int(
    review["normalized_text_exact_match"].sum()
))

print("\nReview-tier counts:")
print(review["review_tier"].value_counts().to_string())

print("\nTop 20 pairs for investigation:")
print(review.head(20)[[
    "source_row_a",
    "source_row_b",
    "category_a",
    "category_b",
    "tfidf_char_similarity",
    "token_jaccard",
    "length_ratio",
    "normalized_text_exact_match",
    "review_tier"
]].to_string(index=False))

print("\nSaved review report:", output_path)
print("Primary dataset unchanged.")

## 5. Prepare and verify train/validation/test splits

Create grouped, stratified splits so duplicate or closely connected records do not leak across sets. Confirm that all 43 labels occur in each split and save split metadata.

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib
import json
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

# =========================================================
# 1. RESTORE SAVED FILES
# =========================================================

PROJECT_DIR = Path("/kaggle/working/resume_classification")
PRIMARY_PATH = PROJECT_DIR / "data/processed/primary_dataset.parquet"
NEAR_DUP_PATH = PROJECT_DIR / "reports/near_duplicate_candidates.csv"

PROCESSED_DIR = PROJECT_DIR / "data/processed"
SPLIT_DIR = PROJECT_DIR / "data/splits"
REPORT_DIR = PROJECT_DIR / "reports"
METADATA_DIR = PROJECT_DIR / "metadata"

for folder in [PROCESSED_DIR, SPLIT_DIR, REPORT_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not PRIMARY_PATH.exists():
    raise FileNotFoundError(f"Missing primary dataset: {PRIMARY_PATH}")

if not NEAR_DUP_PATH.exists():
    raise FileNotFoundError(f"Missing near-duplicate report: {NEAR_DUP_PATH}")

df = pd.read_parquet(PRIMARY_PATH)

assert {"source_row_index", "Category", "Text"}.issubset(df.columns)
assert df["Category"].notna().all()
assert df["Text"].notna().all()

print("Restored primary dataset:", df.shape)

# =========================================================
# 2. CONSERVATIVE TEXT PREPROCESSING
# =========================================================
# The source text has already lost punctuation and stop words.
# Do not attempt to reconstruct them.
#
# We redact recognizable emails, URLs and phone-like strings.
# We do not attempt automatic name/address removal because
# unreliable redaction can damage text and leave PII behind.

def prepare_text(value):
    text = str(value)

    # Remove email-like strings, including flattened email domains
    # where a domain is still recognizable.
    text = re.sub(
        r"\b[\w.+%-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        " [EMAIL] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b[\w.+%-]+(?:gmailcom|yahoocom|hotmailcom|outlookcom|icloudcom)\b",
        " [EMAIL] ",
        text,
        flags=re.IGNORECASE
    )

    # URLs and common flattened URL patterns
    text = re.sub(
        r"(?:https?://|www\.)\S+",
        " [URL] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b(?:linkedincom/in/|linkedincom)\S*",
        " [URL] ",
        text,
        flags=re.IGNORECASE
    )

    # Phone-like strings. This is approximate and cannot catch all cases.
    text = re.sub(
        r"(?<!\w)\+?\d[\d ().-]{6,}\d(?!\w)",
        " [PHONE] ",
        text
    )

    # Normalize whitespace only. Preserve remaining words and symbols.
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["Text_prepared"] = df["Text"].map(prepare_text)

# Remove records made empty by preparation, if any.
empty_after_prep = df["Text_prepared"].str.strip().eq("")
removed_empty = int(empty_after_prep.sum())

df = df.loc[~empty_after_prep].copy()
df["Category"] = df["Category"].astype(str).str.strip()

assert df["Category"].ne("").all()
assert df["Text_prepared"].str.strip().ne("").all()

# =========================================================
# 3. BUILD NEAR-DUPLICATE GROUPS
# =========================================================
# Near-duplicate pairs with cosine similarity >= 0.93 are
# grouped to reduce leakage between data splits.
#
# Cross-category pairs at >= 0.93 are conservatively quarantined.
# This does not claim that their labels are definitely wrong.

pairs = pd.read_csv(NEAR_DUP_PATH)

parent = {int(i): int(i) for i in df.index}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

source_to_index = {
    int(source): int(index)
    for index, source in zip(df.index, df["source_row_index"])
}

threshold = 0.93
accepted_edges = []

for _, pair in pairs.iterrows():
    if float(pair["cosine_similarity"]) < threshold:
        continue

    source_a = int(pair["source_row_a"])
    source_b = int(pair["source_row_b"])

    # Ignore pairs if a source record is no longer present.
    if source_a not in source_to_index or source_b not in source_to_index:
        continue

    a = source_to_index[source_a]
    b = source_to_index[source_b]

    accepted_edges.append((a, b))
    union(a, b)

components = defaultdict(list)

for idx in df.index:
    components[find(int(idx))].append(int(idx))

component_labels = {}

for root, indices in components.items():
    component_labels[root] = set(
        df.loc[indices, "Category"].astype(str)
    )

# Quarantine complete connected components containing multiple labels.
conflict_roots = {
    root
    for root, labels_in_component in component_labels.items()
    if len(labels_in_component) > 1
}

quarantine_indices = {
    idx
    for root in conflict_roots
    for idx in components[root]
}

quarantine = df.loc[
    df.index.isin(quarantine_indices),
    ["source_row_index", "Category"]
].copy()

quarantine["reason"] = (
    "Near-duplicate connected component contains multiple categories "
    f"(similarity threshold >= {threshold})"
)

eligible = df.loc[
    ~df.index.isin(quarantine_indices)
].copy()

# Group IDs ensure related high-similarity records stay together.
eligible["duplicate_group_id"] = [
    hashlib.sha256(
        "|".join(sorted(
            str(df.loc[i, "source_row_index"])
            for i in components[find(int(i))]
        )).encode("utf-8")
    ).hexdigest()[:16]
    for i in eligible.index
]

# =========================================================
# 4. CREATE LEAKAGE-RESISTANT SPLITS
# =========================================================
# GroupShuffleSplit preserves groups but does not guarantee
# exact stratification. We report the actual resulting sizes
# and category counts for review.

X = np.zeros(len(eligible))
y = eligible["Category"].to_numpy()
groups = eligible["duplicate_group_id"].to_numpy()

split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

trainval_pos, test_pos = next(
    split1.split(X, y, groups=groups)
)

trainval = eligible.iloc[trainval_pos].copy()
test = eligible.iloc[test_pos].copy()

# Split the remaining 90% into approximately 80% train and 10% validation.
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=1 / 9,
    random_state=43
)

train_pos, val_pos = next(
    split2.split(
        np.zeros(len(trainval)),
        trainval["Category"].to_numpy(),
        groups=trainval["duplicate_group_id"].to_numpy()
    )
)

train = trainval.iloc[train_pos].copy()
val = trainval.iloc[val_pos].copy()

# =========================================================
# 5. VERIFY NO GROUP LEAKAGE
# =========================================================

train_groups = set(train["duplicate_group_id"])
val_groups = set(val["duplicate_group_id"])
test_groups = set(test["duplicate_group_id"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

# Verify every category appears in training.
all_categories = set(eligible["Category"].unique())
train_categories = set(train["Category"].unique())

missing_train_categories = sorted(all_categories - train_categories)

# =========================================================
# 6. SAVE FINAL DATASETS
# =========================================================

output_columns = [
    "source_row_index",
    "Category",
    "Text_prepared",
    "duplicate_group_id"
]

train[output_columns].to_parquet(
    SPLIT_DIR / "train.parquet", index=False
)
val[output_columns].to_parquet(
    SPLIT_DIR / "validation.parquet", index=False
)
test[output_columns].to_parquet(
    SPLIT_DIR / "test.parquet", index=False
)

quarantine.to_csv(
    REPORT_DIR / "near_duplicate_quarantine.csv",
    index=False
)

# Save category distributions
for name, frame in [
    ("train", train),
    ("validation", val),
    ("test", test)
]:
    (
        frame["Category"]
        .value_counts()
        .rename_axis("Category")
        .reset_index(name="record_count")
        .to_csv(REPORT_DIR / f"{name}_category_distribution.csv", index=False)
    )

# =========================================================
# 7. SAVE METADATA AND RECOVERY NOTES
# =========================================================

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dataset": str(PRIMARY_PATH),
    "near_duplicate_report": str(NEAR_DUP_PATH),
    "primary_input_records": int(len(df)),
    "empty_after_preparation_removed": removed_empty,
    "near_duplicate_similarity_threshold": threshold,
    "near_duplicate_edges_grouped": int(len(accepted_edges)),
    "near_duplicate_components_quarantined": int(len(conflict_roots)),
    "records_quarantined": int(len(quarantine)),
    "eligible_records": int(len(eligible)),
    "train_records": int(len(train)),
    "validation_records": int(len(val)),
    "test_records": int(len(test)),
    "train_categories": int(train["Category"].nunique()),
    "validation_categories": int(val["Category"].nunique()),
    "test_categories": int(test["Category"].nunique()),
    "missing_categories_in_training": missing_train_categories,
    "random_seeds": {"train_test": 42, "train_validation": 43},
    "split_method": "GroupShuffleSplit; groups kept intact",
    "text_preparation": [
        "Recognizable email redaction",
        "Recognizable URL redaction",
        "Approximate phone-pattern redaction",
        "Whitespace normalization"
    ],
    "known_limitations": [
        "Automatic name/address redaction is not comprehensive",
        "Source punctuation and stop words were already absent",
        "Group splits do not guarantee exact category stratification",
        "Near-duplicate threshold is heuristic and requires review"
    ]
}

with open(
    METADATA_DIR / "final_split_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Update the recovery file
status_path = PROJECT_DIR / "PROJECT_STATUS.md"
with open(status_path, "a", encoding="utf-8") as f:
    f.write(
        "\n\n## Final preparation checkpoint\n"
        "- Prepared split files: data/splits/train.parquet, "
        "data/splits/validation.parquet, data/splits/test.parquet\n"
        "- Split metadata: metadata/final_split_metadata.json\n"
        "- Near-duplicate quarantine: reports/near_duplicate_quarantine.csv\n"
        "- Review category distributions before training.\n"
    )

# =========================================================
# 8. RELOAD FILES AND VERIFY PERSISTENCE
# =========================================================

saved_train = pd.read_parquet(SPLIT_DIR / "train.parquet")
saved_val = pd.read_parquet(SPLIT_DIR / "validation.parquet")
saved_test = pd.read_parquet(SPLIT_DIR / "test.parquet")

assert len(saved_train) == len(train)
assert len(saved_val) == len(val)
assert len(saved_test) == len(test)

assert (
    set(saved_train["duplicate_group_id"]).isdisjoint(
        set(saved_val["duplicate_group_id"])
    )
)
assert (
    set(saved_train["duplicate_group_id"]).isdisjoint(
        set(saved_test["duplicate_group_id"])
    )
)
assert (
    set(saved_val["duplicate_group_id"]).isdisjoint(
        set(saved_test["duplicate_group_id"])
    )
)

print("=" * 65)
print("FINAL DATA PREPARATION REPORT")
print("=" * 65)
print("Eligible records:", len(eligible))
print("Quarantined records:", len(quarantine))
print("Train:", len(saved_train))
print("Validation:", len(saved_val))
print("Test:", len(saved_test))
print("Train categories:", saved_train["Category"].nunique())
print("Validation categories:", saved_val["Category"].nunique())
print("Test categories:", saved_test["Category"].nunique())
print("Categories missing from training:", missing_train_categories)

print("\nSaved and reloaded:")
print(SPLIT_DIR / "train.parquet")
print(SPLIT_DIR / "validation.parquet")
print(SPLIT_DIR / "test.parquet")
print(METADATA_DIR / "final_split_metadata.json")

print("\nGroup leakage checks passed.")
print("Review category distributions before training.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# =========================================================
# 1. PATHS AND RESTORE EXISTING SPLITS
# =========================================================

PROJECT = Path("/kaggle/working/resume_classification")
SPLITS = PROJECT / "data/splits"
REPORTS = PROJECT / "reports"
METADATA = PROJECT / "metadata"
PROCESSED = PROJECT / "data/processed"

for folder in [SPLITS, REPORTS, METADATA, PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

split_paths = {
    "train": SPLITS / "train.parquet",
    "validation": SPLITS / "validation.parquet",
    "test": SPLITS / "test.parquet",
}

for name, path in split_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} split: {path}")

parts = [pd.read_parquet(split_paths[name])
         for name in ["train", "validation", "test"]]

data = pd.concat(parts, ignore_index=True)

required = {
    "source_row_index",
    "Category",
    "Text_prepared",
    "duplicate_group_id",
}

assert required.issubset(data.columns)
assert data["source_row_index"].is_unique
assert data["Category"].notna().all()
assert data["Text_prepared"].notna().all()

print("Loaded existing split records:", len(data))

# Backup the existing split files before finalization.
BACKUP = PROJECT / "data/pre_final_split_backup"
BACKUP.mkdir(parents=True, exist_ok=True)

for name, path in split_paths.items():
    shutil.copy2(path, BACKUP / path.name)

print("Backup saved to:", BACKUP)

# =========================================================
# 2. REBUILD GROUPS USING EXISTING DUPLICATE GROUPS AND
#    EXACT TEXT AFTER PRIVACY PREPARATION
# =========================================================

n = len(data)
parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Keep all members of each previously detected near-duplicate group together.
for positions in data.groupby(
    "duplicate_group_id", sort=False
).indices.values():
    positions = list(positions)
    first = positions[0]
    for position in positions[1:]:
        union(first, position)

def normalize_prepared_text(text):
    return " ".join(str(text).casefold().split())

data["_normalized_prepared"] = data["Text_prepared"].map(
    normalize_prepared_text
)

# Texts that become identical after redaction must not cross splits.
text_first_position = {}

for pos, key in enumerate(data["_normalized_prepared"]):
    if not key:
        continue

    if key in text_first_position:
        union(pos, text_first_position[key])
    else:
        text_first_position[key] = pos

# Find connected components.
components = defaultdict(list)

for pos in range(n):
    components[find(pos)].append(pos)

print("Connected groups after final checks:", len(components))

# =========================================================
# 3. QUARANTINE GROUPS WITH CONFLICTING LABELS
# =========================================================

quarantined_positions = set()
quarantine_records = []

for root, positions in components.items():
    labels = sorted(
        data.iloc[positions]["Category"].astype(str).unique()
    )

    if len(labels) > 1:
        quarantined_positions.update(positions)

        for pos in positions:
            row = data.iloc[pos]
            quarantine_records.append({
                "source_row_index": int(row["source_row_index"]),
                "Category": row["Category"],
                "component_size": len(positions),
                "categories_in_component": " | ".join(labels),
                "reason": (
                    "Related or identical prepared texts have conflicting labels"
                ),
            })

quarantine = pd.DataFrame(quarantine_records)

# Create a stable ID for every surviving connected group.
component_ids = {}

for root, positions in components.items():
    source_ids = sorted(
        str(data.iloc[pos]["source_row_index"])
        for pos in positions
    )

    component_ids[root] = hashlib.sha256(
        "|".join(source_ids).encode("utf-8")
    ).hexdigest()[:16]

data["_component_root"] = [
    find(pos) for pos in range(n)
]

eligible = data.loc[
    ~data.index.isin(quarantined_positions)
].copy()

eligible["duplicate_group_id"] = eligible["_component_root"].map(
    component_ids
)

# Remove the internal bookkeeping column.
eligible.drop(columns=["_component_root"], inplace=True)

# =========================================================
# 4. DEDUPLICATE EXACT PREPARED TEXT WITHIN THE ELIGIBLE DATA
# =========================================================

# Cross-label identical texts have already been quarantined.
# For remaining identical inputs, keep one copy and record the
# dropped source row and retained row.
duplicated_mask = eligible["_normalized_prepared"].duplicated(
    keep="first"
)

kept_lookup = (
    eligible.loc[~duplicated_mask]
    .set_index("_normalized_prepared")["source_row_index"]
    .to_dict()
)

dedup_audit = eligible.loc[
    duplicated_mask,
    ["source_row_index", "Category", "_normalized_prepared"]
].copy()

dedup_audit["retained_source_row_index"] = (
    dedup_audit["_normalized_prepared"].map(kept_lookup)
)
dedup_audit["reason"] = (
    "Identical prepared text with the same label; first copy retained"
)

dedup_audit.drop(
    columns=["_normalized_prepared"], inplace=True
)

eligible = eligible.loc[~duplicated_mask].copy()
eligible.reset_index(drop=True, inplace=True)

# Save the final prepared dataset for reuse.
final_columns = [
    "source_row_index",
    "Category",
    "Text_prepared",
    "duplicate_group_id",
]

eligible[final_columns].to_parquet(
    PROCESSED / "final_prepared_dataset.parquet",
    index=False
)

quarantine.to_csv(
    REPORTS / "final_quarantine.csv", index=False
)
dedup_audit.to_csv(
    REPORTS / "post_redaction_duplicates_removed.csv",
    index=False
)

print("Eligible records after final checks:", len(eligible))
print("New conflicting-component records quarantined:",
      len(quarantined_positions))
print("Identical prepared-text duplicates removed:", len(dedup_audit))

# =========================================================
# 5. GROUP-STRATIFIED SPLITS, APPROXIMATELY 80/10/10
# =========================================================

category_group_counts = (
    eligible.groupby("Category")["duplicate_group_id"].nunique()
)

min_groups_per_category = int(category_group_counts.min())

if min_groups_per_category < 3:
    raise RuntimeError(
        "At least one category has fewer than 3 independent groups. "
        "Inspect category_group_counts before splitting."
    )

# Use 10 folds where possible; otherwise use the number of
# independent groups in the rarest category.
n_splits = min(10, min_groups_per_category)

X = np.zeros((len(eligible), 1))
y = eligible["Category"].astype(str).to_numpy()
groups = eligible["duplicate_group_id"].to_numpy()

all_categories = set(eligible["Category"].astype(str).unique())

# Search deterministic seeds for folds that retain all categories
# in train, validation and test, where feasible.
chosen = None

for seed in range(42, 142):
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    fold_assignment = np.full(len(eligible), -1, dtype=int)

    for fold, (_, fold_test_idx) in enumerate(
        splitter.split(X, y, groups)
    ):
        fold_assignment[fold_test_idx] = fold

    test_mask = fold_assignment == 0
    val_mask = fold_assignment == 1
    train_mask = fold_assignment >= 2

    if not (train_mask.any() and val_mask.any() and test_mask.any()):
        continue

    train_classes = set(y[train_mask])
    val_classes = set(y[val_mask])
    test_classes = set(y[test_mask])

    if (
        train_classes == all_categories
        and val_classes == all_categories
        and test_classes == all_categories
    ):
        chosen = (
            seed, fold_assignment,
            train_mask, val_mask, test_mask
        )
        break

if chosen is None:
    raise RuntimeError(
        "Could not find group-stratified folds with all categories "
        "in every split. Review rare-category group counts before training."
    )

seed, fold_assignment, train_mask, val_mask, test_mask = chosen

train = eligible.loc[train_mask, final_columns].copy()
validation = eligible.loc[val_mask, final_columns].copy()
test = eligible.loc[test_mask, final_columns].copy()

# =========================================================
# 6. VERIFY SPLITS BEFORE SAVING
# =========================================================

def normalized_text_set(frame):
    return set(
        frame["Text_prepared"].map(normalize_prepared_text)
    )

for left_name, left in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    assert left["source_row_index"].is_unique
    assert left["Category"].notna().all()
    assert left["Text_prepared"].str.strip().ne("").all()

# No source-row IDs can overlap.
assert set(train["source_row_index"]).isdisjoint(
    set(validation["source_row_index"])
)
assert set(train["source_row_index"]).isdisjoint(
    set(test["source_row_index"])
)
assert set(validation["source_row_index"]).isdisjoint(
    set(test["source_row_index"])
)

# No duplicate groups can cross splits.
assert set(train["duplicate_group_id"]).isdisjoint(
    set(validation["duplicate_group_id"])
)
assert set(train["duplicate_group_id"]).isdisjoint(
    set(test["duplicate_group_id"])
)
assert set(validation["duplicate_group_id"]).isdisjoint(
    set(test["duplicate_group_id"])
)

# No exact normalized prepared text can cross splits.
assert normalized_text_set(train).isdisjoint(
    normalized_text_set(validation)
)
assert normalized_text_set(train).isdisjoint(
    normalized_text_set(test)
)
assert normalized_text_set(validation).isdisjoint(
    normalized_text_set(test)
)

assert set(train["Category"].unique()) == all_categories
assert set(validation["Category"].unique()) == all_categories
assert set(test["Category"].unique()) == all_categories

# =========================================================
# 7. SAVE FINAL SPLITS
# =========================================================

for name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    frame.to_parquet(SPLITS / f"{name}.parquet", index=False)
    frame.to_csv(SPLITS / f"{name}.csv", index=False)

# Stable numeric label mapping.
labels_sorted = sorted(all_categories)
label_mapping = {
    label: idx for idx, label in enumerate(labels_sorted)
}

with open(
    METADATA / "label_mapping.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

# Category-distribution report.
distribution = pd.DataFrame(index=labels_sorted)

for name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    distribution[name] = (
        frame["Category"].value_counts().reindex(labels_sorted, fill_value=0)
    )

distribution["total"] = distribution.sum(axis=1)
distribution.to_csv(
    REPORTS / "final_split_category_counts.csv",
    index_label="Category"
)

# =========================================================
# 8. SAVE METADATA AND RECOVERY STATUS
# =========================================================

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_primary_dataset": str(
        PROJECT / "data/processed/primary_dataset.parquet"
    ),
    "final_prepared_dataset": str(
        PROCESSED / "final_prepared_dataset.parquet"
    ),
    "train_path": str(SPLITS / "train.parquet"),
    "validation_path": str(SPLITS / "validation.parquet"),
    "test_path": str(SPLITS / "test.parquet"),
    "train_records": len(train),
    "validation_records": len(validation),
    "test_records": len(test),
    "total_final_records": len(eligible),
    "quarantined_records": len(quarantine),
    "post_redaction_duplicates_removed": len(dedup_audit),
    "categories": len(all_categories),
    "split_method": "StratifiedGroupKFold; group leakage prevented",
    "fold_count": n_splits,
    "random_seed": seed,
    "verified_source_row_separation": True,
    "verified_group_separation": True,
    "verified_prepared_text_separation": True,
    "all_categories_in_each_split": True,
    "privacy_limitation": (
        "Regex-based redaction is incomplete; names and physical addresses "
        "may remain. Do not describe this data as fully anonymized."
    ),
    "known_data_limitation": (
        "Source text had already lost punctuation and common stop words. "
        "These cannot reliably be reconstructed."
    ),
}

with open(
    METADATA / "final_split_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Update project handover notes for the next session.
status = f"""# ResumeClassification — Current Project Status

## Dataset preparation completed
Source: ahmedheakl/resume-atlas

Original raw data:
- data/raw/resume_atlas_raw.parquet
- data/raw/resume_atlas_raw.csv

Primary dataset:
- data/processed/primary_dataset.parquet

Final prepared dataset:
- data/processed/final_prepared_dataset.parquet

Final splits:
- data/splits/train.parquet
- data/splits/validation.parquet
- data/splits/test.parquet
- CSV copies stored alongside each Parquet split

Audit reports:
- reports/initial_data_audit.json
- reports/preprocessing_diagnostics.json
- reports/near_duplicate_candidates.csv
- reports/label_disagreement_validation.csv
- reports/final_quarantine.csv
- reports/post_redaction_duplicates_removed.csv
- reports/final_split_category_counts.csv

Metadata:
- metadata/primary_dataset_metadata.json
- metadata/final_split_metadata.json
- metadata/label_mapping.json

Current split sizes:
- Train: {len(train)}
- Validation: {len(validation)}
- Test: {len(test)}
- Categories: {len(all_categories)}

Split integrity:
- Source row IDs do not overlap.
- Existing near-duplicate groups do not cross splits.
- Exact prepared text does not cross splits.
- All categories are represented in each split.

Important limitations:
- Contact-detail redaction is not complete anonymization.
- Punctuation and stop words missing from source text cannot be recovered.
- Near-duplicate links were generated by a top-neighbour heuristic.
- Dataset provenance and permissions should be reviewed before publishing data.

Next phase:
1. Reload saved train/validation/test Parquet files.
2. Verify tokenizer/model sequence-length strategy.
3. Fine-tune BERT, DistilBERT, and ModernBERT separately.
4. Save model checkpoints, evaluation reports, and experiment metadata.
"""

(PROJECT / "PROJECT_STATUS.md").write_text(status, encoding="utf-8")

# Reload the persisted files and verify row counts.
for name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    restored = pd.read_parquet(SPLITS / f"{name}.parquet")
    assert len(restored) == len(frame)
    assert restored["Category"].nunique() == len(all_categories)

print("\n" + "=" * 65)
print("FINAL DATASET AND SPLIT VERIFICATION")
print("=" * 65)

print("Final prepared dataset:", len(eligible))
print("Quarantined records:", len(quarantine))
print("Additional exact prepared-text duplicates removed:", len(dedup_audit))
print("Categories:", len(all_categories))
print(f"Train: {len(train)} ({len(train)/len(eligible):.2%})")
print(f"Validation: {len(validation)} ({len(validation)/len(eligible):.2%})")
print(f"Test: {len(test)} ({len(test)/len(eligible):.2%})")

print("\nSmallest category counts in each split:")
print(distribution[["train", "validation", "test"]].min().to_string())

print("\nVerified:")
print("- No source-row overlap")
print("- No duplicate-group overlap")
print("- No exact prepared-text overlap")
print("- All 43 categories present in every split")
print("- Files saved and reloaded successfully")

print("\nSaved label mapping:", METADATA / "label_mapping.json")
print("Saved split metadata:", METADATA / "final_split_metadata.json")
print("Saved handover notes:", PROJECT / "PROJECT_STATUS.md")
print("\nFinal data preparation completed.")

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/kaggle/working/resume_classification")
SPLITS = PROJECT / "data/splits"
REPORTS = PROJECT / "reports"
METADATA = PROJECT / "metadata"

train = pd.read_parquet(SPLITS / "train.parquet")
val = pd.read_parquet(SPLITS / "validation.parquet")
test = pd.read_parquet(SPLITS / "test.parquet")

with open(METADATA / "label_mapping.json", encoding="utf-8") as f:
    label_mapping = json.load(f)

# Verify consistent columns and labels
assert list(train.columns) == list(val.columns) == list(test.columns)

all_labels = (
    set(train["Category"])
    | set(val["Category"])
    | set(test["Category"])
)

assert len(all_labels) == 43
assert set(label_mapping.keys()) == all_labels
assert sorted(label_mapping.values()) == list(range(43))

# Report each category's support across all splits
counts = pd.DataFrame({
    "train": train["Category"].value_counts(),
    "validation": val["Category"].value_counts(),
    "test": test["Category"].value_counts(),
}).fillna(0).astype(int)

counts["total"] = counts.sum(axis=1)
counts = counts.sort_values("total")

counts.to_csv(
    REPORTS / "final_category_support_check.csv",
    index_label="Category"
)

print("Train / validation / test:", len(train), len(val), len(test))
print("Categories:", len(all_labels))

print("\n10 categories with the fewest records:")
print(counts.head(10).to_string())

print("\nCategories with fewer than 5 validation examples:")
print(counts[counts["validation"] < 5][["validation", "total"]])

print("\nCategories with fewer than 5 test examples:")
print(counts[counts["test"] < 5][["test", "total"]])

print("\nLabel mapping verified and category-support report saved.")

## 6. Tokenizer setup and sequence-length audit

Prepare the BERT, DistilBERT, and ModernBERT tokenizers and audit sequence lengths before selecting each model's input limit. The deployed ModernBERT configuration uses 2,048 tokens.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
import numpy as np

PROJECT = Path("/kaggle/working/resume_classification")
SPLITS = PROJECT / "data/splits"
MODEL_DIR = PROJECT / "models/tokenizers"
REPORTS = PROJECT / "reports"
METADATA = PROJECT / "metadata"

for p in [MODEL_DIR, REPORTS, METADATA]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLITS / "train.parquet"
VAL_PATH = SPLITS / "validation.parquet"
TEST_PATH = SPLITS / "test.parquet"
LABEL_PATH = METADATA / "label_mapping.json"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, LABEL_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required saved artifact missing: {p}")

train = pd.read_parquet(TRAIN_PATH)
val = pd.read_parquet(VAL_PATH)
test = pd.read_parquet(TEST_PATH)

with open(LABEL_PATH, encoding="utf-8") as f:
    label_mapping = json.load(f)

assert len(label_mapping) == 43
assert train["Category"].nunique() == 43
assert val["Category"].nunique() == 43
assert test["Category"].nunique() == 43

# Initial fair-comparison configuration
MODEL_CONFIGS = {
    "bert": {
        "model_id": "google-bert/bert-base-uncased",
        "max_length": 512
    },
    "distilbert": {
        "model_id": "distilbert/distilbert-base-uncased",
        "max_length": 512
    },
    "modernbert": {
        "model_id": "answerdotai/ModernBERT-base",
        "max_length": 512
    }
}

config_path = METADATA / "training_plan.json"

training_plan = {
    "project": "ResumeClassification",
    "dataset": "Resume Atlas",
    "label_count": len(label_mapping),
    "label_mapping_path": str(LABEL_PATH),
    "splits": {
        "train": str(TRAIN_PATH),
        "validation": str(VAL_PATH),
        "test": str(TEST_PATH)
    },
    "split_sizes": {
        "train": len(train),
        "validation": len(val),
        "test": len(test)
    },
    "models": MODEL_CONFIGS,
    "initial_max_sequence_length": 512,
    "padding": "dynamic padding per batch during training",
    "truncation": True,
    "metrics": [
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "per_class_precision_recall_f1",
        "confusion_matrix"
    ],
    "selection_rule": (
        "Select the best single model primarily by validation macro-F1; "
        "use the validation set for model selection and evaluate the test "
        "set only after finalizing the selected model/configuration."
    ),
    "raw_text_limitations": [
        "Original punctuation and common stop words are already missing",
        "Regex-based personal-data removal is incomplete"
    ]
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(training_plan, f, indent=2)

print("Saved training plan:", config_path)
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Test rows:", len(test))
print("Classes:", len(label_mapping))

# Tokenizer analysis
try:
    from transformers import AutoTokenizer
except ImportError as e:
    raise RuntimeError(
        "Transformers is not installed. Run %pip install -q transformers "
        "in a separate cell, then rerun this cell."
    ) from e

texts = train["Text_prepared"].fillna("").astype(str).tolist()
batch_size = 32
results = []

for short_name, cfg in MODEL_CONFIGS.items():
    tokenizer_dir = MODEL_DIR / short_name
    tokenizer_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nLoading tokenizer: {cfg['model_id']}")

    # Reuse locally saved tokenizer if available.
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            str(tokenizer_dir),
            local_files_only=True
        )
        print("Loaded saved tokenizer.")
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
        tokenizer.save_pretrained(tokenizer_dir)
        print("Downloaded and saved tokenizer.")

    lengths = []
    max_length = cfg["max_length"]
    start = time.time()

    for start_idx in range(0, len(texts), batch_size):
        batch = texts[start_idx:start_idx + batch_size]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True
        )

        lengths.extend(encoded["length"])

    lengths = np.asarray(lengths)

    record = {
        "model": short_name,
        "model_id": cfg["model_id"],
        "tokenizer_class": tokenizer.__class__.__name__,
        "documents": int(len(lengths)),
        "mean_tokens": round(float(lengths.mean()), 2),
        "median_tokens": round(float(np.percentile(lengths, 50)), 2),
        "p90_tokens": round(float(np.percentile(lengths, 90)), 2),
        "p95_tokens": round(float(np.percentile(lengths, 95)), 2),
        "p99_tokens": round(float(np.percentile(lengths, 99)), 2),
        "max_tokens": int(lengths.max()),
        "over_512_tokens": int((lengths > max_length).sum()),
        "over_512_percentage": round(
            float((lengths > max_length).mean() * 100), 2
        ),
        "estimated_seconds": round(time.time() - start, 1)
    }

    results.append(record)
    print(json.dumps(record, indent=2))

token_report = pd.DataFrame(results)
token_report.to_csv(
    REPORTS / "token_length_analysis.csv",
    index=False
)

with open(
    REPORTS / "token_length_analysis.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, indent=2)

print("\nTOKEN LENGTH ANALYSIS")
print(token_report.to_string(index=False))
print("\nSaved report:", REPORTS / "token_length_analysis.csv")
print("Tokenizer files saved under:", MODEL_DIR)
print("No model training performed.")

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd

# =========================================================
# 1. PROJECT PATHS
# =========================================================

PROJECT = Path("/kaggle/working/resume_classification")
SPLITS = PROJECT / "data/splits"
REPORTS = PROJECT / "reports"
METADATA = PROJECT / "metadata"
TOKENIZERS = PROJECT / "models/tokenizers"

for folder in [REPORTS, METADATA]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("SESSION WIND-UP AND PERSISTENCE VERIFICATION")
print("=" * 65)

# =========================================================
# 2. CHECK REQUIRED FILES
# =========================================================

required_files = [
    SPLITS / "train.parquet",
    SPLITS / "validation.parquet",
    SPLITS / "test.parquet",
    METADATA / "label_mapping.json",
    METADATA / "training_plan.json",
    REPORTS / "token_length_analysis.csv",
    REPORTS / "token_length_analysis.json",
]

missing = [str(p) for p in required_files if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Required files are missing:\n" + "\n".join(missing)
    )

# Optional artifacts: report missing ones, but do not fail.
optional_files = [
    PROJECT / "data/raw/resume_atlas_raw.parquet",
    PROJECT / "data/processed/primary_dataset.parquet",
    PROJECT / "data/processed/final_prepared_dataset.parquet",
    METADATA / "final_split_metadata.json",
    REPORTS / "final_split_verification.json",
    REPORTS / "final_split_category_counts.csv",
    PROJECT / "PROJECT_STATUS.md",
]

print("\nOPTIONAL ARTIFACT STATUS")
for path in optional_files:
    print(f"{'FOUND' if path.is_file() else 'NOT FOUND'}: {path}")

# =========================================================
# 3. RELOAD AND VERIFY SAVED SPLITS
# =========================================================

train_path = SPLITS / "train.parquet"
val_path = SPLITS / "validation.parquet"
test_path = SPLITS / "test.parquet"

train = pd.read_parquet(train_path)
val = pd.read_parquet(val_path)
test = pd.read_parquet(test_path)

with open(METADATA / "label_mapping.json", encoding="utf-8") as f:
    label_mapping = json.load(f)

expected_columns = {
    "source_row_index",
    "Category",
    "Text_prepared",
    "duplicate_group_id",
}

for name, frame in [
    ("train", train),
    ("validation", val),
    ("test", test),
]:
    assert expected_columns.issubset(frame.columns), (
        f"Unexpected schema in {name}"
    )
    assert frame["source_row_index"].is_unique, (
        f"Duplicate source IDs within {name}"
    )
    assert frame["Category"].notna().all()
    assert frame["Text_prepared"].notna().all()
    assert frame["Text_prepared"].astype(str).str.strip().ne("").all()

all_categories = (
    set(train["Category"])
    | set(val["Category"])
    | set(test["Category"])
)

assert len(label_mapping) == 43
assert set(label_mapping.keys()) == all_categories
assert sorted(label_mapping.values()) == list(range(43))

# Confirm source rows do not cross splits.
for left_name, left, right_name, right in [
    ("train", train, "validation", val),
    ("train", train, "test", test),
    ("validation", val, "test", test),
]:
    assert set(left["source_row_index"]).isdisjoint(
        set(right["source_row_index"])
    ), f"Source-row leakage: {left_name}/{right_name}"

    assert set(left["duplicate_group_id"]).isdisjoint(
        set(right["duplicate_group_id"])
    ), f"Duplicate-group leakage: {left_name}/{right_name}"

# Confirm exact prepared-text matches do not cross splits.
def normalized_texts(frame):
    return set(
        frame["Text_prepared"]
        .astype(str)
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

for left_name, left, right_name, right in [
    ("train", train, "validation", val),
    ("train", train, "test", test),
    ("validation", val, "test", test),
]:
    overlap = normalized_texts(left) & normalized_texts(right)
    assert not overlap, (
        f"Prepared-text overlap found: {left_name}/{right_name}: "
        f"{len(overlap)}"
    )

print("\nSPLIT VERIFICATION: PASSED")
print(f"Train:      {len(train):,}")
print(f"Validation: {len(val):,}")
print(f"Test:       {len(test):,}")
print(f"Total:      {len(train) + len(val) + len(test):,}")
print(f"Categories: {len(all_categories)}")
print("Source-row overlap: none")
print("Duplicate-group overlap: none")
print("Exact prepared-text overlap: none")
print("Label mapping: valid")

# =========================================================
# 4. VERIFY TOKENIZER FILES AND TOKEN-LENGTH REPORT
# =========================================================

token_report_path = REPORTS / "token_length_analysis.csv"
token_report = pd.read_csv(token_report_path)

print("\nTOKEN-LENGTH REPORT")
print(token_report.to_string(index=False))

expected_tokenizers = ["bert", "distilbert", "modernbert"]
tokenizer_status = {}

for name in expected_tokenizers:
    tokenizer_dir = TOKENIZERS / name

    if tokenizer_dir.is_dir():
        files = [p for p in tokenizer_dir.rglob("*") if p.is_file()]
        tokenizer_status[name] = {
            "directory": str(tokenizer_dir),
            "file_count": len(files),
            "present": len(files) > 0,
        }
    else:
        tokenizer_status[name] = {
            "directory": str(tokenizer_dir),
            "file_count": 0,
            "present": False,
        }

    print(
        f"{name}: {'SAVED' if tokenizer_status[name]['present'] else 'MISSING'}"
        f" ({tokenizer_status[name]['file_count']} files)"
    )

if not all(v["present"] for v in tokenizer_status.values()):
    print("WARNING: At least one tokenizer directory is missing or empty.")

# =========================================================
# 5. BUILD ARTIFACT CHECKSUM MANIFEST
# =========================================================

artifact_paths = [
    train_path,
    val_path,
    test_path,
    METADATA / "label_mapping.json",
    METADATA / "training_plan.json",
    REPORTS / "token_length_analysis.csv",
    REPORTS / "token_length_analysis.json",
]

# Include available important existing artifacts.
for path in optional_files:
    if path.is_file():
        artifact_paths.append(path)

for name in expected_tokenizers:
    tokenizer_dir = TOKENIZERS / name
    if tokenizer_dir.is_dir():
        artifact_paths.extend(
            p for p in tokenizer_dir.rglob("*") if p.is_file()
        )

# Deduplicate paths and calculate hashes.
manifest_items = []

for path in sorted(set(artifact_paths), key=str):
    if not path.is_file():
        continue

    sha256 = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    manifest_items.append({
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256.hexdigest(),
    })

manifest_path = METADATA / "session_artifact_manifest.json"

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_rows": len(train),
    "validation_rows": len(val),
    "test_rows": len(test),
    "category_count": len(all_categories),
    "label_mapping": str(METADATA / "label_mapping.json"),
    "training_plan": str(METADATA / "training_plan.json"),
    "token_length_report": str(token_report_path),
    "tokenizers": tokenizer_status,
    "artifacts": manifest_items,
}

with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# =========================================================
# 6. SAVE TRAINING HANDOVER NOTES
# =========================================================

status_path = PROJECT / "PROJECT_STATUS.md"

status_text = f"""# ResumeClassification — Session Handover

Last updated (UTC): {datetime.now(timezone.utc).isoformat()}

## Current stage
Dataset preparation and tokenizer-length analysis completed.
No model has been trained yet.

## Saved dataset splits
- data/splits/train.parquet: {len(train)} rows
- data/splits/validation.parquet: {len(val)} rows
- data/splits/test.parquet: {len(test)} rows
- metadata/label_mapping.json: {len(label_mapping)} labels

## Training configuration
- metadata/training_plan.json
- reports/token_length_analysis.csv
- reports/token_length_analysis.json
- models/tokenizers/bert/
- models/tokenizers/distilbert/
- models/tokenizers/modernbert/

## Integrity manifest
- metadata/session_artifact_manifest.json

## Token-length findings
Approximately 48.3% of training resumes exceed 512 tokens
for BERT and DistilBERT; approximately 48.41% exceed 512
tokens for ModernBERT. Review these results before fixing the
final truncation length. The maximum observed token lengths
are much larger than 512.

## Known data limitations
- Blockchain has few examples, especially in validation/test.
- Regex redaction is not complete anonymization.
- Original punctuation and common stop words are already absent.
- Near-duplicate detection is heuristic and may miss related resumes.

## Next session
1. Read this handover and metadata/training_plan.json.
2. Check GPU availability and installed library versions.
3. Reuse saved split files; do not download the dataset again.
4. Review token-length analysis and decide on sequence length.
5. Train models one at a time with checkpoints saved under
   /kaggle/working/resume_classification/.
6. Compare validation macro-F1 and weighted-F1.
7. Use the test set only for final evaluation after model selection.

## Persistence reminder
Kaggle persistence is Files only. Runtime variables and GPU memory
will not survive a session restart. Load required objects from files.
"""

status_path.write_text(status_text, encoding="utf-8")

# =========================================================
# 7. FINAL REPORT
# =========================================================

print("\n" + "=" * 65)
print("SESSION WIND-UP COMPLETE")
print("=" * 65)
print("Required artifacts: PRESENT")
print("Saved splits: VERIFIED")
print("Label mapping: VERIFIED")
print("Tokenizer report: VERIFIED")
print("Artifact manifest:", manifest_path)
print("Handover notes:", status_path)
print("Artifacts included in manifest:", len(manifest_items))
print("\nNo model training was performed.")
print("Next step: select sequence length and begin model training.")


## 7. Restore the prepared project after a session restart

These cells verify the saved prepared dataset, mappings, splits, and tokenizers before training resumes. They are designed for Kaggle's persistent files rather than relying on Python variables surviving a restart.

In [ ]:
from pathlib import Path
import json
import importlib.metadata as pkg

ROOT = Path("/kaggle/working/resume_classification")

print("=" * 65)
print("RESUME CLASSIFICATION — POST-RESTART VERIFICATION")
print("=" * 65)

# 1. Check essential saved artifacts
required_files = [
    "PROJECT_STATUS.md",
    "data/processed/final_prepared_dataset.parquet",
    "data/splits/train.parquet",
    "data/splits/validation.parquet",
    "data/splits/test.parquet",
    "metadata/label_mapping.json",
    "metadata/final_split_metadata.json",
    "metadata/training_plan.json",
    "metadata/session_artifact_manifest.json",
    "reports/token_length_analysis.csv",
    "reports/token_length_analysis.json",
]

print(f"\nProject directory exists: {ROOT.exists()}")
print("\nESSENTIAL FILES")

for relative_path in required_files:
    path = ROOT / relative_path
    if path.is_file():
        print(f"[FOUND]     {relative_path} ({path.stat().st_size:,} bytes)")
    else:
        print(f"[MISSING]   {relative_path}")

# 2. Check saved tokenizer files
print("\nSAVED TOKENIZERS")

for model_name in ["bert", "distilbert", "modernbert"]:
    folder = ROOT / "models" / "tokenizers" / model_name
    files = list(folder.iterdir()) if folder.exists() else []
    files = [f for f in files if f.is_file()]
    print(f"{model_name}: {len(files)} files")
    for file in files:
        print(f"  - {file.name}")

# 3. Read the existing training plan without changing it
plan_path = ROOT / "metadata" / "training_plan.json"

print("\nCURRENT TRAINING PLAN")
if plan_path.is_file():
    try:
        with open(plan_path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2))
    except Exception as e:
        print("Could not read training plan:", repr(e))
else:
    print("Training plan not found.")

# 4. Read handover notes
status_path = ROOT / "PROJECT_STATUS.md"

print("\nPROJECT STATUS / HANDOVER NOTES")
if status_path.is_file():
    print(status_path.read_text(encoding="utf-8"))
else:
    print("PROJECT_STATUS.md not found.")

# 5. Check GPU and key installed packages
print("\nHARDWARE")
try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(
            f"  Total memory: "
            f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB"
        )
except Exception as e:
    print("PyTorch/GPU check failed:", repr(e))

print("\nPACKAGE VERSIONS")
for package in ["transformers", "accelerate", "datasets", "scikit-learn"]:
    try:
        print(f"{package}: {pkg.version(package)}")
    except pkg.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nVERIFICATION COMPLETE — no training was started.")

In [ ]:

# ============================================================
# STEP 2 — SAFE POST-RESTART STATE RESTORATION
# Uses the actual saved columns:
# source_row_index, Category, Text_prepared, duplicate_group_id
#
# READ-ONLY for saved project files. No training or downloads.
# ============================================================

import json
import random
import importlib.metadata as pkg
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from transformers import AutoTokenizer, set_seed

# ------------------------------------------------------------
# 1. Paths and reproducibility
# ------------------------------------------------------------
ROOT = Path("/kaggle/working/resume_classification")
META_DIR = ROOT / "metadata"
TOKENIZER_DIR = ROOT / "models" / "tokenizers"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

assert ROOT.is_dir(), f"Project directory missing: {ROOT}"

print("=" * 65)
print("SAFE PROJECT STATE RESTORATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load saved metadata
# ------------------------------------------------------------
with open(META_DIR / "training_plan.json", encoding="utf-8") as f:
    training_plan = json.load(f)

with open(META_DIR / "label_mapping.json", encoding="utf-8") as f:
    label2id = json.load(f)

# Normalize mapping types for consistent Python lookups.
label2id = {str(label): int(idx) for label, idx in label2id.items()}
id2label = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(label2id)

assert len(set(label2id.values())) == NUM_LABELS, (
    "Label mapping contains duplicate numeric IDs."
)
assert set(id2label) == set(range(NUM_LABELS)), (
    "Label IDs are not consecutive from 0."
)

print(f"\nProject: {training_plan.get('project')}")
print(f"Number of labels: {NUM_LABELS}")
print(f"Seed: {SEED}")

# ------------------------------------------------------------
# 3. Load the existing splits only
# ------------------------------------------------------------
split_paths = training_plan["splits"]

for name in ("train", "validation", "test"):
    path = Path(split_paths[name])
    assert path.is_file(), f"Missing {name} split: {path}"

train_df = pd.read_parquet(split_paths["train"])
val_df = pd.read_parquet(split_paths["validation"])
test_df = pd.read_parquet(split_paths["test"])

print("\nSPLIT SIZES")
print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test:       {len(test_df):,}")
print(f"Total:      {len(train_df) + len(val_df) + len(test_df):,}")

print("\nDATAFRAME COLUMNS")
print("Train:", train_df.columns.tolist())
print("Validation:", val_df.columns.tolist())
print("Test:", test_df.columns.tolist())

# ------------------------------------------------------------
# 4. Detect actual columns, including Text_prepared
# ------------------------------------------------------------
def find_column(df, candidates):
    lookup = {str(col).strip().casefold(): col for col in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    return None

TEXT_COL = find_column(
    train_df,
    [
        "Text_prepared",
        "prepared_text",
        "text",
        "redacted_text",
        "resume_text",
        "content",
    ],
)

LABEL_COL = find_column(
    train_df,
    ["Category", "label", "target", "labels"],
)

if TEXT_COL is None or LABEL_COL is None:
    raise ValueError(
        "Required columns were not found. "
        f"Detected text={TEXT_COL}, label={LABEL_COL}. "
        f"Available columns={train_df.columns.tolist()}"
    )

print(f"\nDetected text column: {TEXT_COL}")
print(f"Detected label column: {LABEL_COL}")

# ------------------------------------------------------------
# 5. Validate split sizes, data, and label consistency
# ------------------------------------------------------------
expected_sizes = training_plan.get("split_sizes", {})

for split_name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    expected = expected_sizes.get(split_name)

    if expected is not None and len(df) != expected:
        raise ValueError(
            f"{split_name} size mismatch: expected {expected}, "
            f"found {len(df)}."
        )

    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"{split_name} has inconsistent columns.")

    if df[TEXT_COL].isna().any():
        raise ValueError(f"{split_name} contains missing text.")

    if df[LABEL_COL].isna().any():
        raise ValueError(f"{split_name} contains missing labels.")

    if df[TEXT_COL].astype(str).str.strip().eq("").any():
        raise ValueError(f"{split_name} contains blank text.")

    # Normalize labels as strings, preserving original column names.
    df[LABEL_COL] = df[LABEL_COL].astype(str)

    unknown = set(df[LABEL_COL].unique()) - set(label2id)
    if unknown:
        raise ValueError(
            f"Unknown labels in {split_name}: {sorted(unknown)}"
        )

    # Add a derived model target; do not replace Category.
    df["label_id"] = df[LABEL_COL].map(label2id).astype("int64")

train_labels = set(train_df[LABEL_COL].unique())
if train_labels != set(label2id):
    missing = set(label2id) - train_labels
    raise ValueError(f"Training split is missing labels: {sorted(missing)}")

print("\nDATA VALIDATION: PASSED")
for split_name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    print(
        f"{split_name:10s}: {df[LABEL_COL].nunique()} categories; "
        f"mean text length = "
        f"{df[TEXT_COL].astype(str).str.len().mean():.1f} characters"
    )

# ------------------------------------------------------------
# 6. Check exact prepared-text overlap across splits
# ------------------------------------------------------------
def normalized_text_set(df):
    return set(
        df[TEXT_COL].astype(str).str.strip().str.casefold()
    )

train_texts = normalized_text_set(train_df)
val_texts = normalized_text_set(val_df)
test_texts = normalized_text_set(test_df)

overlap_counts = {
    "train_validation": len(train_texts & val_texts),
    "train_test": len(train_texts & test_texts),
    "validation_test": len(val_texts & test_texts),
}

print("\nEXACT NORMALIZED TEXT OVERLAP")
for pair, count in overlap_counts.items():
    print(f"{pair}: {count}")

if any(overlap_counts.values()):
    print(
        "WARNING: Text overlap detected. Inspect before training; "
        "this cell has not modified any saved split files."
    )
else:
    print("No exact normalized prepared-text overlap detected.")

# Check group/source IDs where available.
for col in ["source_row_index", "duplicate_group_id"]:
    if all(col in df.columns for df in [train_df, val_df, test_df]):
        a = set(train_df[col].dropna().tolist())
        b = set(val_df[col].dropna().tolist())
        c = set(test_df[col].dropna().tolist())

        counts = {
            "train_validation": len(a & b),
            "train_test": len(a & c),
            "validation_test": len(b & c),
        }

        print(f"\n{col} overlap")
        for pair, count in counts.items():
            print(f"{pair}: {count}")

# ------------------------------------------------------------
# 7. Load the saved tokenizers locally
# ------------------------------------------------------------
# Local-only: if a tokenizer is incomplete, report it instead
# of downloading or changing project files.
tokenizers = {}

model_dirs = {
    "bert": TOKENIZER_DIR / "bert",
    "distilbert": TOKENIZER_DIR / "distilbert",
    "modernbert": TOKENIZER_DIR / "modernbert",
}

print("\nTOKENIZER RESTORATION")

for name, folder in model_dirs.items():
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            str(folder),
            local_files_only=True,
            use_fast=True,
        )
        tokenizers[name] = tokenizer

        print(
            f"[OK] {name}: {type(tokenizer).__name__}; "
            f"vocab size={tokenizer.vocab_size:,}; "
            f"model_max_length={tokenizer.model_max_length}"
        )
    except Exception as exc:
        print(f"[ERROR] {name}: {type(exc).__name__}: {exc}")

if len(tokenizers) != 3:
    print(
        "\nWARNING: Not all three tokenizers loaded. "
        "Do not start training until we resolve the issue."
    )

# ------------------------------------------------------------
# 8. Hardware and package checks
# ------------------------------------------------------------
print("\nHARDWARE")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {gpu.name}, "
        f"{gpu.total_memory / 1024**3:.2f} GiB"
    )

print("\nPACKAGE VERSIONS")
for package in [
    "transformers",
    "accelerate",
    "datasets",
    "scikit-learn",
]:
    try:
        print(f"{package}: {pkg.version(package)}")
    except pkg.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# ------------------------------------------------------------
# 9. Final state summary
# ------------------------------------------------------------
print("\nRESTORED IN-MEMORY OBJECTS")
print("ROOT:", ROOT)
print("training_plan: READY")
print("label2id / id2label: READY")
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)
print("TEXT_COL:", TEXT_COL)
print("LABEL_COL:", LABEL_COL)
print("tokenizers loaded:", sorted(tokenizers.keys()))

print("\nRESTORATION CELL FINISHED.")
print("No model training or saved-file modification was performed.")


In [ ]:

# ============================================================
# STEP 3 — AUDIT FULL TOKEN LENGTHS ACROSS ALL SAVED SPLITS
# No training, no dataset download, no changes to split files.
# Creates new audit reports only.
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Reuse the objects restored in the previous cell.
assert "train_df" in globals()
assert "val_df" in globals()
assert "test_df" in globals()
assert "tokenizers" in globals()
assert len(tokenizers) == 3

AUDIT_LIMITS = {
    "bert": 512,
    "distilbert": 512,
    "modernbert": 8192,
}

SPLITS = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

audit_rows = []

print("=" * 70)
print("FULL TOKEN-LENGTH AUDIT")
print("Lengths include special tokens and are measured without truncation.")
print("=" * 70)

for model_name, tokenizer in tokenizers.items():
    target_limit = AUDIT_LIMITS[model_name]

    print(f"\nAuditing {model_name} (target maximum {target_limit:,})")

    # Temporarily raise only the tokenizer's warning threshold.
    # This does NOT truncate or remove tokens.
    original_max_length = tokenizer.model_max_length

    try:
        tokenizer.model_max_length = 1_000_000

        for split_name, df in SPLITS.items():
            texts = df[TEXT_COL].astype(str).tolist()
            lengths = []

            # Batch tokenization limits temporary memory use.
            batch_size = 32

            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]

                encoded = tokenizer(
                    batch,
                    add_special_tokens=True,
                    truncation=False,
                    padding=False,
                    return_attention_mask=False,
                )

                lengths.extend(
                    len(token_ids) for token_ids in encoded["input_ids"]
                )

            lengths_array = np.asarray(lengths, dtype=np.int64)

            if len(lengths_array) != len(df):
                raise RuntimeError(
                    f"Length count mismatch for {model_name}/{split_name}"
                )

            over_limit = int((lengths_array > target_limit).sum())

            row = {
                "model": model_name,
                "split": split_name,
                "target_max_length": target_limit,
                "documents": int(len(lengths_array)),
                "mean_tokens": round(float(lengths_array.mean()), 2),
                "median_tokens": round(float(np.median(lengths_array)), 2),
                "p90_tokens": round(float(np.percentile(lengths_array, 90)), 2),
                "p95_tokens": round(float(np.percentile(lengths_array, 95)), 2),
                "p99_tokens": round(float(np.percentile(lengths_array, 99)), 2),
                "max_observed_tokens": int(lengths_array.max()),
                "over_limit_count": over_limit,
                "over_limit_percent": round(
                    100 * over_limit / len(lengths_array), 2
                ),
            }

            audit_rows.append(row)

            print(
                f"  {split_name:10s} | "
                f"n={row['documents']:,} | "
                f"mean={row['mean_tokens']:.1f} | "
                f"p95={row['p95_tokens']:.0f} | "
                f"max={row['max_observed_tokens']:,} | "
                f"over limit={over_limit:,} "
                f"({row['over_limit_percent']:.2f}%)"
            )

    finally:
        tokenizer.model_max_length = original_max_length

# ------------------------------------------------------------
# Save a separate audit report, preserving previous artifacts.
# ------------------------------------------------------------
audit_df = pd.DataFrame(audit_rows)

csv_path = REPORT_DIR / "token_length_all_splits.csv"
json_path = REPORT_DIR / "token_length_all_splits.json"

audit_df.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(audit_rows, f, indent=2)

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)
print(audit_df.to_string(index=False))
print(f"\nSaved CSV:  {csv_path}")
print(f"Saved JSON: {json_path}")
print("\nNo model was trained and no split file was modified.")


In [ ]:

from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/resume_classification")

assert ROOT.exists(), f"Project directory not found: {ROOT}"

# Safety checks: preserve the prepared data and existing splits.
required_files = [
    ROOT / "data/splits/train.parquet",
    ROOT / "data/splits/validation.parquet",
    ROOT / "data/splits/test.parquet",
    ROOT / "metadata/label_mapping.json",
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Cleanup cancelled. Required files are missing:\n"
        + "\n".join(missing)
    )

models_dir = ROOT / "models"
reports_dir = ROOT / "reports"
processed_dir = ROOT / "data/processed"

removed = []
preserved = []

# 1. Delete model experiment folders and files.
# Keep all saved tokenizers.
if models_dir.exists():
    for path in list(models_dir.iterdir()):
        if path.name.lower() == "tokenizers":
            preserved.append(str(path))
            continue

        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()

        removed.append(str(path))

# 2. Delete outputs generated by discarded model-training experiments.
# Keep preprocessing, dataset-quality, category, and token-length reports.
model_report_prefixes = (
    "bert_",
    "distilbert_",
    "modernbert_",
    "model_comparison",
    "best_hyperparameters",
    "final_test_summary",
    "hyperparameter_search",
    "training_history",
    "overfit_",
    "last_epoch_recovery",
    "checkpoint_",
)

if reports_dir.exists():
    for path in list(reports_dir.iterdir()):
        name = path.name.lower()

        if name.startswith(model_report_prefixes):
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()

            removed.append(str(path))

# 3. Delete caches specifically associated with sliding windows/chunks.
# Keep the final prepared dataset and other processed files.
if processed_dir.exists():
    cache_dirs = [
        p for p in processed_dir.rglob("*")
        if p.is_dir()
        and (
            "sliding" in p.name.lower()
            or "chunk_cache" in p.name.lower()
            or p.name.lower().endswith("_chunks")
        )
    ]

    # Delete deepest paths first to avoid nested-directory conflicts.
    for path in sorted(cache_dirs, key=lambda p: len(p.parts), reverse=True):
        if path.exists():
            shutil.rmtree(path)
            removed.append(str(path))

print("=" * 65)
print("CLEANUP COMPLETE")
print("=" * 65)

print(f"\nRemoved {len(removed)} paths:")
for path in removed:
    print("  DELETED:", path)

print("\nPreserved:")
for path in preserved:
    print("  KEPT:", path)

print("\nThe existing train, validation and test splits are unchanged.")
print("The prepared dataset, metadata and preprocessing reports are retained.")
print("No model was trained by this cleanup cell.")


In [ ]:

from pathlib import Path

ROOT = Path("/kaggle/working/resume_classification")

suspicious_terms = (
    "sliding_window",
    "sliding_chunks",
    "chunk_cache",
    "model_comparison",
    "best_checkpoint",
    "best_model",
    "last_epoch_recovery",
    "hyperparameter",
    "training_history",
    "confusion_matrix",
    "classification_report",
    "test_predictions",
    "test_metrics",
    "overfit",
    "adaptive",
)

candidates = []

for path in ROOT.rglob("*"):
    if not path.exists():
        continue

    # Saved tokenizers are intentionally retained.
    if "tokenizers" in path.parts:
        continue

    name = path.name.lower()

    if any(term in name for term in suspicious_terms):
        size_mb = (
            sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
            if path.is_dir()
            else path.stat().st_size
        ) / (1024 ** 2)

        candidates.append((str(path), "directory" if path.is_dir() else "file",
                           round(size_mb, 2)))

print("Potential remnants of discarded experiments:")
if candidates:
    for path, kind, size_mb in candidates:
        print(f"{kind:9} | {size_mb:8.2f} MB | {path}")
else:
    print("None found by filename.")

print("\nAudit complete. No files were modified.")


## 8. BERT baseline: tuning and final test evaluation

The BERT section provides a baseline. The final test evaluation is inference-only: it loads the selected checkpoint and does not update model weights.

In [ ]:

# ================================================================
# RESUME CLASSIFICATION — BERT BASE HYPERPARAMETER OPTIMIZATION
# Fresh model per trial | Dynamic padding | Mixed precision
# Validation-based checkpoint selection | No test-set evaluation
# ================================================================

import os
import gc
import json
import time
import random
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# ----------------------------- SETTINGS --------------------------

ROOT = Path("/kaggle/working/resume_classification")
TRAIN_PATH = ROOT / "data/splits/train.parquet"
VAL_PATH = ROOT / "data/splits/validation.parquet"
LABEL_PATH = ROOT / "metadata/label_mapping.json"
TOKENIZER_PATH = ROOT / "models/tokenizers/bert"

MODEL_ID = "google-bert/bert-base-uncased"

TEXT_COL = "Text_prepared"
LABEL_COL = "Category"

SEED = 42
MAX_LENGTH = 512

# Eight independent hyperparameter trials.
N_TRIALS = 8
MAX_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
MIN_DELTA = 1e-4

# Physical batches are kept conservative for Kaggle T4 GPUs.
# Effective batch = physical batch * gradient accumulation.
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

MAX_GRAD_NORM = 1.0
ADAM_EPSILON = 1e-8

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Temporary checkpoints are removed after the winning model is copied.
TEMP_MODEL_ROOT = ROOT / "models" / f".bert_trials_{STAMP}"
REPORT_DIR = ROOT / "reports" / f"bert_tuning_{STAMP}"
FINAL_MODEL_DIR = ROOT / "models" / "bert_best"

TEMP_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------ SAFETY CHECKS --------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. In Kaggle, enable Settings > Accelerator > GPU."
    )

for path in [TRAIN_PATH, VAL_PATH, LABEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required project file not found: {path}")

device = torch.device("cuda")

print("=" * 72)
print("BERT FINE-TUNING EXPERIMENT")
print("=" * 72)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Temporary run directory:", TEMP_MODEL_ROOT)
print("Reports directory:", REPORT_DIR)

# --------------------------- LOAD DATA ----------------------------

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)

for split_name, frame in [("train", train_df), ("validation", val_df)]:
    required = {TEXT_COL, LABEL_COL}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} split missing columns: {missing}")

    frame[TEXT_COL] = frame[TEXT_COL].fillna("").astype(str)
    frame[LABEL_COL] = frame[LABEL_COL].astype(str).str.strip()

if train_df.empty or val_df.empty:
    raise ValueError("Train or validation split is empty.")

# A deterministic alphabetical mapping shared by all future model runs.
# The original dataset mapping remains unchanged.
classes = sorted(train_df[LABEL_COL].unique().tolist())

unknown_val_labels = sorted(set(val_df[LABEL_COL]) - set(classes))
if unknown_val_labels:
    raise ValueError(
        "Validation contains labels not present in training: "
        f"{unknown_val_labels}"
    )

label2id = {label: i for i, label in enumerate(classes)}
id2label = {i: label for label, i in label2id.items()}

y_train = train_df[LABEL_COL].map(label2id).to_numpy(dtype=np.int64)
y_val = val_df[LABEL_COL].map(label2id).to_numpy(dtype=np.int64)

num_labels = len(classes)

mapping_path = REPORT_DIR / "label_mapping.json"
with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
            "num_labels": num_labels,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"\nTraining resumes:   {len(train_df):,}")
print(f"Validation resumes: {len(val_df):,}")
print(f"Number of classes:  {num_labels}")

if num_labels != 43:
    print(
        f"WARNING: Expected 43 categories based on the project plan; "
        f"found {num_labels}. Verify the labels before continuing."
    )

# -------------------------- TOKENIZER ----------------------------

if TOKENIZER_PATH.exists():
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            str(TOKENIZER_PATH),
            local_files_only=True,
        )
        print("\nLoaded saved BERT tokenizer:", TOKENIZER_PATH)
    except Exception as exc:
        print("Saved tokenizer could not be loaded:", str(exc)[:300])
        print("Trying the tokenizer from the model repository...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
else:
    print("\nSaved BERT tokenizer not found; downloading from model repository.")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token_id is None:
    raise ValueError("The tokenizer has no pad token; aborting safely.")

print("Tokenizer vocabulary size:", len(tokenizer))
print("Truncation limit:", MAX_LENGTH)

# ---------------------- TOKENIZE ONCE ----------------------------

# Tokenization is done once and reused across all hyperparameter trials.
# Padding stays dynamic: each batch is padded only to its longest sequence.

def encode_split(texts, name):
    print(f"\nTokenizing {name}...")
    encoded = tokenizer(
        texts.tolist(),
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=True,
    )
    lengths = np.array([len(ids) for ids in encoded["input_ids"]])
    at_limit = int(np.sum(lengths >= MAX_LENGTH))

    print(
        f"{name}: {len(lengths):,} records | "
        f"median encoded length={np.median(lengths):.0f} | "
        f"95th percentile={np.percentile(lengths, 95):.0f} | "
        f"at the {MAX_LENGTH}-token limit={at_limit:,}"
    )
    return encoded

train_enc = encode_split(train_df[TEXT_COL], "Train")
val_enc = encode_split(val_df[TEXT_COL], "Validation")


class EncodedResumeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: values[idx] for key, values in self.encodings.items()}
        item["labels"] = int(self.labels[idx])
        return item


def make_collator(tok):
    def collate(features):
        labels = torch.tensor(
            [feature.pop("labels") for feature in features],
            dtype=torch.long,
        )
        batch = tok.pad(
            features,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        batch["labels"] = labels
        return batch
    return collate


collator = make_collator(tokenizer)

train_dataset = EncodedResumeDataset(train_enc, y_train)
val_dataset = EncodedResumeDataset(val_enc, y_val)


def make_loaders(grad_accum):
    # Same physical batch size for fair comparisons; accumulation changes
    # the effective optimizer batch size.
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
        generator=train_generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    return train_loader, val_loader


# ----------------------- EVALUATION FUNCTION ---------------------

def evaluate_model(model, loader):
    model.eval()

    true_ids = []
    pred_ids = []
    weighted_loss_sum = 0.0
    number_seen = 0

    with torch.inference_mode():
        for batch in loader:
            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):
                outputs = model(**batch)

            batch_n = batch["labels"].size(0)
            weighted_loss_sum += outputs.loss.item() * batch_n
            number_seen += batch_n

            predictions = outputs.logits.argmax(dim=-1)
            true_ids.extend(batch["labels"].cpu().tolist())
            pred_ids.extend(predictions.cpu().tolist())

    true_ids = np.asarray(true_ids, dtype=np.int64)
    pred_ids = np.asarray(pred_ids, dtype=np.int64)

    val_loss = weighted_loss_sum / max(number_seen, 1)
    accuracy = accuracy_score(true_ids, pred_ids)

    macro_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=np.arange(num_labels),
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=np.arange(num_labels),
        average="weighted",
        zero_division=0,
    )

    return {
        "loss": float(val_loss),
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "true_ids": true_ids,
        "pred_ids": pred_ids,
    }


# ----------------------- TRAIN ONE TRIAL -------------------------

epoch_history = []
successful_trials = []
failed_trials = []


def train_one_trial(trial_id, hp):
    print("\n" + "=" * 72)
    print(f"TRIAL {trial_id:02d}/{N_TRIALS}")
    print(json.dumps(hp, indent=2))
    print("=" * 72)

    # Reset the random state before every trial. This improves comparability
    # between configurations, though it does not remove all run variability.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    train_loader, val_loader = make_loaders(hp["grad_accum"])

    config = AutoConfig.from_pretrained(MODEL_ID)
    config.num_labels = num_labels
    config.id2label = id2label
    config.label2id = label2id

    # Tune dropout while loading the same pretrained BERT encoder.
    config.hidden_dropout_prob = hp["dropout"]
    config.attention_probs_dropout_prob = hp["dropout"]
    config.classifier_dropout = hp["dropout"]

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        config=config,
        ignore_mismatched_sizes=True,
    )

    # Normally the saved tokenizer uses the same vocabulary as the pretrained
    # checkpoint. Resize only if the local tokenizer includes extra tokens.
    if model.get_input_embeddings().num_embeddings != len(tokenizer):
        model.resize_token_embeddings(len(tokenizer))

    model.to(device)
    model.config.use_cache = False

    # Do not apply weight decay to biases or LayerNorm parameters.
    no_decay = ("bias", "LayerNorm.weight")
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(nd in name for nd in no_decay):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": hp["weight_decay"]},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=hp["learning_rate"],
        betas=(0.9, 0.999),
        eps=ADAM_EPSILON,
    )

    optimizer_steps_per_epoch = int(
        np.ceil(len(train_loader) / hp["grad_accum"])
    )
    total_optimizer_steps = optimizer_steps_per_epoch * MAX_EPOCHS
    warmup_steps = int(total_optimizer_steps * hp["warmup_ratio"])

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_optimizer_steps,
    )

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=True)

    trial_checkpoint = TEMP_MODEL_ROOT / f"trial_{trial_id:02d}" / "best_model"
    trial_checkpoint.parent.mkdir(parents=True, exist_ok=True)

    best_score = None
    best_metrics = None
    best_epoch = None
    best_val_loss_seen = float("inf")
    epochs_without_val_loss_improvement = 0

    started = time.time()

    try:
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            train_loss_sum = 0.0
            train_examples_seen = 0
            number_of_batches = len(train_loader)

            for step, batch in enumerate(train_loader, start=1):
                batch = {
                    key: value.to(device, non_blocking=True)
                    for key, value in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    outputs = model(**batch)
                    raw_loss = outputs.loss
                    loss = raw_loss / hp["grad_accum"]

                scaler.scale(loss).backward()

                batch_n = batch["labels"].size(0)
                train_loss_sum += raw_loss.item() * batch_n
                train_examples_seen += batch_n

                is_update_step = (
                    step % hp["grad_accum"] == 0
                    or step == number_of_batches
                )

                if is_update_step:
                    scaler.unscale_(optimizer)

                    # Correct the final partial gradient-accumulation group.
                    remainder = step % hp["grad_accum"]
                    if step == number_of_batches and remainder:
                        correction = hp["grad_accum"] / remainder
                        for parameter in model.parameters():
                            if parameter.grad is not None:
                                parameter.grad.mul_(correction)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), MAX_GRAD_NORM
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

                if step % 100 == 0:
                    print(
                        f"  epoch={epoch}/{MAX_EPOCHS}, "
                        f"batch={step}/{number_of_batches}"
                    )

            train_loss = train_loss_sum / max(train_examples_seen, 1)

            val_result = evaluate_model(model, val_loader)
            val_loss = val_result["loss"]
            val_acc = val_result["accuracy"]
            val_macro_f1 = val_result["macro_f1"]
            val_weighted_f1 = val_result["weighted_f1"]

            history_row = {
                "trial": trial_id,
                "epoch": epoch,
                **hp,
                "train_loss": train_loss,
                "validation_loss": val_loss,
                "validation_accuracy": val_acc,
                "validation_macro_f1": val_macro_f1,
                "validation_weighted_f1": val_weighted_f1,
                "learning_rate_after_epoch": scheduler.get_last_lr()[0],
            }
            epoch_history.append(history_row)

            pd.DataFrame(epoch_history).to_csv(
                REPORT_DIR / "bert_epoch_history.csv",
                index=False,
            )

            print(
                f"\nTrial {trial_id:02d} | Epoch {epoch}/{MAX_EPOCHS} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"val_accuracy={val_acc:.4f} | "
                f"macro_F1={val_macro_f1:.4f} | "
                f"weighted_F1={val_weighted_f1:.4f}"
            )

            # Accuracy is primary; macro-F1 and lower loss break ties.
            score = (
                val_acc,
                val_macro_f1,
                -val_loss,
            )

            if best_score is None or score > best_score:
                best_score = score
                best_epoch = epoch
                best_metrics = {
                    "best_val_accuracy": val_acc,
                    "best_val_macro_f1": val_macro_f1,
                    "best_val_weighted_f1": val_weighted_f1,
                    "best_val_loss": val_loss,
                }

                if trial_checkpoint.exists():
                    shutil.rmtree(trial_checkpoint)

                trial_checkpoint.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(trial_checkpoint)
                tokenizer.save_pretrained(trial_checkpoint)

                with open(
                    trial_checkpoint / "training_selection.json",
                    "w",
                    encoding="utf-8",
                ) as f:
                    json.dump(
                        {
                            "trial": trial_id,
                            "best_epoch": best_epoch,
                            "hyperparameters": hp,
                            **best_metrics,
                        },
                        f,
                        indent=2,
                    )

                print("  Saved new best checkpoint for this trial.")

            # Early stopping monitors validation loss independently of
            # checkpoint selection. A lower loss does not always mean
            # higher accuracy, so both are recorded.
            if val_loss < best_val_loss_seen - MIN_DELTA:
                best_val_loss_seen = val_loss
                epochs_without_val_loss_improvement = 0
            else:
                epochs_without_val_loss_improvement += 1

            if (
                epochs_without_val_loss_improvement
                >= EARLY_STOPPING_PATIENCE
            ):
                print(
                    f"Early stopping trial {trial_id}: "
                    f"validation loss did not improve for "
                    f"{EARLY_STOPPING_PATIENCE} consecutive epochs."
                )
                break

        result = {
            "trial": trial_id,
            **hp,
            **best_metrics,
            "best_epoch": best_epoch,
            "training_seconds": time.time() - started,
            "checkpoint_path": str(trial_checkpoint),
        }

        print("\nTrial result:")
        print(json.dumps(
            {k: v for k, v in result.items() if k != "checkpoint_path"},
            indent=2,
        ))

        return result

    finally:
        del model, optimizer, scheduler, scaler
        del train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()


# --------------------- HYPERPARAMETER SEARCH ---------------------

# Optuna uses TPE when installed. If unavailable, a documented preset search
# is used instead; no network install is required.

try:
    import optuna
    OPTUNA_AVAILABLE = True
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print("\nOptuna is available; using TPE hyperparameter optimization.")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("\nOptuna is unavailable; using a predefined hyperparameter search.")

fallback_configs = [
    {
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.10,
        "grad_accum": 2,
        "dropout": 0.10,
    },
    {
        "learning_rate": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "grad_accum": 2,
        "dropout": 0.10,
    },
    {
        "learning_rate": 1e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.10,
        "grad_accum": 2,
        "dropout": 0.10,
    },
    {
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "grad_accum": 2,
        "dropout": 0.10,
    },
    {
        "learning_rate": 2e-5,
        "weight_decay": 0.00,
        "warmup_ratio": 0.03,
        "grad_accum": 4,
        "dropout": 0.10,
    },
    {
        "learning_rate": 3e-5,
        "weight_decay": 0.03,
        "warmup_ratio": 0.10,
        "grad_accum": 4,
        "dropout": 0.20,
    },
    {
        "learning_rate": 1.5e-5,
        "weight_decay": 0.05,
        "warmup_ratio": 0.03,
        "grad_accum": 2,
        "dropout": 0.20,
    },
    {
        "learning_rate": 4e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.10,
        "grad_accum": 4,
        "dropout": 0.20,
    },
]

successful_trials = []
failed_trials = []


def run_trial_safely(trial_id, hp):
    try:
        result = train_one_trial(trial_id, hp)
        successful_trials.append(result)

        pd.DataFrame([
            {k: v for k, v in item.items() if k != "checkpoint_path"}
            for item in successful_trials
        ]).to_csv(
            REPORT_DIR / "bert_hyperparameter_trials.csv",
            index=False,
        )
        return result

    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            failed_trials.append({
                "trial": trial_id,
                "hyperparameters": hp,
                "error": "CUDA out of memory",
            })
            print(
                f"\nTrial {trial_id} ran out of GPU memory and was skipped. "
                "Clear GPU memory or reduce TRAIN_BATCH_SIZE to 4 before "
                "rerunning if this happens repeatedly."
            )
            gc.collect()
            torch.cuda.empty_cache()
            return None
        raise


if OPTUNA_AVAILABLE:
    def objective(trial):
        hp = {
            "learning_rate": trial.suggest_float(
                "learning_rate", 1e-5, 5e-5, log=True
            ),
            "weight_decay": trial.suggest_categorical(
                "weight_decay", [0.0, 0.01, 0.03, 0.05]
            ),
            "warmup_ratio": trial.suggest_categorical(
                "warmup_ratio", [0.03, 0.06, 0.10]
            ),
            "grad_accum": trial.suggest_categorical(
                "grad_accum", [2, 4]
            ),
            "dropout": trial.suggest_categorical(
                "dropout", [0.10, 0.20]
            ),
        }

        result = run_trial_safely(trial.number + 1, hp)

        if result is None:
            raise optuna.TrialPruned("CUDA OOM")

        trial.set_user_attr(
            "best_val_macro_f1", result["best_val_macro_f1"]
        )
        trial.set_user_attr(
            "best_val_loss", result["best_val_loss"]
        )
        trial.set_user_attr(
            "best_epoch", result["best_epoch"]
        )

        # Primary optimization objective: validation accuracy.
        return result["best_val_accuracy"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        gc_after_trial=True,
        show_progress_bar=False,
    )

else:
    for trial_id, hp in enumerate(fallback_configs[:N_TRIALS], start=1):
        run_trial_safely(trial_id, hp)

if not successful_trials:
    raise RuntimeError(
        "No BERT trial completed successfully. Review the output above, "
        "confirm GPU/internet availability, and inspect the project paths."
    )

# ---------------------- SELECT BEST TRIAL ------------------------

# Use validation accuracy first, then macro-F1, then lower validation loss.
best_trial = max(
    successful_trials,
    key=lambda r: (
        r["best_val_accuracy"],
        r["best_val_macro_f1"],
        -r["best_val_loss"],
    ),
)

best_source = Path(best_trial["checkpoint_path"])

if not best_source.exists():
    raise FileNotFoundError(
        f"Winning checkpoint does not exist: {best_source}"
    )

# Preserve an existing model rather than silently deleting it.
if FINAL_MODEL_DIR.exists():
    previous_dir = ROOT / "models" / f"bert_best_previous_{STAMP}"
    FINAL_MODEL_DIR.rename(previous_dir)
    print("Previous BERT best checkpoint preserved at:", previous_dir)

shutil.copytree(best_source, FINAL_MODEL_DIR)

# Save the stable label map alongside the deployable checkpoint.
with open(
    FINAL_MODEL_DIR / "label_mapping.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
            "num_labels": num_labels,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

best_summary = {
    "model": MODEL_ID,
    "max_length": MAX_LENGTH,
    "selection_metric": "validation_accuracy",
    "tie_breakers": [
        "validation_macro_f1",
        "lower_validation_loss",
    ],
    "best_trial": int(best_trial["trial"]),
    "best_epoch": int(best_trial["best_epoch"]),
    "best_hyperparameters": {
        key: best_trial[key]
        for key in [
            "learning_rate",
            "weight_decay",
            "warmup_ratio",
            "grad_accum",
            "dropout",
        ]
    },
    "best_validation_metrics": {
        "accuracy": best_trial["best_val_accuracy"],
        "macro_f1": best_trial["best_val_macro_f1"],
        "weighted_f1": best_trial["best_val_weighted_f1"],
        "loss": best_trial["best_val_loss"],
    },
    "model_path": str(FINAL_MODEL_DIR),
    "seed": SEED,
    "classes": num_labels,
}

with open(
    REPORT_DIR / "bert_best_hyperparameters.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(best_summary, f, indent=2)

with open(
    FINAL_MODEL_DIR / "experiment_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(best_summary, f, indent=2)

# ----------------- RELOAD AND VERIFY THE WINNER -----------------

print("\nReloading selected checkpoint for a final validation pass...")

del train_enc
gc.collect()
torch.cuda.empty_cache()

best_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR,
    local_files_only=True,
)
best_model = AutoModelForSequenceClassification.from_pretrained(
    FINAL_MODEL_DIR,
    local_files_only=True,
).to(device)

best_collator = make_collator(best_tokenizer)
final_val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=best_collator,
    num_workers=0,
    pin_memory=True,
)

final_val_result = evaluate_model(best_model, final_val_loader)

true_val = final_val_result["true_ids"]
pred_val = final_val_result["pred_ids"]

final_validation_metrics = {
    "accuracy": final_val_result["accuracy"],
    "macro_f1": final_val_result["macro_f1"],
    "weighted_f1": final_val_result["weighted_f1"],
    "loss": final_val_result["loss"],
    "samples": len(true_val),
    "classes": num_labels,
}

with open(
    REPORT_DIR / "bert_final_validation_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(final_validation_metrics, f, indent=2)

report_dict = classification_report(
    true_val,
    pred_val,
    labels=np.arange(num_labels),
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(report_dict).transpose().to_csv(
    REPORT_DIR / "bert_validation_classification_report.csv"
)

cm = confusion_matrix(
    true_val,
    pred_val,
    labels=np.arange(num_labels),
)
pd.DataFrame(cm, index=classes, columns=classes).to_csv(
    REPORT_DIR / "bert_validation_confusion_matrix.csv"
)

pd.DataFrame({
    "true_label": [id2label[int(x)] for x in true_val],
    "predicted_label": [id2label[int(x)] for x in pred_val],
    "correct": true_val == pred_val,
}).to_csv(
    REPORT_DIR / "bert_validation_predictions.csv",
    index=False,
)

if failed_trials:
    with open(
        REPORT_DIR / "failed_trials.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(failed_trials, f, indent=2)

# Keep the selected model and detailed reports, remove temporary trial
# checkpoints so they do not clutter the project directory.
shutil.rmtree(TEMP_MODEL_ROOT, ignore_errors=True)

del best_model, final_val_loader, best_tokenizer
gc.collect()
torch.cuda.empty_cache()

# -------------------------- FINAL OUTPUT -------------------------

print("\n" + "=" * 72)
print("BERT FINE-TUNING COMPLETE")
print("=" * 72)
print("Winning hyperparameters:")
print(json.dumps(best_summary["best_hyperparameters"], indent=2))

print("\nFinal validation metrics:")
for key, value in final_validation_metrics.items():
    print(f"  {key}: {value}")

print("\nSaved best model:", FINAL_MODEL_DIR)
print("Saved experiment reports:", REPORT_DIR)
print("Saved trial summary:", REPORT_DIR / "bert_hyperparameter_trials.csv")
print("Saved per-class validation report:",
      REPORT_DIR / "bert_validation_classification_report.csv")

if final_validation_metrics["accuracy"] < 0.90:
    print(
        "\nThe 90% validation-accuracy target was not reached. "
        "Do not change the test set or claim the target was met. "
        "Review error patterns, label quality, class imbalance and "
        "preprocessing before planning the next tuning round."
    )
else:
    print(
        "\nThe 90% validation-accuracy target has been reached for this run. "
        "This still does not guarantee the same test accuracy."
    )

print(
    "\nThe test split was not loaded or evaluated. "
    "We will use it for the final comparison of BERT, DistilBERT and ModernBERT."
)


In [ ]:

# ================================================================
# BERT FINAL TEST EVALUATION — CORRECTED VERSION
# Inference only. No training and no model-weight updates.
# ================================================================

import gc
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# --------------------------- CONFIG --------------------------------

ROOT = Path("/kaggle/working/resume_classification")
MODEL_DIR = ROOT / "models/bert_best"
TEST_PATH = ROOT / "data/splits/test.parquet"

TEXT_COL = "Text_prepared"
LABEL_COL = "Category"
MAX_LENGTH = 512
BATCH_SIZE = 16
SEED = 42

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
REPORT_DIR = ROOT / "reports" / f"bert_final_test_{STAMP}"
REPORT_DIR.mkdir(parents=True, exist_ok=False)

# ------------------------- VERIFY FILES ----------------------------

required_paths = [
    MODEL_DIR / "config.json",
    MODEL_DIR / "label_mapping.json",
    TEST_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------- LOAD LABEL MAPPING -------------------------

with open(MODEL_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

# IMPORTANT: create both mappings with the correct direction.
label2id = {
    str(label): int(idx)
    for label, idx in mapping["label2id"].items()
}
id2label = {
    idx: label
    for label, idx in label2id.items()
}

num_labels = len(label2id)
expected_ids = set(range(num_labels))

if set(id2label.keys()) != expected_ids:
    raise ValueError(
        "Label IDs must be contiguous from 0 to num_labels - 1. "
        "Please inspect models/bert_best/label_mapping.json."
    )

classes = [id2label[i] for i in range(num_labels)]
label_ids = np.arange(num_labels)

print("Number of classes:", num_labels)
print("Label mapping verified.")

# -------------------- LOAD MODEL AND TOKENIZER ---------------------

print("\nLoading saved BERT checkpoint...")

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
).to(device)

model.eval()

if model.config.num_labels != num_labels:
    raise ValueError(
        f"Model has {model.config.num_labels} outputs but label mapping "
        f"contains {num_labels} classes."
    )

configured_limit = getattr(model.config, "max_position_embeddings", 512)
if MAX_LENGTH > configured_limit:
    raise ValueError(
        f"Input length {MAX_LENGTH} exceeds model limit {configured_limit}."
    )

# --------------------------- LOAD TEST -----------------------------

test_df = pd.read_parquet(TEST_PATH).reset_index(drop=True)

if TEXT_COL not in test_df.columns or LABEL_COL not in test_df.columns:
    raise ValueError(
        f"Test data must contain {TEXT_COL!r} and {LABEL_COL!r}."
    )

test_df[TEXT_COL] = test_df[TEXT_COL].fillna("").astype(str)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(str).str.strip()

unknown_labels = sorted(set(test_df[LABEL_COL]) - set(label2id))
if unknown_labels:
    raise ValueError(
        f"Unknown test labels absent from training mapping: {unknown_labels}"
    )

y_true = test_df[LABEL_COL].map(label2id).to_numpy(dtype=np.int64)

print("Test resumes:", f"{len(test_df):,}")
print("Input limit:", MAX_LENGTH, "tokens")

# ------------------------- TOKENIZATION ----------------------------

print("\nTokenizing test resumes...")

encoded = tokenizer(
    test_df[TEXT_COL].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,
    return_attention_mask=True,
)

encoded_lengths = np.array([
    len(token_ids) for token_ids in encoded["input_ids"]
])

print("Median encoded length:", int(np.median(encoded_lengths)))
print("95th percentile:", int(np.percentile(encoded_lengths, 95)))
print(
    "Sequences reaching the truncated input limit:",
    int(np.sum(encoded_lengths >= MAX_LENGTH)),
)

class TestResumeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = int(self.labels[index])
        return item


def collate_test_batch(features):
    # Keep labels aligned with their corresponding resumes.
    labels = torch.tensor(
        [item["labels"] for item in features],
        dtype=torch.long,
    )

    inputs = [
        {key: value for key, value in item.items() if key != "labels"}
        for item in features
    ]

    batch = tokenizer.pad(
        inputs,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )
    batch["labels"] = labels
    return batch


test_dataset = TestResumeDataset(encoded, y_true)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_test_batch,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

# ------------------------- TEST INFERENCE --------------------------

print("\nRunning test inference...")

all_logits = []
all_probabilities = []
all_true = []
loss_sum = 0.0
processed = 0

with torch.inference_mode():
    for batch_number, batch in enumerate(test_loader, start=1):
        labels = batch.pop("labels").to(device, non_blocking=True)
        inputs = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        if device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)

        logits = outputs.logits.float()
        batch_loss = F.cross_entropy(
            logits,
            labels,
            reduction="sum",
        )

        batch_size = labels.shape[0]
        loss_sum += batch_loss.item()
        processed += batch_size

        probabilities = torch.softmax(logits, dim=-1)

        all_logits.append(logits.cpu().numpy())
        all_probabilities.append(probabilities.cpu().numpy())
        all_true.extend(labels.cpu().numpy().tolist())

        if batch_number % 10 == 0 or processed == len(test_dataset):
            print(f"Processed {processed:,}/{len(test_dataset):,}")

logits = np.concatenate(all_logits, axis=0)
probabilities = np.concatenate(all_probabilities, axis=0)
y_true_eval = np.asarray(all_true, dtype=np.int64)
y_pred = logits.argmax(axis=1)

test_loss = loss_sum / max(processed, 1)

# Ensure inference order and dataset labels stayed aligned.
if not np.array_equal(y_true_eval, y_true):
    raise RuntimeError("Test label order mismatch. Results were not saved.")

# --------------------------- METRICS --------------------------------

accuracy = accuracy_score(y_true_eval, y_pred)
balanced_accuracy = balanced_accuracy_score(y_true_eval, y_pred)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true_eval,
        y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0,
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_true_eval,
        y_pred,
        labels=label_ids,
        average="weighted",
        zero_division=0,
    )
)

metrics = {
    "model": "BERT Base fine-tuned",
    "checkpoint": str(MODEL_DIR),
    "input_limit_tokens": MAX_LENGTH,
    "test_samples": int(len(y_true_eval)),
    "number_of_classes": int(num_labels),
    "test_loss": float(test_loss),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_precision),
    "weighted_recall": float(weighted_recall),
    "weighted_f1": float(weighted_f1),
}

print("\n" + "=" * 65)
print("BERT — FINAL TEST METRICS")
print("=" * 65)

for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")

with open(REPORT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# -------------------- PER-CLASS CLASSIFICATION ---------------------

report = classification_report(
    y_true_eval,
    y_pred,
    labels=label_ids,
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(report).transpose().to_csv(
    REPORT_DIR / "classification_report.csv"
)

# ------------------------- CONFUSION MATRICES ----------------------

cm = confusion_matrix(
    y_true_eval,
    y_pred,
    labels=label_ids,
)

# Matrix 1: counts
cm_counts_df = pd.DataFrame(cm, index=classes, columns=classes)
cm_counts_df.index.name = "true_label"
cm_counts_df.columns.name = "predicted_label"
cm_counts_df.to_csv(REPORT_DIR / "confusion_matrix_counts.csv")

# Matrix 2: row-normalized fractions
row_totals = cm.sum(axis=1, keepdims=True)

cm_normalized = np.divide(
    cm.astype(np.float64),
    row_totals,
    out=np.zeros_like(cm, dtype=np.float64),
    where=row_totals != 0,
)

cm_normalized_df = pd.DataFrame(
    cm_normalized, index=classes, columns=classes
)
cm_normalized_df.index.name = "true_label"
cm_normalized_df.columns.name = "predicted_label"
cm_normalized_df.to_csv(
    REPORT_DIR / "confusion_matrix_normalized.csv"
)


def save_confusion_matrix_plot(matrix, title, filename, colorbar_label):
    size = max(14, min(24, len(classes) * 0.48))

    fig, ax = plt.subplots(figsize=(size, size))
    image = ax.imshow(
        matrix,
        interpolation="nearest",
        aspect="auto",
        cmap="viridis",
    )

    ax.set_title(title)
    ax.set_xlabel("Predicted category")
    ax.set_ylabel("True category")

    ax.set_xticks(label_ids)
    ax.set_yticks(label_ids)
    ax.set_xticklabels(classes, rotation=90, fontsize=6)
    ax.set_yticklabels(classes, fontsize=6)

    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label(colorbar_label)

    fig.tight_layout()
    fig.savefig(REPORT_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


save_confusion_matrix_plot(
    cm,
    "BERT — Test Confusion Matrix (Counts)",
    "confusion_matrix_counts.png",
    "Number of resumes",
)

save_confusion_matrix_plot(
    cm_normalized,
    "BERT — Test Confusion Matrix (Row Normalized)",
    "confusion_matrix_normalized.png",
    "Fraction of actual-category resumes",
)

# ----------------------- INDIVIDUAL PREDICTIONS --------------------

top_k = min(3, num_labels)
top_indices = np.argsort(probabilities, axis=1)[:, -top_k:][:, ::-1]
confidence = probabilities.max(axis=1)

# Do not copy the full resume text into another report unnecessarily.
predictions_df = pd.DataFrame({
    "test_row_index": np.arange(len(test_df)),
    "true_label": [id2label[int(x)] for x in y_true_eval],
    "predicted_label": [id2label[int(x)] for x in y_pred],
    "confidence": confidence,
    "correct": (y_true_eval == y_pred),
    "encoded_input_length": encoded_lengths,
})

# Preserve the source record index if the split contains it.
if "source_row_index" in test_df.columns:
    predictions_df.insert(
        1,
        "source_row_index",
        test_df["source_row_index"].to_numpy(),
    )

predictions_df["top_3_predictions"] = [
    json.dumps(
        [
            {
                "category": id2label[int(class_idx)],
                "probability": float(probabilities[row_idx, class_idx]),
            }
            for class_idx in top_indices[row_idx]
        ],
        ensure_ascii=False,
    )
    for row_idx in range(len(test_df))
]

predictions_df.to_csv(
    REPORT_DIR / "test_predictions.csv",
    index=False,
)

# --------------------- EVALUATION PROVENANCE -----------------------

metadata = {
    "model_checkpoint": str(MODEL_DIR),
    "test_split": str(TEST_PATH),
    "timestamp": STAMP,
    "input_length_limit": MAX_LENGTH,
    "training_performed": False,
    "model_weights_modified": False,
    "test_evaluated": True,
    "test_samples": int(len(test_df)),
    "class_mapping_file": str(MODEL_DIR / "label_mapping.json"),
}

with open(
    REPORT_DIR / "evaluation_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 65)
print("TEST EVALUATION COMPLETE")
print("=" * 65)
print("Reports directory:", REPORT_DIR)
print("\nSaved artifacts:")

for path in sorted(REPORT_DIR.iterdir()):
    print(" -", path.name)

# ---------------------------- CLEANUP ------------------------------

del model, test_loader, test_dataset, encoded
del all_logits, all_probabilities, probabilities, logits
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nNo fine-tuning was performed. The saved BERT model is unchanged.")


## 9. ModernBERT: initial hyperparameter search

This is an earlier tuning implementation retained as part of the experiment history. It tests validation-based checkpoint selection with a 2,048-token input limit. The later memory-safe tuning cell below is the refined run used for the final selected checkpoint.

In [ ]:

# =====================================================================
# MODERNBERT — OPTUNA TUNING + FULL FINE-TUNING
# TPE sampler + MedianPruner | 2048 tokens | Tesla T4 safeguards
# Test set is deliberately NOT loaded or evaluated in this cell.
# =====================================================================

import os
import gc
import json
import time
import random
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Optuna is required for this cell. Install it in a separate Kaggle "
        "cell with %pip install optuna, then rerun this cell."
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)

# --------------------------- CONFIGURATION ----------------------------

ROOT = Path("/kaggle/working/resume_classification")

TRAIN_PATH = ROOT / "data/splits/train.parquet"
VAL_PATH = ROOT / "data/splits/validation.parquet"
BERT_MODEL_DIR = ROOT / "models/bert_best"
TOKENIZER_PATH = ROOT / "models/tokenizers/modernbert"

MODEL_ID = "answerdotai/ModernBERT-base"

TEXT_COL = "Text_prepared"
LABEL_COL = "Category"

SEED = 42
MAX_LENGTH = 2048

# Six adaptive trials. Each can train for up to four epochs.
N_TRIALS = 6
MAX_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
MIN_DELTA = 1e-4

# A memory preflight selects the largest safe physical batch from these.
BATCH_CANDIDATES = [4, 2, 1]

# Effective batch size = physical batch * gradient accumulation.
GRAD_ACCUM_CHOICES = [4, 8]

EVAL_BATCH_SIZE = 4
MAX_GRAD_NORM = 1.0
ADAM_EPSILON = 1e-8

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

TEMP_ROOT = ROOT / "models" / f".modernbert_trials_{STAMP}"
REPORT_DIR = ROOT / "reports" / f"modernbert_tuning_{STAMP}"
FINAL_MODEL_DIR = ROOT / "models" / "modernbert_best"

TEMP_ROOT.mkdir(parents=True, exist_ok=False)
REPORT_DIR.mkdir(parents=True, exist_ok=False)

# --------------------------- SAFETY CHECKS ----------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. Enable GPU in Kaggle notebook settings."
    )

for path in [TRAIN_PATH, VAL_PATH, BERT_MODEL_DIR / "label_mapping.json"]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

device = torch.device("cuda")

print("=" * 76)
print("MODERNBERT FINE-TUNING AND HYPERPARAMETER SEARCH")
print("=" * 76)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Optuna:", optuna.__version__)
print("Model:", MODEL_ID)
print("Maximum input length:", MAX_LENGTH)
print("Study report directory:", REPORT_DIR)

# ----------------------------- LOAD DATA ------------------------------

train_df = pd.read_parquet(TRAIN_PATH).reset_index(drop=True)
val_df = pd.read_parquet(VAL_PATH).reset_index(drop=True)

for split_name, frame in [
    ("train", train_df),
    ("validation", val_df),
]:
    missing = {TEXT_COL, LABEL_COL} - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} split is missing columns: {missing}")

    frame[TEXT_COL] = frame[TEXT_COL].fillna("").astype(str)
    frame[LABEL_COL] = frame[LABEL_COL].astype(str).str.strip()

# Reuse the BERT mapping so category IDs are identical across both models.
with open(BERT_MODEL_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

label2id = {
    str(label): int(idx)
    for label, idx in mapping["label2id"].items()
}
id2label = {idx: label for label, idx in label2id.items()}

classes = [id2label[i] for i in range(len(id2label))]
label_ids = np.arange(len(classes))
num_labels = len(classes)

if set(id2label) != set(range(num_labels)):
    raise ValueError("Class IDs must be contiguous from 0 through num_labels - 1.")

unknown_train = sorted(set(train_df[LABEL_COL]) - set(label2id))
unknown_val = sorted(set(val_df[LABEL_COL]) - set(label2id))

if unknown_train or unknown_val:
    raise ValueError(
        f"Label mismatch: unknown training labels={unknown_train[:10]}, "
        f"unknown validation labels={unknown_val[:10]}"
    )

if set(train_df[LABEL_COL].unique()) != set(classes):
    raise ValueError(
        "The training split's category set does not match BERT's saved "
        "label mapping. Resolve the mapping before fine-tuning."
    )

y_train = train_df[LABEL_COL].map(label2id).to_numpy(dtype=np.int64)
y_val = val_df[LABEL_COL].map(label2id).to_numpy(dtype=np.int64)

print(f"Train records:      {len(train_df):,}")
print(f"Validation records: {len(val_df):,}")
print(f"Categories:         {num_labels}")

if num_labels != 43:
    print(f"WARNING: expected 43 categories, found {num_labels}.")

# --------------------------- TOKENIZER --------------------------------

if TOKENIZER_PATH.exists():
    tokenizer = AutoTokenizer.from_pretrained(
        str(TOKENIZER_PATH),
        local_files_only=True,
    )
    print("Loaded local ModernBERT tokenizer:", TOKENIZER_PATH)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    print("Downloaded ModernBERT tokenizer from the model repository.")

# ModernBERT is designed for right padding.
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_LENGTH

if tokenizer.pad_token_id is None:
    raise ValueError("ModernBERT tokenizer has no pad token.")

# ------------------------ TOKENIZE TRAIN/VAL --------------------------

def encode_split(texts, split_name):
    print(f"\nTokenizing {split_name}...")
    result = tokenizer(
        texts.tolist(),
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=True,
    )

    lengths = np.asarray([len(row) for row in result["input_ids"]])
    print(
        f"{split_name}: count={len(lengths):,} | "
        f"median={np.median(lengths):.0f} | "
        f"p95={np.percentile(lengths, 95):.0f} | "
        f"at limit={np.sum(lengths >= MAX_LENGTH):,} | "
        f"observed max after truncation={lengths.max()}"
    )

    return result, lengths


train_enc, train_lengths = encode_split(
    train_df[TEXT_COL], "Train"
)
val_enc, val_lengths = encode_split(
    val_df[TEXT_COL], "Validation"
)

class ResumeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        row = {key: value[index] for key, value in self.encodings.items()}
        row["labels"] = int(self.labels[index])
        return row


train_dataset = ResumeDataset(train_enc, y_train)
val_dataset = ResumeDataset(val_enc, y_val)


def collate_batch(features):
    labels = torch.tensor(
        [item["labels"] for item in features],
        dtype=torch.long,
    )
    inputs = [
        {key: value for key, value in item.items() if key != "labels"}
        for item in features
    ]

    batch = tokenizer.pad(
        inputs,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )
    batch["labels"] = labels
    return batch


def make_loaders(train_batch_size, eval_batch_size):
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True,
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True,
    )

    return train_loader, val_loader


# ---------------------------- MODEL BUILDER ---------------------------

def create_model(classifier_dropout=0.1):
    config = AutoConfig.from_pretrained(MODEL_ID)

    supported_length = getattr(
        config, "max_position_embeddings", MAX_LENGTH
    )
    if MAX_LENGTH > supported_length:
        raise ValueError(
            f"MAX_LENGTH={MAX_LENGTH} exceeds model limit {supported_length}."
        )

    config.num_labels = num_labels
    config.id2label = id2label
    config.label2id = label2id
    config.classifier_dropout = classifier_dropout

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        config=config,
    )

    embedding_rows = model.get_input_embeddings().num_embeddings
    if embedding_rows != len(tokenizer):
        raise ValueError(
            f"Tokenizer vocabulary ({len(tokenizer)}) does not match model "
            f"embedding table ({embedding_rows}). Check the local tokenizer."
        )

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    model.to(device)
    return model


# ---------------------- MEMORY-SAFETY PREFLIGHT -----------------------

# Test a batch containing the longest tokenized training resumes. If batch
# size 4 does not fit, try 2 and then 1. This does not update model weights.
TRAIN_BATCH_SIZE = None

print("\nRunning a memory preflight using the longest training resumes...")

longest_indices = np.argsort(train_lengths)[::-1]
preflight_passed = False

for candidate_batch in BATCH_CANDIDATES:
    preflight_model = None
    preflight_batch = None

    try:
        preflight_model = create_model(classifier_dropout=0.1)
        preflight_model.train()

        indices = longest_indices[:candidate_batch]
        features = [
            {
                key: values[i]
                for key, values in train_enc.items()
            } | {"labels": int(y_train[i])}
            for i in indices
        ]

        preflight_batch = collate_batch(features)
        preflight_batch = {
            key: value.to(device)
            for key, value in preflight_batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            output = preflight_model(**preflight_batch)

        output.loss.backward()

        TRAIN_BATCH_SIZE = candidate_batch
        preflight_passed = True
        print(f"Memory preflight passed with physical batch size {candidate_batch}.")
        break

    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print(f"Batch size {candidate_batch} exceeded GPU memory. Trying smaller.")

    finally:
        if preflight_model is not None:
            del preflight_model
        if preflight_batch is not None:
            del preflight_batch
        gc.collect()
        torch.cuda.empty_cache()

if not preflight_passed:
    raise RuntimeError(
        "ModernBERT cannot complete a backward pass even at batch size 1 "
        "with 2048-token inputs on this GPU. Do not continue with the search. "
        "We need to lower the input limit or change the compute setup."
    )

EVAL_BATCH_SIZE = min(EVAL_BATCH_SIZE, TRAIN_BATCH_SIZE)
print("Training batch size:", TRAIN_BATCH_SIZE)
print("Evaluation batch size:", EVAL_BATCH_SIZE)

# --------------------------- EVALUATION -------------------------------

def evaluate_model(model, loader):
    model.eval()

    y_true_all = []
    y_pred_all = []
    loss_sum = 0.0
    number_seen = 0

    with torch.inference_mode():
        for batch in loader:
            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):
                outputs = model(**batch)

            n = len(batch["labels"])
            loss_sum += outputs.loss.item() * n
            number_seen += n

            predictions = outputs.logits.argmax(dim=-1)
            y_true_all.extend(batch["labels"].cpu().tolist())
            y_pred_all.extend(predictions.cpu().tolist())

    y_true_all = np.asarray(y_true_all, dtype=np.int64)
    y_pred_all = np.asarray(y_pred_all, dtype=np.int64)

    return {
        "loss": loss_sum / max(number_seen, 1),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(
            y_true_all, y_pred_all
        ),
        "macro_f1": f1_score(
            y_true_all,
            y_pred_all,
            labels=label_ids,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true_all,
            y_pred_all,
            labels=label_ids,
            average="weighted",
            zero_division=0,
        ),
        "y_true": y_true_all,
        "y_pred": y_pred_all,
    }


# -------------------------- TRAINING TRIAL ----------------------------

epoch_history = []


def train_trial(trial):
    hp = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-5, 5e-5, log=True
        ),
        "weight_decay": trial.suggest_categorical(
            "weight_decay", [0.0, 0.01, 0.03, 0.05]
        ),
        "warmup_ratio": trial.suggest_categorical(
            "warmup_ratio", [0.03, 0.06, 0.10]
        ),
        "grad_accum": trial.suggest_categorical(
            "grad_accum", GRAD_ACCUM_CHOICES
        ),
        "classifier_dropout": trial.suggest_categorical(
            "classifier_dropout", [0.0, 0.1, 0.2]
        ),
    }

    trial_id = trial.number
    checkpoint_dir = TEMP_ROOT / f"trial_{trial_id:02d}" / "best_model"
    checkpoint_dir.parent.mkdir(parents=True, exist_ok=True)
    trial.set_user_attr("checkpoint_path", str(checkpoint_dir))

    print("\n" + "=" * 72)
    print(f"MODERNBERT TRIAL {trial_id + 1}/{N_TRIALS}")
    print(json.dumps(hp, indent=2))
    print("=" * 72)

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    train_loader, val_loader = make_loaders(
        TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE
    )

    model = None
    optimizer = None
    scheduler = None
    scaler = None

    try:
        model = create_model(
            classifier_dropout=hp["classifier_dropout"]
        )
        model.train()

        # Standard AdamW with decay excluded from biases and LayerNorm.
        decay_params, no_decay_params = [], []

        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue

            if "bias" in name or "LayerNorm.weight" in name:
                no_decay_params.append(param)
            else:
                decay_params.append(param)

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": decay_params,
                    "weight_decay": hp["weight_decay"],
                },
                {
                    "params": no_decay_params,
                    "weight_decay": 0.0,
                },
            ],
            lr=hp["learning_rate"],
            betas=(0.9, 0.999),
            eps=ADAM_EPSILON,
        )

        optimizer_steps_per_epoch = int(
            np.ceil(len(train_loader) / hp["grad_accum"])
        )
        total_steps = optimizer_steps_per_epoch * MAX_EPOCHS
        warmup_steps = int(total_steps * hp["warmup_ratio"])

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        scaler = torch.amp.GradScaler("cuda", enabled=True)

        best_score = None
        best_metrics = None
        best_epoch = None
        bad_epochs = 0
        start_time = time.time()

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            epoch_loss_sum = 0.0
            examples_seen = 0
            n_batches = len(train_loader)

            for step, batch in enumerate(train_loader, start=1):
                batch = {
                    key: value.to(device, non_blocking=True)
                    for key, value in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    outputs = model(**batch)
                    raw_loss = outputs.loss
                    loss = raw_loss / hp["grad_accum"]

                scaler.scale(loss).backward()

                batch_n = len(batch["labels"])
                epoch_loss_sum += raw_loss.item() * batch_n
                examples_seen += batch_n

                update_now = (
                    step % hp["grad_accum"] == 0
                    or step == n_batches
                )

                if update_now:
                    scaler.unscale_(optimizer)

                    # Correct loss scaling for a final partial accumulation.
                    remainder = step % hp["grad_accum"]
                    if step == n_batches and remainder:
                        correction = hp["grad_accum"] / remainder
                        for parameter in model.parameters():
                            if parameter.grad is not None:
                                parameter.grad.mul_(correction)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), MAX_GRAD_NORM
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

                if step % 250 == 0 or step == n_batches:
                    print(
                        f"trial={trial_id + 1} | epoch={epoch}/{MAX_EPOCHS} | "
                        f"batch={step}/{n_batches}"
                    )

            train_loss = epoch_loss_sum / max(examples_seen, 1)
            val_result = evaluate_model(model, val_loader)

            val_loss = float(val_result["loss"])
            val_accuracy = float(val_result["accuracy"])
            val_macro_f1 = float(val_result["macro_f1"])
            val_weighted_f1 = float(val_result["weighted_f1"])

            row = {
                "trial": trial_id,
                "epoch": epoch,
                **hp,
                "train_loss": train_loss,
                "validation_loss": val_loss,
                "validation_accuracy": val_accuracy,
                "validation_macro_f1": val_macro_f1,
                "validation_weighted_f1": val_weighted_f1,
                "learning_rate": scheduler.get_last_lr()[0],
            }
            epoch_history.append(row)
            pd.DataFrame(epoch_history).to_csv(
                REPORT_DIR / "modernbert_epoch_history.csv",
                index=False,
            )

            print(
                f"\nTrial {trial_id + 1}, epoch {epoch}: "
                f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
                f"accuracy={val_accuracy:.4f} | macro_F1={val_macro_f1:.4f} | "
                f"weighted_F1={val_weighted_f1:.4f}"
            )

            # Match the BERT selection criterion: validation accuracy first,
            # macro-F1 second, and lower validation loss third.
            score = (val_accuracy, val_macro_f1, -val_loss)

            if best_score is None or score > best_score:
                best_score = score
                best_epoch = epoch
                bad_epochs = 0

                best_metrics = {
                    "best_val_accuracy": val_accuracy,
                    "best_val_macro_f1": val_macro_f1,
                    "best_val_weighted_f1": val_weighted_f1,
                    "best_val_loss": val_loss,
                    "best_epoch": epoch,
                }

                if checkpoint_dir.exists():
                    shutil.rmtree(checkpoint_dir)
                checkpoint_dir.mkdir(parents=True, exist_ok=True)

                model.save_pretrained(checkpoint_dir)
                tokenizer.save_pretrained(checkpoint_dir)

                with open(
                    checkpoint_dir / "training_selection.json",
                    "w",
                    encoding="utf-8",
                ) as f:
                    json.dump({
                        "trial": trial_id,
                        "hyperparameters": hp,
                        **best_metrics,
                    }, f, indent=2)

                trial.set_user_attr("best_val_accuracy", val_accuracy)
                trial.set_user_attr("best_val_macro_f1", val_macro_f1)
                trial.set_user_attr("best_val_loss", val_loss)
                trial.set_user_attr("best_epoch", epoch)

                print("Saved improved trial checkpoint.")

            else:
                bad_epochs += 1

            # Report validation accuracy to Optuna's pruning scheduler.
            trial.report(val_accuracy, step=epoch)

            if trial.should_prune():
                print("Optuna pruned this trial based on intermediate validation accuracy.")
                raise optuna.TrialPruned(
                    f"Pruned after epoch {epoch}."
                )

            if bad_epochs >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Early stopping after {EARLY_STOPPING_PATIENCE} epochs "
                    "without checkpoint improvement."
                )
                break

        trial.set_user_attr("training_seconds", time.time() - start_time)

        print("\nBest metrics for this trial:")
        print(json.dumps(best_metrics, indent=2))

        # Optuna's main objective is validation accuracy.
        return best_metrics["best_val_accuracy"]

    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            gc.collect()
            torch.cuda.empty_cache()
            print(
                "GPU OOM: this trial was stopped. If this repeats, use a "
                "smaller training batch and rerun the search."
            )
            raise optuna.TrialPruned("GPU out of memory") from exc
        raise

    finally:
        if model is not None:
            del model
        if optimizer is not None:
            del optimizer
        if scheduler is not None:
            del scheduler
        if scaler is not None:
            del scaler

        del train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()


# ------------------------ OPTUNA TPE STUDY ----------------------------

STUDY_NAME = f"modernbert_resume_tuning_{STAMP}"
DB_PATH = REPORT_DIR / "optuna_study.db"
STORAGE_URL = f"sqlite:///{DB_PATH}"

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=2,
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=2,
    n_warmup_steps=1,
    interval_steps=1,
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    load_if_exists=True,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
)

print("\nStarting Optuna optimization...")
print("Sampler: TPE")
print("Pruner: MedianPruner")
print("Objective: validation accuracy")
print("Maximum trials:", N_TRIALS)

study.optimize(
    train_trial,
    n_trials=N_TRIALS,
    gc_after_trial=True,
    show_progress_bar=False,
)

# -------------------------- SAVE SEARCH RESULTS -----------------------

trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "state", "user_attrs")
)
trials_df.to_csv(
    REPORT_DIR / "modernbert_hyperparameter_trials.csv",
    index=False,
)

completed = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]

if not completed:
    raise RuntimeError(
        "No ModernBERT trial completed successfully. Review the logs and "
        "memory preflight output before rerunning."
    )

best_trial = study.best_trial
best_checkpoint = Path(best_trial.user_attrs["checkpoint_path"])

if not (best_checkpoint / "config.json").exists():
    raise FileNotFoundError(
        f"Winning checkpoint is missing: {best_checkpoint}"
    )

print("\n" + "=" * 76)
print("MODERNBERT SEARCH COMPLETE")
print("=" * 76)
print("Best trial:", best_trial.number + 1)
print("Winning parameters:")
print(json.dumps(best_trial.params, indent=2))
print("Best validation accuracy:", best_trial.value)
print("Best validation macro-F1:", best_trial.user_attrs.get("best_val_macro_f1"))
print("Best epoch:", best_trial.user_attrs.get("best_epoch"))

# ------------------------- PROMOTE WINNER -----------------------------

# Preserve any prior best ModernBERT checkpoint rather than overwriting it.
if FINAL_MODEL_DIR.exists():
    previous_dir = ROOT / "models" / f"modernbert_best_previous_{STAMP}"
    FINAL_MODEL_DIR.rename(previous_dir)
    print("Previous checkpoint preserved at:", previous_dir)

shutil.copytree(best_checkpoint, FINAL_MODEL_DIR)

# Add the same label mapping used by BERT.
with open(
    FINAL_MODEL_DIR / "label_mapping.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump({
        "label2id": label2id,
        "id2label": {str(k): v for k, v in id2label.items()},
        "num_labels": num_labels,
    }, f, indent=2, ensure_ascii=False)

summary = {
    "model": MODEL_ID,
    "max_length": MAX_LENGTH,
    "selection_metric": "validation_accuracy",
    "tie_breakers": ["validation_macro_f1", "lower_validation_loss"],
    "objective_value": best_trial.value,
    "best_trial_number": best_trial.number + 1,
    "best_epoch": best_trial.user_attrs.get("best_epoch"),
    "best_hyperparameters": best_trial.params,
    "best_validation_metrics": {
        "accuracy": best_trial.user_attrs.get("best_val_accuracy"),
        "macro_f1": best_trial.user_attrs.get("best_val_macro_f1"),
        "loss": best_trial.user_attrs.get("best_val_loss"),
    },
    "training_batch_size": TRAIN_BATCH_SIZE,
    "effective_batch_size_options": [
        TRAIN_BATCH_SIZE * accum for accum in GRAD_ACCUM_CHOICES
    ],
    "seed": SEED,
    "num_labels": num_labels,
    "checkpoint_path": str(FINAL_MODEL_DIR),
    "study_name": STUDY_NAME,
    "optuna_storage": str(DB_PATH),
}

with open(
    REPORT_DIR / "modernbert_best_hyperparameters.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(summary, f, indent=2)

with open(
    FINAL_MODEL_DIR / "experiment_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(summary, f, indent=2)

# ---------------------- VERIFY WINNER ON VALIDATION ------------------

print("\nReloading the saved winner for final validation verification...")

best_tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_MODEL_DIR),
    local_files_only=True,
)
best_tokenizer.padding_side = "right"

best_model = AutoModelForSequenceClassification.from_pretrained(
    str(FINAL_MODEL_DIR),
    local_files_only=True,
).to(device)

best_model.eval()

_, final_val_loader = make_loaders(
    TRAIN_BATCH_SIZE,
    EVAL_BATCH_SIZE,
)

final_val = evaluate_model(best_model, final_val_loader)

val_metrics = {
    "accuracy": float(final_val["accuracy"]),
    "balanced_accuracy": float(final_val["balanced_accuracy"]),
    "macro_f1": float(final_val["macro_f1"]),
    "weighted_f1": float(final_val["weighted_f1"]),
    "loss": float(final_val["loss"]),
    "samples": len(y_val),
    "classes": num_labels,
}

with open(
    REPORT_DIR / "modernbert_final_validation_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(val_metrics, f, indent=2)

val_report = classification_report(
    final_val["y_true"],
    final_val["y_pred"],
    labels=label_ids,
    target_names=classes,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(val_report).transpose().to_csv(
    REPORT_DIR / "modernbert_validation_classification_report.csv"
)

val_cm = confusion_matrix(
    final_val["y_true"],
    final_val["y_pred"],
    labels=label_ids,
)
pd.DataFrame(val_cm, index=classes, columns=classes).to_csv(
    REPORT_DIR / "modernbert_validation_confusion_matrix_counts.csv"
)

val_row_totals = val_cm.sum(axis=1, keepdims=True)
val_cm_norm = np.divide(
    val_cm.astype(np.float64),
    val_row_totals,
    out=np.zeros_like(val_cm, dtype=np.float64),
    where=val_row_totals != 0,
)
pd.DataFrame(val_cm_norm, index=classes, columns=classes).to_csv(
    REPORT_DIR / "modernbert_validation_confusion_matrix_normalized.csv"
)

# Save successful checkpoints and history before cleaning temporary trials.
print("\nVerified saved ModernBERT validation metrics:")
for metric, value in val_metrics.items():
    print(f"{metric:25s}: {value}")

print("\nSaved winning model:", FINAL_MODEL_DIR)
print("Saved reports:", REPORT_DIR)
print("Saved trial database:", DB_PATH)

# Keep the actual winner in models/modernbert_best; remove temporary trial
# copies to avoid using unnecessary disk space.
shutil.rmtree(TEMP_ROOT, ignore_errors=True)

del best_model, best_tokenizer, final_val_loader
gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 76)
print("MODERNBERT FINE-TUNING COMPLETE")
print("The test split has not been loaded or evaluated.")
print("Next: run a separate test-evaluation cell against the saved winner.")
print("=" * 76)


## 10. ModernBERT: preflight and refined memory-safe tuning

Check saved prerequisites and the visible GPU devices before launching the revised Optuna search. The refined run uses length-bucketed batches, gradient checkpointing, mixed precision, conservative physical batches, and validation metrics for checkpoint selection.

In [ ]:

from pathlib import Path
import importlib.util
import importlib.metadata
import torch

ROOT = Path("/kaggle/working/resume_classification")

required_files = [
    ROOT / "data/splits/train.parquet",
    ROOT / "data/splits/validation.parquet",
    ROOT / "models/bert_best/label_mapping.json",
]

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for package in ["torch", "transformers", "optuna", "pandas", "scikit-learn"]:
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nRequired files:")
all_files_exist = True
for path in required_files:
    exists = path.exists()
    print(f"{'OK' if exists else 'MISSING'}: {path}")
    all_files_exist &= exists

tokenizer_dir = ROOT / "models/tokenizers/modernbert"
print(
    "\nModernBERT tokenizer:",
    "FOUND" if tokenizer_dir.exists() else
    "NOT FOUND — download will be needed"
)

if not all_files_exist:
    print("\nSTOP: restore the missing project files before training.")
elif not torch.cuda.is_available():
    print("\nSTOP: enable a GPU before training.")
elif importlib.util.find_spec("optuna") is None:
    print("\nSTOP: install Optuna in a separate cell, then rerun this check.")
else:
    print("\nPrerequisite checks passed. You can run the ModernBERT cell.")


In [ ]:

# =====================================================================
# MODERNBERT: MEMORY-SAFE FINE-TUNING + OPTUNA HYPERPARAMETER SEARCH
# - 2048-token input limit
# - Length-bucketed batches
# - Gradient checkpointing + FP16
# - Multi-GPU DataParallel if multiple GPUs are visible
# - Adaptive physical batch selection
# - Four trials, up to three epochs per trial
# - Validation checkpoint selection only; test set untouched
# =====================================================================

import gc
import json
import math
import random
import shutil
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Install Optuna first in a separate cell with %pip install optuna."
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)

# -------------------------- CONFIGURATION -----------------------------

MB_ROOT = Path("/kaggle/working/resume_classification")
MB_TRAIN_PATH = MB_ROOT / "data/splits/train.parquet"
MB_VAL_PATH = MB_ROOT / "data/splits/validation.parquet"
MB_BERT_DIR = MB_ROOT / "models/bert_best"
MB_TOKENIZER_DIR = MB_ROOT / "models/tokenizers/modernbert"

MB_MODEL_ID = "answerdotai/ModernBERT-base"
MB_TEXT_COL = "Text_prepared"
MB_LABEL_COL = "Category"

MB_SEED = 42
MB_MAX_LENGTH = 2048
MB_N_TRIALS = 4
MB_MAX_EPOCHS = 3
MB_PATIENCE = 2
MB_TARGET_EFFECTIVE_BATCH = 16
MB_MAX_GRAD_NORM = 1.0
MB_HEADROOM_RATIO = 0.85

MB_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
MB_REPORT_DIR = MB_ROOT / "reports" / f"modernbert_fixed_tuning_{MB_STAMP}"
MB_TRIAL_ROOT = MB_REPORT_DIR / "trial_checkpoints"
MB_FINAL_DIR = MB_ROOT / "models/modernbert_best"

# ---------------------------- GPU CHECK -------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is visible. Enable GPU in Kaggle notebook settings."
    )

MB_DEVICE = torch.device("cuda:0")
MB_GPU_COUNT = torch.cuda.device_count()
MB_GPU_IDS = list(range(MB_GPU_COUNT))

print("=" * 76)
print("MODERNBERT TRAINING ENVIRONMENT")
print("=" * 76)
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("Optuna:", optuna.__version__)
print("Visible CUDA GPUs:", MB_GPU_COUNT)

for i in MB_GPU_IDS:
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"VRAM={props.total_memory / (1024**3):.2f} GiB"
    )

print(
    "Training strategy:",
    "DataParallel across visible GPUs" if MB_GPU_COUNT > 1
    else "Single visible GPU",
)
print("Input limit:", MB_MAX_LENGTH)

# Use new report/checkpoint folders. Don't touch BERT or its reports.
MB_REPORT_DIR.mkdir(parents=True, exist_ok=False)
MB_TRIAL_ROOT.mkdir(parents=True, exist_ok=False)

# Clear orphaned scratch directories left by the failed ModernBERT cells.
# Only hidden .modernbert_trials_* directories are targeted.
for old_dir in (MB_ROOT / "models").glob(".modernbert_trials_*"):
    if old_dir.is_dir():
        print("Removing old failed scratch directory:", old_dir)
        shutil.rmtree(old_dir, ignore_errors=True)

for path in [
    MB_TRAIN_PATH,
    MB_VAL_PATH,
    MB_BERT_DIR / "label_mapping.json",
]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# ---------------------------- LOAD SPLITS -----------------------------

MB_train_df = pd.read_parquet(MB_TRAIN_PATH).reset_index(drop=True)
MB_val_df = pd.read_parquet(MB_VAL_PATH).reset_index(drop=True)

for split_name, frame in [
    ("train", MB_train_df),
    ("validation", MB_val_df),
]:
    missing = {MB_TEXT_COL, MB_LABEL_COL} - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} split missing columns: {missing}")

    frame[MB_TEXT_COL] = frame[MB_TEXT_COL].fillna("").astype(str)
    frame[MB_LABEL_COL] = frame[MB_LABEL_COL].astype(str).str.strip()

# Use exactly the same label IDs as the fine-tuned BERT model.
with open(
    MB_BERT_DIR / "label_mapping.json", "r", encoding="utf-8"
) as f:
    MB_mapping = json.load(f)

MB_label2id = {
    str(label): int(idx)
    for label, idx in MB_mapping["label2id"].items()
}
MB_id2label = {
    idx: label for label, idx in MB_label2id.items()
}
MB_num_labels = len(MB_label2id)
MB_classes = [
    MB_id2label[i] for i in range(MB_num_labels)
]
MB_label_ids = np.arange(MB_num_labels)

if set(MB_id2label) != set(range(MB_num_labels)):
    raise ValueError("Label IDs are not contiguous.")

for split_name, frame in [
    ("train", MB_train_df),
    ("validation", MB_val_df),
]:
    unknown = sorted(set(frame[MB_LABEL_COL]) - set(MB_label2id))
    if unknown:
        raise ValueError(
            f"Unknown {split_name} categories: {unknown[:10]}"
        )

MB_y_train = MB_train_df[MB_LABEL_COL].map(
    MB_label2id
).to_numpy(dtype=np.int64)

MB_y_val = MB_val_df[MB_LABEL_COL].map(
    MB_label2id
).to_numpy(dtype=np.int64)

print(f"\nTrain resumes: {len(MB_train_df):,}")
print(f"Validation resumes: {len(MB_val_df):,}")
print(f"Categories: {MB_num_labels}")

# --------------------------- TOKENIZER --------------------------------

if MB_TOKENIZER_DIR.exists():
    MB_tokenizer = AutoTokenizer.from_pretrained(
        str(MB_TOKENIZER_DIR),
        local_files_only=True,
    )
    print("Loaded saved ModernBERT tokenizer.")
else:
    MB_tokenizer = AutoTokenizer.from_pretrained(MB_MODEL_ID)
    print("Downloaded ModernBERT tokenizer.")

MB_tokenizer.padding_side = "right"

if MB_tokenizer.pad_token_id is None:
    raise ValueError("ModernBERT tokenizer has no pad token.")

# ----------------------- TOKENIZATION ONCE ----------------------------

def MB_encode(texts, name):
    print(f"\nTokenizing {name}...")

    enc = MB_tokenizer(
        texts.tolist(),
        truncation=True,
        max_length=MB_MAX_LENGTH,
        padding=False,
        return_attention_mask=True,
    )

    lengths = np.asarray([len(row) for row in enc["input_ids"]])
    print(
        f"{name}: n={len(lengths):,} | "
        f"median={np.median(lengths):.0f} | "
        f"p95={np.percentile(lengths, 95):.0f} | "
        f"max={lengths.max()} | "
        f"at limit={np.sum(lengths >= MB_MAX_LENGTH):,}"
    )
    return enc, lengths


MB_train_enc, MB_train_lengths = MB_encode(
    MB_train_df[MB_TEXT_COL], "Train"
)
MB_val_enc, MB_val_lengths = MB_encode(
    MB_val_df[MB_TEXT_COL], "Validation"
)


class MB_ResumeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values in self.encodings.items()
        }
        item["labels"] = int(self.labels[idx])
        return item


MB_train_dataset = MB_ResumeDataset(MB_train_enc, MB_y_train)
MB_val_dataset = MB_ResumeDataset(MB_val_enc, MB_y_val)


def MB_collate(features):
    labels = torch.tensor(
        [f["labels"] for f in features],
        dtype=torch.long,
    )
    inputs = [
        {k: v for k, v in f.items() if k != "labels"}
        for f in features
    ]
    padded = MB_tokenizer.pad(
        inputs,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )
    padded["labels"] = labels
    return padded


# ---------------------- LENGTH-BUCKETED BATCHES -----------------------

class MB_LengthBucketBatchSampler:
    def __init__(self, lengths, batch_size, seed=42, bucket_multiplier=40):
        self.lengths = np.asarray(lengths)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.bucket_multiplier = int(bucket_multiplier)
        self.epoch = 0

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        self.epoch += 1

        indices = list(range(len(self.lengths)))
        rng.shuffle(indices)

        bucket_size = max(
            self.batch_size * self.bucket_multiplier,
            self.batch_size,
        )
        batches = []

        for start in range(0, len(indices), bucket_size):
            bucket = indices[start:start + bucket_size]
            bucket.sort(key=lambda i: self.lengths[i])

            for offset in range(0, len(bucket), self.batch_size):
                batches.append(bucket[offset:offset + self.batch_size])

        rng.shuffle(batches)
        yield from batches


# --------------------------- MODEL HELPERS ----------------------------

def MB_unwrap(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def MB_build_model(classifier_dropout=0.1, checkpointing=True):
    config = AutoConfig.from_pretrained(MB_MODEL_ID)
    model_limit = getattr(config, "max_position_embeddings", MB_MAX_LENGTH)

    if MB_MAX_LENGTH > model_limit:
        raise ValueError(
            f"Input limit {MB_MAX_LENGTH} exceeds ModernBERT's configured "
            f"limit of {model_limit}."
        )

    config.num_labels = MB_num_labels
    config.id2label = MB_id2label
    config.label2id = MB_label2id
    config.classifier_dropout = classifier_dropout

    base = AutoModelForSequenceClassification.from_pretrained(
        MB_MODEL_ID,
        config=config,
    )

    # A newly initialized classification head is expected for a new task.
    if base.get_input_embeddings().num_embeddings != len(MB_tokenizer):
        raise ValueError(
            "Tokenizer vocabulary does not match ModernBERT embeddings."
        )

    base.config.use_cache = False

    if checkpointing:
        base.gradient_checkpointing_enable()

    base.to(MB_DEVICE)

    if MB_GPU_COUNT > 1:
        return torch.nn.DataParallel(
            base,
            device_ids=MB_GPU_IDS,
            output_device=0,
        )

    return base


def MB_clear_cuda():
    gc.collect()
    for gpu_id in MB_GPU_IDS:
        with torch.cuda.device(gpu_id):
            torch.cuda.empty_cache()


# ------------------- MEMORY PREFLIGHT WITH BACKWARD ------------------

# Use conservative candidates. For multiple GPUs, physical batch size
# means the total batch distributed across the visible GPUs.
if MB_GPU_COUNT == 1:
    MB_batch_candidates = [4, 2, 1]
else:
    MB_batch_candidates = sorted(
        {
            max(4, 2 * MB_GPU_COUNT),
            max(2, MB_GPU_COUNT),
            MB_GPU_COUNT,
        },
        reverse=True,
    )

MB_batch_candidates = list(dict.fromkeys(MB_batch_candidates))

def MB_memory_probe(batch_size):
    model = None
    optimizer = None
    scaler = None
    batch = None
    peak_ratio = 1.0
    success = False

    try:
        MB_clear_cuda()
        for gpu_id in MB_GPU_IDS:
            torch.cuda.reset_peak_memory_stats(gpu_id)

        model = MB_build_model(
            classifier_dropout=0.1,
            checkpointing=True,
        )
        model.train()

        # Select the longest resumes to approximate a high-memory batch.
        longest = np.argsort(MB_train_lengths)[::-1][:batch_size]

        features = [
            {
                key: values[int(i)]
                for key, values in MB_train_enc.items()
            } | {"labels": int(MB_y_train[int(i)])}
            for i in longest
        ]

        batch = MB_collate(features)
        labels = batch.pop("labels").to(MB_DEVICE)
        inputs = {
            key: value.to(MB_DEVICE)
            for key, value in batch.items()
        }

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=2e-5,
        )
        scaler = torch.amp.GradScaler("cuda", enabled=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(
                **inputs,
                return_dict=False,
            )[0]
            loss = F.cross_entropy(logits.float(), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), MB_MAX_GRAD_NORM
        )
        scaler.step(optimizer)
        scaler.update()

        # Ensure AdamW optimizer states were also allocated.
        ratios = []
        for gpu_id in MB_GPU_IDS:
            peak_reserved = torch.cuda.max_memory_reserved(gpu_id)
            total = torch.cuda.get_device_properties(gpu_id).total_memory
            ratios.append(peak_reserved / total)

        peak_ratio = max(ratios)
        success = True

    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print(f"OOM in memory preflight at physical batch={batch_size}.")

    finally:
        if model is not None:
            del model
        if optimizer is not None:
            del optimizer
        if scaler is not None:
            del scaler
        if batch is not None:
            del batch
        MB_clear_cuda()

    return success, peak_ratio


print("\nMemory preflight: actual forward + backward + optimizer step")
MB_TRAIN_BATCH = None

for candidate_index, candidate in enumerate(MB_batch_candidates):
    print(f"\nTesting physical batch size {candidate}...")
    passed, peak_ratio = MB_memory_probe(candidate)

    if not passed:
        continue

    print(f"Peak reserved-memory ratio: {peak_ratio:.1%}")

    # If the probe leaves little memory headroom, try a smaller batch.
    if (
        peak_ratio > MB_HEADROOM_RATIO
        and candidate_index < len(MB_batch_candidates) - 1
    ):
        print("Insufficient memory headroom; testing a smaller batch.")
        continue

    MB_TRAIN_BATCH = candidate
    break

if MB_TRAIN_BATCH is None:
    raise RuntimeError(
        "No physical batch size passed the memory preflight, even with "
        "gradient checkpointing. Try MAX_LENGTH=1536 or use a larger GPU."
    )

# Maintain approximately the same effective batch size for all trials.
MB_GRAD_ACCUM = max(
    1,
    math.ceil(MB_TARGET_EFFECTIVE_BATCH / MB_TRAIN_BATCH),
)
MB_EVAL_BATCH = max(
    MB_GPU_COUNT,
    min(4, MB_TRAIN_BATCH),
)

print("\nMemory preflight passed.")
print("Visible GPUs:", MB_GPU_COUNT)
print("Physical batch size:", MB_TRAIN_BATCH)
print("Gradient accumulation:", MB_GRAD_ACCUM)
print("Effective batch size:", MB_TRAIN_BATCH * MB_GRAD_ACCUM)
print("Gradient checkpointing: enabled")
print("Evaluation batch size:", MB_EVAL_BATCH)

# ------------------------------ LOADERS -------------------------------

def MB_make_loaders():
    sampler = MB_LengthBucketBatchSampler(
        MB_train_lengths,
        batch_size=MB_TRAIN_BATCH,
        seed=MB_SEED,
    )

    train_loader = DataLoader(
        MB_train_dataset,
        batch_sampler=sampler,
        collate_fn=MB_collate,
        num_workers=0,
        pin_memory=True,
    )

    val_loader = DataLoader(
        MB_val_dataset,
        batch_size=MB_EVAL_BATCH,
        shuffle=False,
        collate_fn=MB_collate,
        num_workers=0,
        pin_memory=True,
    )

    expected_batches = math.ceil(
        len(MB_train_dataset) / MB_TRAIN_BATCH
    )

    if len(train_loader) != expected_batches:
        raise RuntimeError(
            f"Batch-size mismatch: expected {expected_batches} batches but "
            f"loader has {len(train_loader)}. Training aborted before tuning."
        )

    print(
        f"Verified loader: physical_batch={MB_TRAIN_BATCH}, "
        f"batches_per_epoch={len(train_loader)}"
    )

    return train_loader, val_loader


# ----------------------------- EVALUATION -----------------------------

def MB_evaluate(model, loader):
    model.eval()

    true_all = []
    pred_all = []
    loss_sum = 0.0
    count = 0

    with torch.inference_mode():
        for batch in loader:
            labels = batch.pop("labels").to(MB_DEVICE)
            inputs = {
                k: v.to(MB_DEVICE, non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True,
            ):
                logits = model(**inputs, return_dict=False)[0]

            loss = F.cross_entropy(
                logits.float(),
                labels,
                reduction="sum",
            )

            n = len(labels)
            loss_sum += loss.item()
            count += n

            true_all.extend(labels.cpu().tolist())
            pred_all.extend(logits.argmax(dim=-1).cpu().tolist())

    y_true = np.asarray(true_all, dtype=np.int64)
    y_pred = np.asarray(pred_all, dtype=np.int64)

    return {
        "loss": loss_sum / max(count, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=MB_label_ids,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=MB_label_ids,
            average="weighted",
            zero_division=0,
        ),
        "y_true": y_true,
        "y_pred": y_pred,
    }


# ------------------------------- TRIALS -------------------------------

MB_epoch_history = []


def MB_train_trial(trial):
    hp = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-5, 4e-5, log=True
        ),
        "weight_decay": trial.suggest_categorical(
            "weight_decay", [0.01, 0.03, 0.05]
        ),
        "warmup_ratio": trial.suggest_categorical(
            "warmup_ratio", [0.03, 0.06, 0.10]
        ),
        "classifier_dropout": trial.suggest_categorical(
            "classifier_dropout", [0.0, 0.1, 0.2]
        ),
    }

    trial_id = trial.number
    checkpoint_dir = (
        MB_TRIAL_ROOT / f"trial_{trial_id:02d}" / "best_model"
    )
    checkpoint_dir.parent.mkdir(parents=True, exist_ok=True)
    trial.set_user_attr("checkpoint_path", str(checkpoint_dir))

    print("\n" + "=" * 72)
    print(f"ModernBERT trial {trial_id + 1}/{MB_N_TRIALS}")
    print(json.dumps(hp, indent=2))
    print("=" * 72)

    trial_seed = MB_SEED + trial_id
    random.seed(trial_seed)
    np.random.seed(trial_seed)
    torch.manual_seed(trial_seed)
    torch.cuda.manual_seed_all(trial_seed)

    train_loader, val_loader = MB_make_loaders()
    model = optimizer = scheduler = scaler = None

    try:
        model = MB_build_model(
            classifier_dropout=hp["classifier_dropout"],
            checkpointing=True,
        )

        decay_params = []
        no_decay_params = []

        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue

            lname = name.lower()
            if lname.endswith("bias") or "norm" in lname:
                no_decay_params.append(param)
            else:
                decay_params.append(param)

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": decay_params,
                    "weight_decay": hp["weight_decay"],
                },
                {
                    "params": no_decay_params,
                    "weight_decay": 0.0,
                },
            ],
            lr=hp["learning_rate"],
            betas=(0.9, 0.999),
            eps=1e-8,
        )

        update_steps = math.ceil(
            len(train_loader) / MB_GRAD_ACCUM
        )
        total_steps = update_steps * MB_MAX_EPOCHS
        warmup_steps = int(total_steps * hp["warmup_ratio"])

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
        scaler = torch.amp.GradScaler("cuda", enabled=True)

        best_score = None
        best_metrics = None
        stale_epochs = 0

        for epoch in range(1, MB_MAX_EPOCHS + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            train_loss_sum = 0.0
            train_count = 0
            total_batches = len(train_loader)

            for step, batch in enumerate(train_loader, start=1):
                labels = batch.pop("labels").to(MB_DEVICE)
                inputs = {
                    k: v.to(MB_DEVICE, non_blocking=True)
                    for k, v in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    logits = model(
                        **inputs,
                        return_dict=False,
                    )[0]
                    raw_loss = F.cross_entropy(
                        logits.float(), labels
                    )
                    scaled_loss = raw_loss / MB_GRAD_ACCUM

                scaler.scale(scaled_loss).backward()

                batch_n = len(labels)
                train_loss_sum += raw_loss.item() * batch_n
                train_count += batch_n

                update_now = (
                    step % MB_GRAD_ACCUM == 0
                    or step == total_batches
                )

                if update_now:
                    scaler.unscale_(optimizer)

                    remainder = step % MB_GRAD_ACCUM
                    if step == total_batches and remainder:
                        correction = MB_GRAD_ACCUM / remainder
                        for parameter in model.parameters():
                            if parameter.grad is not None:
                                parameter.grad.mul_(correction)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        MB_MAX_GRAD_NORM,
                    )

                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

                if step % 250 == 0 or step == total_batches:
                    print(
                        f"trial={trial_id + 1} | epoch={epoch}/{MB_MAX_EPOCHS} | "
                        f"batch={step}/{total_batches}"
                    )

            train_loss = train_loss_sum / max(train_count, 1)
            val = MB_evaluate(model, val_loader)

            # Match the previous BERT search: accuracy first,
            # macro-F1 second, validation loss as the final tie-breaker.
            score = (
                float(val["accuracy"]),
                float(val["macro_f1"]),
                -float(val["loss"]),
            )

            row = {
                "trial": trial_id,
                "epoch": epoch,
                **hp,
                "physical_batch_size": MB_TRAIN_BATCH,
                "gradient_accumulation": MB_GRAD_ACCUM,
                "train_loss": train_loss,
                "validation_loss": val["loss"],
                "validation_accuracy": val["accuracy"],
                "validation_macro_f1": val["macro_f1"],
                "validation_weighted_f1": val["weighted_f1"],
            }
            MB_epoch_history.append(row)
            pd.DataFrame(MB_epoch_history).to_csv(
                MB_REPORT_DIR / "modernbert_epoch_history.csv",
                index=False,
            )

            print(
                f"\nTrial {trial_id + 1}, epoch {epoch}: "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val['loss']:.4f} | "
                f"accuracy={val['accuracy']:.4f} | "
                f"macro_F1={val['macro_f1']:.4f} | "
                f"weighted_F1={val['weighted_f1']:.4f}"
            )

            if best_score is None or score > best_score:
                best_score = score
                stale_epochs = 0

                best_metrics = {
                    "best_val_accuracy": float(val["accuracy"]),
                    "best_val_macro_f1": float(val["macro_f1"]),
                    "best_val_weighted_f1": float(val["weighted_f1"]),
                    "best_val_loss": float(val["loss"]),
                    "best_epoch": int(epoch),
                }

                if checkpoint_dir.exists():
                    shutil.rmtree(checkpoint_dir)
                checkpoint_dir.mkdir(parents=True, exist_ok=True)

                MB_unwrap(model).save_pretrained(checkpoint_dir)
                MB_tokenizer.save_pretrained(checkpoint_dir)

                for key, value in best_metrics.items():
                    trial.set_user_attr(key, value)

                print("Saved improved trial checkpoint.")
            else:
                stale_epochs += 1

            # MedianPruner gets accuracy after each epoch. The first epoch
            # is a warm-up, so trials are not pruned immediately.
            trial.report(float(val["accuracy"]), step=epoch - 1)

            if trial.should_prune():
                print("Optuna pruned this trial based on validation results.")
                raise optuna.TrialPruned(
                    f"Not competitive after epoch {epoch}"
                )

            if stale_epochs >= MB_PATIENCE:
                print("Early stopping: validation checkpoint has plateaued.")
                break

        trial.set_user_attr("training_seconds", float(
            time.time() - trial.datetime_start.timestamp()
        ) if trial.datetime_start else None)

        return best_metrics["best_val_accuracy"]

    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print(
                "CUDA OOM despite preflight. Trial will be pruned; "
                "the training loader and actual physical batch were verified."
            )
            raise optuna.TrialPruned("CUDA out of memory") from exc
        raise

    finally:
        if model is not None:
            del model
        if optimizer is not None:
            del optimizer
        if scheduler is not None:
            del scheduler
        if scaler is not None:
            del scaler
        del train_loader, val_loader
        MB_clear_cuda()


# -------------------------- OPTUNA SEARCH -----------------------------

MB_DB_PATH = MB_REPORT_DIR / "modernbert_study.db"
MB_STUDY = optuna.create_study(
    study_name=f"modernbert_tuning_{MB_STAMP}",
    storage=f"sqlite:///{MB_DB_PATH}",
    direction="maximize",
    load_if_exists=False,
    sampler=optuna.samplers.TPESampler(
        seed=MB_SEED,
        n_startup_trials=2,
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=2,
        n_warmup_steps=1,
        interval_steps=1,
    ),
)

print("\nStarting ModernBERT adaptive hyperparameter optimization.")
print(f"Trials={MB_N_TRIALS}; max epochs/trial={MB_MAX_EPOCHS}")
print("Primary objective: validation accuracy")
print("Tie-breakers: validation macro-F1, then lower validation loss")

MB_STUDY.optimize(
    MB_train_trial,
    n_trials=MB_N_TRIALS,
    gc_after_trial=True,
    show_progress_bar=False,
)

MB_STUDY.trials_dataframe(
    attrs=("number", "value", "params", "state", "user_attrs")
).to_csv(
    MB_REPORT_DIR / "modernbert_hyperparameter_trials.csv",
    index=False,
)

# Select from checkpoints that actually exist. This also allows a trial
# pruned after an epoch to contribute its saved validation checkpoint.
MB_candidates = []

for trial in MB_STUDY.trials:
    checkpoint_value = trial.user_attrs.get("checkpoint_path")
    accuracy = trial.user_attrs.get("best_val_accuracy")
    macro_f1 = trial.user_attrs.get("best_val_macro_f1")
    loss = trial.user_attrs.get("best_val_loss")

    if not checkpoint_value or accuracy is None:
        continue

    checkpoint_path = Path(checkpoint_value)
    if not (checkpoint_path / "config.json").exists():
        continue

    MB_candidates.append({
        "trial": trial,
        "path": checkpoint_path,
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1 if macro_f1 is not None else -1),
        "loss": float(loss if loss is not None else float("inf")),
    })

if not MB_candidates:
    raise RuntimeError(
        "No ModernBERT trial saved a checkpoint. All trials may have run "
        "out of memory before completing an epoch. Reduce MAX_LENGTH to "
        "1536 or use a larger GPU before rerunning."
    )

MB_winner = max(
    MB_candidates,
    key=lambda x: (
        x["accuracy"],
        x["macro_f1"],
        -x["loss"],
    ),
)

MB_best_trial = MB_winner["trial"]
MB_best_checkpoint = MB_winner["path"]

print("\n" + "=" * 76)
print("MODERNBERT TUNING COMPLETE")
print("=" * 76)
print("Selected trial:", MB_best_trial.number + 1)
print("Trial state:", MB_best_trial.state.name)
print("Selected hyperparameters:")
print(json.dumps(MB_best_trial.params, indent=2))
print("Validation accuracy:", MB_winner["accuracy"])
print("Validation macro-F1:", MB_winner["macro_f1"])
print("Validation loss:", MB_winner["loss"])

# ------------------------- SAVE WINNING MODEL -------------------------

if MB_FINAL_DIR.exists():
    MB_backup_dir = MB_ROOT / "models" / f"modernbert_best_previous_{MB_STAMP}"
    MB_FINAL_DIR.rename(MB_backup_dir)
    print("Previous ModernBERT checkpoint preserved at:", MB_backup_dir)

shutil.copytree(MB_best_checkpoint, MB_FINAL_DIR)

with open(
    MB_FINAL_DIR / "label_mapping.json", "w", encoding="utf-8"
) as f:
    json.dump(
        {
            "label2id": MB_label2id,
            "id2label": {
                str(idx): label for idx, label in MB_id2label.items()
            },
            "num_labels": MB_num_labels,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

MB_summary = {
    "model": MB_MODEL_ID,
    "input_limit_tokens": MB_MAX_LENGTH,
    "selection_metric": "validation_accuracy",
    "tie_breakers": ["validation_macro_f1", "lower_validation_loss"],
    "selected_trial": MB_best_trial.number + 1,
    "selected_trial_state": MB_best_trial.state.name,
    "hyperparameters": MB_best_trial.params,
    "validation_metrics": {
        "accuracy": MB_winner["accuracy"],
        "macro_f1": MB_winner["macro_f1"],
        "loss": MB_winner["loss"],
    },
    "physical_batch_size": MB_TRAIN_BATCH,
    "gradient_accumulation": MB_GRAD_ACCUM,
    "effective_batch_size": MB_TRAIN_BATCH * MB_GRAD_ACCUM,
    "gradient_checkpointing": True,
    "visible_gpu_count": MB_GPU_COUNT,
    "num_labels": MB_num_labels,
    "checkpoint": str(MB_FINAL_DIR),
    "study_database": str(MB_DB_PATH),
}

with open(
    MB_REPORT_DIR / "modernbert_best_hyperparameters.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(MB_summary, f, indent=2)

with open(
    MB_FINAL_DIR / "experiment_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(MB_summary, f, indent=2)

# --------------------- VERIFY WINNER ON VALIDATION -------------------

print("\nReloading selected checkpoint to verify it can be loaded...")

MB_best_model = AutoModelForSequenceClassification.from_pretrained(
    str(MB_FINAL_DIR),
    local_files_only=True,
).to(MB_DEVICE)

if MB_GPU_COUNT > 1:
    MB_best_model = torch.nn.DataParallel(
        MB_best_model,
        device_ids=MB_GPU_IDS,
        output_device=0,
    )

MB_best_model.eval()
_, MB_final_val_loader = MB_make_loaders()
MB_final_val = MB_evaluate(MB_best_model, MB_final_val_loader)

MB_final_metrics = {
    "accuracy": float(MB_final_val["accuracy"]),
    "balanced_accuracy": float(MB_final_val["balanced_accuracy"]),
    "macro_f1": float(MB_final_val["macro_f1"]),
    "weighted_f1": float(MB_final_val["weighted_f1"]),
    "loss": float(MB_final_val["loss"]),
    "samples": len(MB_y_val),
    "classes": MB_num_labels,
}

with open(
    MB_REPORT_DIR / "modernbert_final_validation_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(MB_final_metrics, f, indent=2)

MB_report = classification_report(
    MB_final_val["y_true"],
    MB_final_val["y_pred"],
    labels=MB_label_ids,
    target_names=MB_classes,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(MB_report).transpose().to_csv(
    MB_REPORT_DIR / "modernbert_validation_classification_report.csv"
)

MB_cm = confusion_matrix(
    MB_final_val["y_true"],
    MB_final_val["y_pred"],
    labels=MB_label_ids,
)
pd.DataFrame(MB_cm, index=MB_classes, columns=MB_classes).to_csv(
    MB_REPORT_DIR / "modernbert_validation_confusion_matrix_counts.csv"
)

MB_row_sums = MB_cm.sum(axis=1, keepdims=True)
MB_cm_norm = np.divide(
    MB_cm.astype(float),
    MB_row_sums,
    out=np.zeros_like(MB_cm, dtype=float),
    where=MB_row_sums != 0,
)
pd.DataFrame(
    MB_cm_norm,
    index=MB_classes,
    columns=MB_classes,
).to_csv(
    MB_REPORT_DIR / "modernbert_validation_confusion_matrix_normalized.csv"
)

print("\nVerified validation metrics:")
for metric, value in MB_final_metrics.items():
    print(f"{metric:25s}: {value}")

print("\nSaved winner:", MB_FINAL_DIR)
print("Saved reports:", MB_REPORT_DIR)
print("Optuna database:", MB_DB_PATH)
print("The test split has NOT been loaded or evaluated.")

# Keep final model/report artifacts; discard duplicate per-trial checkpoints.
shutil.rmtree(MB_TRIAL_ROOT, ignore_errors=True)

del MB_best_model, MB_final_val_loader
MB_clear_cuda()

print("\nModernBERT fine-tuning and hyperparameter tuning finished.")
print("Next step: a separate test evaluation and BERT-vs-ModernBERT comparison.")


## 11. Session recovery and training-artifact verification

These cells check the saved recovery archive, Optuna study, epoch history, and available checkpoints. They do not train a new model or modify the held-out test set.

In [ ]:

# ================================================================
# MODERNBERT SESSION RECOVERY + ENVIRONMENT PREFLIGHT
# No training; no test-set evaluation; no model files deleted
# ================================================================

import gc
import json
import zipfile
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import transformers
import optuna

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ------------------------- PATHS --------------------------------

ROOT = Path("/kaggle/working/resume_classification")
WORKING = Path("/kaggle/working")

TRAIN_PATH = ROOT / "data/splits/train.parquet"
VAL_PATH = ROOT / "data/splits/validation.parquet"
BERT_MAPPING = ROOT / "models/bert_best/label_mapping.json"

RECOVERY_DIR = ROOT / "models/modernbert_recovered_trial1_epoch3"
ZIP_PATTERN = "modernbert_best_recovery_*.zip"

# --------------------- ENVIRONMENT CHECK ------------------------

print("=" * 72)
print("MODERNBERT RECOVERY — ENVIRONMENT CHECK")
print("=" * 72)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Optuna:", optuna.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is visible. In Kaggle, open Settings > Accelerator and "
        "select a GPU option that exposes the two T4 GPUs."
    )

GPU_COUNT = torch.cuda.device_count()
print("Visible CUDA GPUs:", GPU_COUNT)

for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"{props.total_memory / (1024**3):.2f} GiB VRAM"
    )

if GPU_COUNT < 2:
    print(
        "\nWARNING: only one GPU is visible. Python cannot enable a second "
        "GPU if Kaggle has not exposed it. Check the notebook accelerator setting."
    )
else:
    print("\nBoth GPUs are visible to PyTorch.")

for required in [TRAIN_PATH, VAL_PATH, BERT_MAPPING]:
    if not required.exists():
        raise FileNotFoundError(f"Required project file missing: {required}")
    print("FOUND:", required)

# ------------------- FIND AND RESTORE CHECKPOINT ----------------

def has_model_weights(folder):
    weight_patterns = [
        "model.safetensors",
        "model-*.safetensors",
        "pytorch_model.bin",
        "pytorch_model-*.bin",
    ]
    return (
        folder.exists()
        and (folder / "config.json").exists()
        and any(list(folder.glob(p)) for p in weight_patterns)
    )


checkpoint_ready = has_model_weights(RECOVERY_DIR)
recovery_zip = None

if not checkpoint_ready:
    zip_files = sorted(
        WORKING.glob(ZIP_PATTERN),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if zip_files:
        recovery_zip = zip_files[0]
        print("\nUsing recovery archive:", recovery_zip)

        with zipfile.ZipFile(recovery_zip, "r") as zf:
            members = zf.namelist()

            if "best_model/config.json" not in members:
                raise FileNotFoundError(
                    "Recovery ZIP does not contain best_model/config.json. "
                    "Do not overwrite it; inspect the archive's contents."
                )

            # Extract only best_model/ files and reject unsafe paths.
            RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

            for member in members:
                if not member.startswith("best_model/"):
                    continue
                if member.endswith("/"):
                    continue

                relative = Path(member).relative_to("best_model")

                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError(
                        f"Unsafe archive path rejected: {member}"
                    )

                destination = (RECOVERY_DIR / relative).resolve()
                if RECOVERY_DIR.resolve() not in destination.parents and (
                    destination != RECOVERY_DIR.resolve()
                ):
                    raise ValueError(
                        f"Archive path escapes recovery folder: {member}"
                    )

                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member, "r") as source, open(destination, "wb") as target:
                    shutil.copyfileobj(source, target)

            # Display the saved training history, if present.
            history_members = [
                n for n in members
                if n.endswith("modernbert_epoch_history.csv")
            ]

            if history_members:
                history = pd.read_csv(zf.open(history_members[0]))
                print("\nSaved training-history summary:")
                print(history.tail(12).to_string(index=False))

            manifest_members = [
                n for n in members
                if n.endswith("recovery_manifest.json")
            ]
            if manifest_members:
                print("\nRecovery manifest:")
                print(
                    zf.read(manifest_members[0]).decode(
                        "utf-8", errors="replace"
                    )
                )

        checkpoint_ready = has_model_weights(RECOVERY_DIR)

    else:
        # Last fallback: use the original checkpoint if it still exists.
        candidates = list(
            ROOT.glob(
                "reports/modernbert_fixed_tuning_*/trial_checkpoints/"
                "trial_00/best_model"
            )
        )
        candidates = [
            p for p in candidates if has_model_weights(p)
        ]

        if candidates:
            source = max(
                candidates,
                key=lambda p: max(
                    f.stat().st_mtime for f in p.iterdir() if f.is_file()
                ),
            )
            print("\nUsing existing checkpoint directory:", source)
            shutil.copytree(source, RECOVERY_DIR, dirs_exist_ok=True)
            checkpoint_ready = has_model_weights(RECOVERY_DIR)

if not checkpoint_ready:
    raise FileNotFoundError(
        "Could not restore a complete ModernBERT checkpoint. "
        "No training has been started. Verify that the recovery ZIP exists "
        "in /kaggle/working or upload it using Kaggle's Upload button."
    )

print("\nRecovered model folder:", RECOVERY_DIR)
print("Model weights found:", [
    p.name for p in RECOVERY_DIR.iterdir()
    if p.is_file() and (
        p.name.endswith(".safetensors") or p.name.endswith(".bin")
    )
])

# Use exactly the same category mapping as BERT.
shutil.copy2(BERT_MAPPING, RECOVERY_DIR / "label_mapping.json")

with open(
    RECOVERY_DIR / "label_mapping.json", "r", encoding="utf-8"
) as f:
    mapping = json.load(f)

label2id = {
    str(k): int(v) for k, v in mapping["label2id"].items()
}
num_labels = len(label2id)

print("Label-map categories:", num_labels)

# ------------------- LOAD MODEL + TOKENIZER ---------------------

device = torch.device("cuda:0")

tokenizer = AutoTokenizer.from_pretrained(
    str(RECOVERY_DIR),
    local_files_only=True,
)
tokenizer.padding_side = "right"

model = AutoModelForSequenceClassification.from_pretrained(
    str(RECOVERY_DIR),
    local_files_only=True,
)
model.to(device)
model.eval()

if model.config.num_labels != num_labels:
    raise ValueError(
        f"Model outputs={model.config.num_labels}, mapping classes={num_labels}."
    )

# --------------------- VERIFY SPLITS ----------------------------

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)

print("\nTrain records:", len(train_df))
print("Validation records:", len(val_df))
print("Class count:", num_labels)

if "Text_prepared" not in val_df.columns:
    raise ValueError("Validation split lacks Text_prepared.")

# ------------------ VERIFY MULTI-GPU FORWARD --------------------

# This is inference only: no gradients and no weight updates.
sample_texts = (
    val_df["Text_prepared"].fillna("").astype(str).head(4).tolist()
)

batch = tokenizer(
    sample_texts,
    max_length=2048,
    truncation=True,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt",
)
batch = {k: v.to(device) for k, v in batch.items()}

if GPU_COUNT > 1:
    model_for_test = torch.nn.DataParallel(
        model,
        device_ids=list(range(GPU_COUNT)),
        output_device=0,
    )
    print("\nMulti-GPU inference wrapper: DataParallel")
    print("GPU IDs passed to DataParallel:", list(range(GPU_COUNT)))
else:
    model_for_test = model
    print("\nSingle-GPU inference wrapper.")

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda", dtype=torch.float16, enabled=True
    ):
        logits = model_for_test(**batch, return_dict=False)[0]

print("Inference output shape:", tuple(logits.shape))
print("Expected shape:", (len(sample_texts), num_labels))

if tuple(logits.shape) != (len(sample_texts), num_labels):
    raise RuntimeError("Checkpoint inference verification failed.")

# --------------------- GPU MEMORY SUMMARY -----------------------

torch.cuda.synchronize()

for i in range(GPU_COUNT):
    allocated = torch.cuda.memory_allocated(i) / (1024**3)
    reserved = torch.cuda.memory_reserved(i) / (1024**3)
    print(
        f"GPU {i}: allocated={allocated:.2f} GiB, "
        f"reserved={reserved:.2f} GiB"
    )

try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.used,utilization.gpu",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=10,
    )
    print("\nnvidia-smi snapshot:")
    print(result.stdout.strip() or result.stderr.strip())
except Exception as exc:
    print("Could not query nvidia-smi:", exc)

del model_for_test, batch, logits, model, tokenizer
gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 72)
print("RECOVERY AND ENVIRONMENT PREFLIGHT PASSED")
print("=" * 72)
print("Recovered checkpoint:", RECOVERY_DIR)
print("Training was NOT started.")
print("BERT and test-set files were NOT modified.")


In [ ]:

from pathlib import Path
import zipfile

ROOT = Path("/kaggle/working/resume_classification")
ZIP_PATH = Path(
    "/kaggle/working/modernbert_best_recovery_20261024_114250.zip"
)
OLD_RUN = ROOT / "reports/modernbert_fixed_tuning_20261024_091706"

print("=== RECOVERY AUDIT ===")

if ZIP_PATH.exists():
    print("\nDatabase/history files inside recovery ZIP:")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = zf.namelist()
        relevant = [
            name for name in members
            if name.lower().endswith((".db", ".sqlite", ".sqlite3"))
            or "epoch_history" in name.lower()
            or "hyperparameter_trials" in name.lower()
        ]
        print("\n".join(relevant) if relevant else "No study database found in ZIP.")
else:
    print("Recovery ZIP is not at the expected path:", ZIP_PATH)

print("\nExisting files from the original run:")
if OLD_RUN.exists():
    relevant_files = [
        p for p in OLD_RUN.rglob("*")
        if p.is_file() and (
            p.suffix.lower() in {".db", ".sqlite", ".sqlite3"}
            or "epoch_history" in p.name.lower()
            or "hyperparameter_trials" in p.name.lower()
        )
    ]
    print("\n".join(str(p) for p in relevant_files) if relevant_files
          else "No Optuna database or matching reports found in the original run folder.")
else:
    print("Original run directory is not present.")

print("\nAudit complete. No files were modified.")


In [ ]:

from pathlib import Path
import optuna
import pandas as pd

ROOT = Path("/kaggle/working/resume_classification")
RUN_DIR = ROOT / "reports/modernbert_fixed_tuning_20261024_091706"
DB_PATH = RUN_DIR / "modernbert_study.db"
HISTORY_PATH = RUN_DIR / "modernbert_epoch_history.csv"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Optuna database not found: {DB_PATH}")

if not HISTORY_PATH.exists():
    raise FileNotFoundError(f"Training history not found: {HISTORY_PATH}")

storage = f"sqlite:///{DB_PATH}"

print("=" * 72)
print("SAVED OPTUNA STUDY AUDIT")
print("=" * 72)

summaries = optuna.study.get_all_study_summaries(storage=storage)

if not summaries:
    raise RuntimeError("No Optuna studies were found in the saved database.")

for summary in summaries:
    print(f"\nStudy: {summary.study_name}")
    print("Direction:", summary.direction.name)
    print("Total trials:", summary.n_trials)

    study = optuna.load_study(
        study_name=summary.study_name,
        storage=storage,
    )

    rows = []
    for trial in study.trials:
        rows.append({
            "trial_number": trial.number + 1,
            "state": trial.state.name,
            "objective_value": trial.value,
            "learning_rate": trial.params.get("learning_rate"),
            "weight_decay": trial.params.get("weight_decay"),
            "warmup_ratio": trial.params.get("warmup_ratio"),
            "classifier_dropout": trial.params.get("classifier_dropout"),
            "best_epoch": trial.user_attrs.get("best_epoch"),
            "best_val_accuracy": trial.user_attrs.get("best_val_accuracy"),
            "best_val_macro_f1": trial.user_attrs.get("best_val_macro_f1"),
            "best_val_loss": trial.user_attrs.get("best_val_loss"),
        })

    display(pd.DataFrame(rows))

print("\n" + "=" * 72)
print("SAVED EPOCH HISTORY")
print("=" * 72)

history = pd.read_csv(HISTORY_PATH)
display(history.sort_values(["trial", "epoch"]))

print("\nAudit complete. No files were modified and no training was run.")


In [ ]:

from pathlib import Path
import optuna
import pandas as pd

ROOT = Path("/kaggle/working/resume_classification")
RUN = ROOT / "reports/modernbert_fixed_tuning_20261024_091706"
DB = RUN / "modernbert_study.db"
HISTORY = RUN / "modernbert_epoch_history.csv"
CHECKPOINTS = RUN / "trial_checkpoints"

assert DB.exists(), f"Missing Optuna database: {DB}"
assert HISTORY.exists(), f"Missing epoch history: {HISTORY}"

study_names = [
    s.study_name
    for s in optuna.study.get_all_study_summaries(
        storage=f"sqlite:///{DB}"
    )
]
assert study_names, "No saved Optuna study found."

study = optuna.load_study(
    study_name=study_names[0],
    storage=f"sqlite:///{DB}",
)

print("Study:", study.study_name)
print("Recorded trials:", len(study.trials))

trial_rows = []
for t in study.trials:
    trial_rows.append({
        "trial": t.number + 1,
        "state": t.state.name,
        "objective": t.value,
        "parameters": t.params,
    })

display(pd.DataFrame(trial_rows))

print("\nCheckpoint availability:")
for i in range(4):
    path = CHECKPOINTS / f"trial_{i:02d}" / "best_model"
    weights = list(path.glob("*.safetensors")) + list(path.glob("*.bin"))
    print(
        f"Trial {i+1}:",
        "weights found" if weights and (path / "config.json").exists()
        else "no complete checkpoint",
        "|", path
    )

print("\nNo training or file modification performed.")


## 12. Final ModernBERT test evaluation and comparison

Evaluate the recovered best checkpoint on the held-out test set, save aggregate metrics, and compare those results with the saved BERT baseline. Test data is used for final reporting rather than hyperparameter selection.

In [ ]:

# ================================================================
# MODERNBERT — FINAL TEST EVALUATION (INFERENCE ONLY)
# Loads recovered Trial 1, Epoch 3. No fine-tuning.
# Saves complete metrics and compares with saved BERT results.
# ================================================================

import gc
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# --------------------------- PATHS --------------------------------

ROOT = Path("/kaggle/working/resume_classification")
MODEL_DIR = ROOT / "models/modernbert_recovered_trial1_epoch3"
TEST_PATH = ROOT / "data/splits/test.parquet"
BERT_METRICS_PATTERN = "bert_final_test_*/test_metrics.json"

TEXT_COL = "Text_prepared"
LABEL_COL = "Category"
MAX_LENGTH = 2048
BATCH_SIZE = 4

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before running this evaluation.")

DEVICE = torch.device("cuda:0")
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
REPORT_DIR = ROOT / "reports" / f"modernbert_final_test_{STAMP}"
REPORT_DIR.mkdir(parents=True, exist_ok=False)

# ---------------------- VERIFY INPUT FILES -------------------------

for path in [
    MODEL_DIR / "config.json",
    MODEL_DIR / "model.safetensors",
    MODEL_DIR / "label_mapping.json",
    TEST_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("=" * 72)
print("MODERNBERT FINAL TEST EVALUATION")
print("=" * 72)
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", MODEL_DIR)
print("Input limit:", MAX_LENGTH)

# --------------------- LOAD LABEL MAPPING --------------------------

with open(MODEL_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

label2id = {
    str(label): int(idx)
    for label, idx in mapping["label2id"].items()
}
id2label = {idx: label for label, idx in label2id.items()}
classes = [id2label[i] for i in range(len(id2label))]
label_ids = np.arange(len(classes))
num_labels = len(classes)

if set(id2label) != set(range(num_labels)):
    raise ValueError("Label IDs must be contiguous.")

# ---------------------- LOAD MODEL AND TOKENIZER -------------------

print("\nLoading recovered ModernBERT checkpoint...")

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR), local_files_only=True
)
tokenizer.padding_side = "right"

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR), local_files_only=True
).to(DEVICE)

model.eval()

if model.config.num_labels != num_labels:
    raise ValueError(
        f"Checkpoint outputs {model.config.num_labels} classes, "
        f"but the mapping contains {num_labels}."
    )

if model.config.max_position_embeddings < MAX_LENGTH:
    raise ValueError("Requested input length exceeds the model configuration.")

# --------------------------- LOAD TEST SET --------------------------

test_df = pd.read_parquet(TEST_PATH).reset_index(drop=True)

if TEXT_COL not in test_df.columns or LABEL_COL not in test_df.columns:
    raise ValueError(f"Test split must contain {TEXT_COL!r} and {LABEL_COL!r}.")

test_df[TEXT_COL] = test_df[TEXT_COL].fillna("").astype(str)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(str).str.strip()

unknown = sorted(set(test_df[LABEL_COL]) - set(label2id))
if unknown:
    raise ValueError(f"Test labels missing from the training mapping: {unknown}")

y_true_expected = test_df[LABEL_COL].map(label2id).to_numpy(dtype=np.int64)

print("Test resumes:", f"{len(test_df):,}")
print("Number of categories:", num_labels)

# ---------------------------- TOKENIZATION --------------------------

print("\nTokenizing test resumes...")

encodings = tokenizer(
    test_df[TEXT_COL].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,
    return_attention_mask=True,
)

lengths = np.asarray([len(row) for row in encodings["input_ids"]])

print("Median encoded length:", int(np.median(lengths)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("Sequences reaching the limit:", int(np.sum(lengths >= MAX_LENGTH)))

class TestDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[idx])
        return item


def collate_batch(features):
    labels = torch.tensor(
        [item["labels"] for item in features],
        dtype=torch.long,
    )
    inputs = [
        {k: v for k, v in item.items() if k != "labels"}
        for item in features
    ]
    padded = tokenizer.pad(
        inputs,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )
    padded["labels"] = labels
    return padded


dataset = TestDataset(encodings, y_true_expected)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True,
)

# --------------------------- INFERENCE -------------------------------

all_true = []
all_pred = []
all_probs = []
loss_sum = 0.0
count = 0

print("\nRunning inference over the complete test set...")

with torch.inference_mode():
    for batch_idx, batch in enumerate(loader, start=1):
        labels = batch.pop("labels").to(DEVICE)
        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(**inputs, return_dict=False)[0]

        logits = logits.float()
        probs = torch.softmax(logits, dim=-1)

        loss_sum += F.cross_entropy(
            logits, labels, reduction="sum"
        ).item()
        count += len(labels)

        all_true.extend(labels.cpu().numpy().tolist())
        all_pred.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
        all_probs.append(probs.cpu().numpy())

        if batch_idx % 20 == 0 or count == len(dataset):
            print(f"Processed {count:,}/{len(dataset):,} test resumes")

y_true = np.asarray(all_true, dtype=np.int64)
y_pred = np.asarray(all_pred, dtype=np.int64)
probabilities = np.concatenate(all_probs, axis=0)
test_loss = loss_sum / max(count, 1)

if not np.array_equal(y_true, y_true_expected):
    raise RuntimeError("Test labels and inference order do not match.")

# --------------------------- METRICS ---------------------------------

accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred,
    labels=label_ids,
    average="macro",
    zero_division=0,
)

p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred,
    labels=label_ids,
    average="weighted",
    zero_division=0,
)

metrics = {
    "model": "ModernBERT Base fine-tuned",
    "checkpoint": str(MODEL_DIR),
    "input_limit_tokens": MAX_LENGTH,
    "test_samples": int(len(y_true)),
    "number_of_classes": int(num_labels),
    "test_loss": float(test_loss),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_precision": float(p_macro),
    "macro_recall": float(r_macro),
    "macro_f1": float(f1_macro),
    "weighted_precision": float(p_weighted),
    "weighted_recall": float(r_weighted),
    "weighted_f1": float(f1_weighted),
}

print("\n" + "=" * 65)
print("MODERNBERT — FINAL TEST METRICS")
print("=" * 65)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")

with open(REPORT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# ------------------------- CLASSIFICATION REPORT ---------------------

report = classification_report(
    y_true, y_pred,
    labels=label_ids,
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(report).transpose().to_csv(
    REPORT_DIR / "classification_report.csv"
)

# ----------------------- CONFUSION MATRICES ---------------------------

cm = confusion_matrix(y_true, y_pred, labels=label_ids)

pd.DataFrame(cm, index=classes, columns=classes).to_csv(
    REPORT_DIR / "confusion_matrix_counts.csv"
)

row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm.astype(float),
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0,
)

pd.DataFrame(cm_normalized, index=classes, columns=classes).to_csv(
    REPORT_DIR / "confusion_matrix_normalized.csv"
)


def plot_confusion(matrix, title, filename, colorbar_label):
    size = max(14, min(24, len(classes) * 0.48))
    fig, ax = plt.subplots(figsize=(size, size))

    image = ax.imshow(
        matrix, interpolation="nearest", aspect="auto", cmap="viridis"
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted category")
    ax.set_ylabel("True category")
    ax.set_xticks(label_ids)
    ax.set_yticks(label_ids)
    ax.set_xticklabels(classes, rotation=90, fontsize=6)
    ax.set_yticklabels(classes, fontsize=6)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04).set_label(
        colorbar_label
    )
    fig.tight_layout()
    fig.savefig(REPORT_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_confusion(
    cm,
    "ModernBERT — Test Confusion Matrix (Counts)",
    "confusion_matrix_counts.png",
    "Number of resumes",
)
plot_confusion(
    cm_normalized,
    "ModernBERT — Test Confusion Matrix (Row Normalized)",
    "confusion_matrix_normalized.png",
    "Fraction of actual-category resumes",
)

# ---------------------- PER-RESUME PREDICTIONS -----------------------

top_k = min(3, num_labels)
top_indices = np.argsort(probabilities, axis=1)[:, -top_k:][:, ::-1]

predictions = pd.DataFrame({
    "test_row_index": np.arange(len(test_df)),
    "true_label": [id2label[int(i)] for i in y_true],
    "predicted_label": [id2label[int(i)] for i in y_pred],
    "confidence": probabilities.max(axis=1),
    "correct": y_true == y_pred,
    "encoded_input_length": lengths,
})

if "source_row_index" in test_df.columns:
    predictions.insert(
        1, "source_row_index", test_df["source_row_index"].to_numpy()
    )

predictions["top_3_predictions"] = [
    json.dumps(
        [
            {
                "category": id2label[int(idx)],
                "probability": float(probabilities[row, idx]),
            }
            for idx in top_indices[row]
        ],
        ensure_ascii=False,
    )
    for row in range(len(test_df))
]

predictions.to_csv(
    REPORT_DIR / "test_predictions.csv",
    index=False,
)

# ---------------------- BERT COMPARISON -------------------------------

bert_report_files = list(ROOT.glob(BERT_METRICS_PATTERN))
comparison_rows = [{"model": "ModernBERT", **metrics}]

if bert_report_files:
    bert_json = max(
        bert_report_files,
        key=lambda p: p.stat().st_mtime,
    )

    with open(bert_json, "r", encoding="utf-8") as f:
        bert_metrics = json.load(f)

    comparison_metrics = [
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "test_loss",
    ]

    compare_table = pd.DataFrame([
        {
            "model": "BERT Base",
            **{
                key: bert_metrics.get(key, np.nan)
                for key in comparison_metrics
            },
        },
        {
            "model": "ModernBERT Base",
            **{
                key: metrics.get(key, np.nan)
                for key in comparison_metrics
            },
        },
    ])

    compare_table.to_csv(
        REPORT_DIR / "bert_vs_modernbert_test_comparison.csv",
        index=False,
    )

    print("\n" + "=" * 72)
    print("BERT VS MODERNBERT — TEST COMPARISON")
    print("=" * 72)
    print("BERT report loaded from:", bert_json)
    display(compare_table.round(4))

else:
    print(
        "\nNo saved BERT metrics JSON was found automatically. "
        "ModernBERT results are saved; compare with your BERT test report."
    )

# ------------------------- METADATA ----------------------------------

metadata = {
    "checkpoint": str(MODEL_DIR),
    "test_split": str(TEST_PATH),
    "timestamp": STAMP,
    "training_performed": False,
    "weights_modified": False,
    "test_evaluated": True,
    "max_length": MAX_LENGTH,
    "test_samples": int(len(test_df)),
    "num_classes": num_labels,
}

with open(
    REPORT_DIR / "evaluation_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 72)
print("MODERNBERT TEST EVALUATION COMPLETE")
print("=" * 72)
print("Reports saved to:", REPORT_DIR)
print("\nArtifacts:")
for path in sorted(REPORT_DIR.iterdir()):
    print(" -", path.name)

del model, loader, dataset, encodings, probabilities
gc.collect()
torch.cuda.empty_cache()

print("\nThe saved ModernBERT checkpoint was not modified.")


In [ ]:

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("/kaggle/working/resume_classification")
REPORTS = ROOT / "reports"

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = REPORTS / f"final_model_comparison_{STAMP}"
OUT.mkdir(parents=True, exist_ok=False)

# ---------------------- Find the final test reports ------------------

def newest_file(pattern):
    candidates = [
        p for p in REPORTS.glob(pattern)
        if p.is_file()
    ]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None

modern_metrics_path = newest_file(
    "modernbert_final_test_*/test_metrics.json"
)
bert_metrics_path = newest_file(
    "bert_final_test_*/test_metrics.json"
)

if modern_metrics_path is None:
    raise FileNotFoundError(
        "ModernBERT test_metrics.json was not found. "
        "Check reports/modernbert_final_test_20261024_132314/."
    )

with open(modern_metrics_path, "r", encoding="utf-8") as f:
    modern_metrics = json.load(f)

if bert_metrics_path is not None:
    with open(bert_metrics_path, "r", encoding="utf-8") as f:
        bert_metrics = json.load(f)
    bert_source = str(bert_metrics_path)
else:
    # Values from the previously displayed BERT test output.
    # These are manually transcribed fallback values, not reloaded files.
    bert_metrics = {
        "model": "BERT Base fine-tuned",
        "test_samples": 1188,
        "number_of_classes": 43,
        "test_loss": 0.5788,
        "accuracy": 0.8830,
        "balanced_accuracy": 0.8816,
        "macro_precision": 0.8799,
        "macro_recall": 0.8816,
        "macro_f1": 0.8748,
        "weighted_precision": 0.8877,
        "weighted_recall": 0.8830,
        "weighted_f1": 0.8808,
    }
    bert_source = "Manually transcribed from earlier BERT evaluation output"

modern_source = str(modern_metrics_path)

# ---------------------- Overall metric comparison --------------------

metric_names = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_precision",
    "weighted_recall",
    "weighted_f1",
    "test_loss",
]

rows = []
for key in metric_names:
    b = bert_metrics.get(key, np.nan)
    m = modern_metrics.get(key, np.nan)

    if pd.notna(b) and pd.notna(m):
        delta = float(m) - float(b)
    else:
        delta = np.nan

    rows.append({
        "metric": key,
        "BERT_Base": b,
        "ModernBERT_Base": m,
        "ModernBERT_minus_BERT": delta,
        "difference_percentage_points": (
            delta * 100 if key != "test_loss" and pd.notna(delta)
            else np.nan
        ),
    })

comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "overall_metric_comparison.csv", index=False)

print("=" * 72)
print("FINAL TEST METRIC COMPARISON")
print("=" * 72)
print("BERT source:", bert_source)
print("ModernBERT source:", modern_source)
display(comparison.round(4))

# ---------------------- Per-category comparison ----------------------

bert_class_report = newest_file(
    "bert_final_test_*/classification_report.csv"
)
modern_class_report = newest_file(
    "modernbert_final_test_*/classification_report.csv"
)

if bert_class_report and modern_class_report:
    bert_df = pd.read_csv(bert_class_report, index_col=0)
    modern_df = pd.read_csv(modern_class_report, index_col=0)

    # Exclude aggregate rows; retain class-level records.
    aggregate_rows = {
        "accuracy", "macro avg", "weighted avg", "micro avg", "samples avg"
    }

    bert_df.index = bert_df.index.astype(str)
    modern_df.index = modern_df.index.astype(str)

    bert_df = bert_df.loc[~bert_df.index.isin(aggregate_rows)]
    modern_df = modern_df.loc[~modern_df.index.isin(aggregate_rows)]

    common_classes = sorted(set(bert_df.index) & set(modern_df.index))

    per_class = pd.DataFrame({
        "category": common_classes,
        "BERT_precision": [
            bert_df.loc[c, "precision"] for c in common_classes
        ],
        "ModernBERT_precision": [
            modern_df.loc[c, "precision"] for c in common_classes
        ],
        "BERT_recall": [
            bert_df.loc[c, "recall"] for c in common_classes
        ],
        "ModernBERT_recall": [
            modern_df.loc[c, "recall"] for c in common_classes
        ],
        "BERT_F1": [
            bert_df.loc[c, "f1-score"] for c in common_classes
        ],
        "ModernBERT_F1": [
            modern_df.loc[c, "f1-score"] for c in common_classes
        ],
        "BERT_support": [
            bert_df.loc[c, "support"] for c in common_classes
        ],
        "ModernBERT_support": [
            modern_df.loc[c, "support"] for c in common_classes
        ],
    })

    per_class["F1_difference"] = (
        per_class["ModernBERT_F1"] - per_class["BERT_F1"]
    )
    per_class["better_F1"] = np.where(
        per_class["F1_difference"] > 0,
        "ModernBERT",
        np.where(
            per_class["F1_difference"] < 0,
            "BERT",
            "Tie",
        ),
    )

    per_class = per_class.sort_values(
        "F1_difference", ascending=True
    )
    per_class.to_csv(
        OUT / "per_category_metric_comparison.csv",
        index=False,
    )

    print("\nPer-category F1 comparison: weakest relative ModernBERT gains first")
    display(per_class.round(4))

    print("\nCategories where BERT performed better:")
    display(
        per_class.loc[per_class["F1_difference"] < 0, [
            "category", "BERT_F1", "ModernBERT_F1", "F1_difference"
        ]].round(4)
    )
else:
    print(
        "\nAt least one saved per-class report was not found. "
        "Overall comparison is available; inspect report folders for class reports."
    )

# ---------------------- Save decision summary ------------------------

summary = {
    "bert_metrics_source": bert_source,
    "modernbert_metrics_source": modern_source,
    "test_samples_bert": bert_metrics.get("test_samples", 1188),
    "test_samples_modernbert": modern_metrics.get("test_samples"),
    "bert_accuracy": bert_metrics.get("accuracy"),
    "modernbert_accuracy": modern_metrics.get("accuracy"),
    "bert_macro_f1": bert_metrics.get("macro_f1"),
    "modernbert_macro_f1": modern_metrics.get("macro_f1"),
    "bert_weighted_f1": bert_metrics.get("weighted_f1"),
    "modernbert_weighted_f1": modern_metrics.get("weighted_f1"),
    "provisional_candidate": "ModernBERT Base",
    "selection_basis": (
        "Better displayed final test accuracy and F1 metrics. "
        "Confirm latency, memory, per-class results, and preprocessing parity "
        "before production deployment."
    ),
    "warning": (
        "Test-set results have now been used to compare the two candidates. "
        "Do not tune hyperparameters against this same test set."
    ),
}

with open(OUT / "model_selection_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 72)
print("COMPARISON ARTIFACTS SAVED")
print("=" * 72)
print("Folder:", OUT)
for path in sorted(OUT.iterdir()):
    print(" -", path.name)


## 13. Package the selected checkpoint for FastAPI inference

Package the selected checkpoint, tokenizer files, label mapping, API source, dependency list, and evaluation reports. The resulting bundle is the source used for the current API deployment.

In [ ]:

# =====================================================================
# PACKAGE MODERNBERT FOR FASTAPI
# Model + tokenizer + label mapping + API + requirements + reports
# Inference-only packaging: no training and no model-weight changes.
# =====================================================================

import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime
from IPython.display import FileLink, display

ROOT = Path("/kaggle/working/resume_classification")

# Best ModernBERT checkpoint used in the successful test evaluation.
MODEL_DIR = ROOT / "models/modernbert_recovered_trial1_epoch3"

# Final ModernBERT test report and model comparison artifacts.
TEST_REPORTS = ROOT / "reports"
COMPARISON_REPORTS = ROOT / "reports"

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
BUNDLE_DIR = ROOT / "deployment" / f"modernbert_fastapi_{STAMP}"
ZIP_PATH = Path("/kaggle/working") / f"modernbert_fastapi_bundle_{STAMP}.zip"

# ---------------------- VERIFY MODEL FILES ----------------------------

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Model folder not found: {MODEL_DIR}")

required_files = [
    MODEL_DIR / "config.json",
    MODEL_DIR / "model.safetensors",
    MODEL_DIR / "tokenizer.json",
    MODEL_DIR / "label_mapping.json",
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required model artifact missing: {path}\n"
            "Do not package an incomplete checkpoint."
        )

with open(MODEL_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

num_labels = len(label_mapping["label2id"])
if num_labels != 43:
    raise ValueError(
        f"Expected 43 categories, found {num_labels}. "
        "Verify the checkpoint and label mapping."
    )

# -------------------------- CREATE BUNDLE ------------------------------

BUNDLE_DIR.mkdir(parents=True, exist_ok=False)
BUNDLE_MODEL_DIR = BUNDLE_DIR / "model"
shutil.copytree(MODEL_DIR, BUNDLE_MODEL_DIR)

print("Model checkpoint copied.")
print("Categories:", num_labels)
print("Bundle directory:", BUNDLE_DIR)

# --------------------------- FASTAPI APP --------------------------------

app_source = r'''
import json
import os
import time
from contextlib import asynccontextmanager
from pathlib import Path
from typing import List

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "model"

MAX_LENGTH = int(os.getenv("MODEL_MAX_LENGTH", "2048"))
MODEL_NAME = "ModernBERT Base — Resume Classification"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = None
model = None
id2label = None
label2id = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global tokenizer, model, id2label, label2id

    mapping_file = MODEL_DIR / "label_mapping.json"
    if not mapping_file.exists():
        raise RuntimeError(f"Label mapping is missing: {mapping_file}")

    with open(mapping_file, "r", encoding="utf-8") as f:
        mapping = json.load(f)

    label2id = {str(k): int(v) for k, v in mapping["label2id"].items()}
    id2label = {int(v): str(k) for k, v in mapping["label2id"].items()}

    if len(label2id) != 43:
        raise RuntimeError(
            f"Expected 43 label categories, found {len(label2id)}."
        )

    tokenizer = AutoTokenizer.from_pretrained(
        str(MODEL_DIR), local_files_only=True
    )
    tokenizer.padding_side = "right"

    model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR), local_files_only=True
    ).to(DEVICE)
    model.eval()

    if model.config.num_labels != len(label2id):
        raise RuntimeError(
            "Model output count does not match the saved label mapping."
        )

    if MAX_LENGTH > getattr(model.config, "max_position_embeddings", 8192):
        raise RuntimeError("Configured input limit exceeds model capacity.")

    print(f"Loaded {MODEL_NAME} on {DEVICE}; max tokens={MAX_LENGTH}")
    yield

    model = None
    tokenizer = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


app = FastAPI(
    title="Resume Classification API",
    description=(
        "Fine-tuned ModernBERT classifier. "
        "Input must use the same prepared-text representation as training."
    ),
    version="1.0.0",
    lifespan=lifespan,
)


class PredictRequest(BaseModel):
    prepared_text: str = Field(
        ...,
        min_length=1,
        max_length=200000,
        description=(
            "Resume text already processed using the project's training-time "
            "Text_prepared preprocessing. Raw PDF/DOCX text is not accepted "
            "as equivalent until that preprocessing pipeline is reproduced."
        ),
    )
    top_k: int = Field(default=3, ge=1, le=5)


class ClassPrediction(BaseModel):
    category: str
    probability: float


class PredictResponse(BaseModel):
    predicted_category: str
    confidence: float
    top_predictions: List[ClassPrediction]
    token_count_before_truncation: int
    input_limit_tokens: int
    was_truncated: bool
    inference_ms: float
    model: str


@app.get("/health")
def health():
    return {
        "status": "ok" if model is not None else "loading",
        "model": MODEL_NAME,
        "device": str(DEVICE),
        "max_input_tokens": MAX_LENGTH,
        "classes": len(label2id) if label2id else 0,
    }


@app.get("/")
def root():
    return {
        "message": "Resume Classification API",
        "docs": "/docs",
        "health": "/health",
    }


@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model is not ready.")

    text = request.prepared_text.strip()
    if not text:
        raise HTTPException(
            status_code=422,
            detail="prepared_text must not be blank."
        )

    try:
        start = time.perf_counter()

        # Count tokens before truncation so the client knows whether
        # information was omitted at the configured inference limit.
        untruncated = tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )
        original_token_count = len(untruncated["input_ids"])

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        with torch.inference_mode():
            if DEVICE.type == "cuda":
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    logits = model(**inputs).logits
            else:
                logits = model(**inputs).logits

            probabilities = torch.softmax(logits.float(), dim=-1)[0]

        k = min(request.top_k, len(label2id))
        top_probs, top_indices = torch.topk(probabilities, k=k)

        top_predictions = [
            ClassPrediction(
                category=id2label[int(idx)],
                probability=round(float(prob), 6),
            )
            for prob, idx in zip(
                top_probs.cpu().tolist(),
                top_indices.cpu().tolist(),
            )
        ]

        elapsed_ms = (time.perf_counter() - start) * 1000.0
        best = top_predictions[0]

        return PredictResponse(
            predicted_category=best.category,
            confidence=best.probability,
            top_predictions=top_predictions,
            token_count_before_truncation=original_token_count,
            input_limit_tokens=MAX_LENGTH,
            was_truncated=original_token_count > MAX_LENGTH,
            inference_ms=round(elapsed_ms, 2),
            model=MODEL_NAME,
        )

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Inference failed: {type(exc).__name__}",
        ) from exc
'''

(BUNDLE_DIR / "app.py").write_text(app_source, encoding="utf-8")

# ---------------------- DEPENDENCIES ----------------------------------

requirements = """\
fastapi>=0.115,<1.0
uvicorn[standard]>=0.30,<1.0
transformers==5.16.1
safetensors>=0.4.5
numpy>=1.26
"""

(BUNDLE_DIR / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

# --------------------------- README ------------------------------------

readme = """\
# ModernBERT Resume Classification API

## Model
- Architecture: answerdotai/ModernBERT-base, fine-tuned for 43 categories
- Input limit used during evaluation: 2048 tokens
- Test accuracy: 92.09%
- Test macro-F1: 92.12%
- Test weighted-F1: 92.06%
- Test sample count: 1188

These metrics come from the recorded test evaluation. Deployment performance
may differ depending on input preprocessing, hardware, and request patterns.

## Important preprocessing requirement
The classifier was trained on the dataset's `Text_prepared` column.
The API accepts `prepared_text` in that representation. It does not extract
PDF or DOCX files and does not recreate the training preprocessing pipeline.

Before connecting raw resumes, reproduce the exact preprocessing applied to
training data. Do not assume that generic whitespace cleanup or lowercasing
is equivalent to the training inputs.

## Install
Create a virtual environment, then install the requirements.

```bash
python -m venv .venv
```

Activate the environment and install:

```bash
pip install -r requirements.txt
```

PyTorch installation may need to be selected for the deployment hardware.
Use an appropriate CPU or CUDA build from the official PyTorch installation
selector rather than assuming the Kaggle CUDA build is portable everywhere.

## Run locally
From the directory containing `app.py`:

```bash
uvicorn app:app --host 127.0.0.1 --port 8000
```

Open the interactive documentation at:
http://127.0.0.1:8000/docs

Health check:
http://127.0.0.1:8000/health

## Request example
POST `/predict`:

```json
{
  "prepared_text": "already prepared resume text in the same representation as Text_prepared",
  "top_k": 3
}
```

## Response
The API returns the predicted category, confidence, top-k category probabilities,
token count before truncation, whether truncation occurred, and inference time.

## Deployment notes
- Load the model once when the service starts; do not reload it per request.
- Keep the label mapping bundled with the model.
- Use HTTPS and appropriate authentication/rate limiting before public exposure.
- Resume text can contain personal data. Avoid logging full request bodies.
- The 2048-token inference limit is intentional and matches the evaluated setup.
- Benchmark latency and memory on the actual deployment platform.
"""

(BUNDLE_DIR / "README.md").write_text(readme, encoding="utf-8")

# Save model provenance.
manifest = {
    "model_source": str(MODEL_DIR),
    "architecture": "answerdotai/ModernBERT-base",
    "num_classes": num_labels,
    "max_length_tokens": 2048,
    "test_metrics": {
        "accuracy": 0.9209,
        "macro_f1": 0.9212,
        "weighted_f1": 0.9206,
        "test_loss": 0.3881,
        "test_samples": 1188,
    },
    "input_contract": (
        "prepared_text should match the training dataset's Text_prepared representation."
    ),
    "created_at": STAMP,
}

(BUNDLE_DIR / "model_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

# Include final evaluation evidence if available.
test_report_dirs = sorted(
    ROOT.glob("reports/modernbert_final_test_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if test_report_dirs:
    source_report = test_report_dirs[0]
    eval_dir = BUNDLE_DIR / "evaluation"
    eval_dir.mkdir()

    for filename in [
        "test_metrics.json",
        "classification_report.csv",
        "confusion_matrix_counts.csv",
        "confusion_matrix_normalized.csv",
        "evaluation_metadata.json",
    ]:
        source = source_report / filename
        if source.exists():
            shutil.copy2(source, eval_dir / filename)

# Include the BERT-vs-ModernBERT comparison CSV where present.
comparison_dirs = sorted(
    ROOT.glob("reports/final_model_comparison_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if comparison_dirs:
    compare_file = comparison_dirs[0] / "overall_metric_comparison.csv"
    if compare_file.exists():
        eval_dir = BUNDLE_DIR / "evaluation"
        eval_dir.mkdir(exist_ok=True)
        shutil.copy2(
            compare_file,
            eval_dir / "bert_vs_modernbert_metrics.csv",
        )

# ----------------------------- PACKAGE ZIP -----------------------------

print("\nBundle contents:")
for path in sorted(BUNDLE_DIR.rglob("*")):
    if path.is_file():
        print(
            f"{path.relative_to(BUNDLE_DIR)} "
            f"({path.stat().st_size / (1024**2):.1f} MB)"
        )

# Do not recompress large model weights; this avoids wasting time/CPU.
with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_STORED,
) as archive:
    for path in BUNDLE_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    Path(BUNDLE_DIR.name) / path.relative_to(BUNDLE_DIR)
                ),
            )

print("\n" + "=" * 72)
print("FASTAPI DEPLOYMENT BUNDLE CREATED")
print("=" * 72)
print("Folder:", BUNDLE_DIR)
print("ZIP:", ZIP_PATH)
print("ZIP size:", round(ZIP_PATH.stat().st_size / (1024**2), 1), "MB")
print("\nDownload the bundle to your machine:")
display(FileLink(str(ZIP_PATH)))
